# 🚲 CitiBike Brooklyn Expansion Analysis
## Phase 1: Data Collection & Ingestion

**Project Goal:** Build a data-driven case for CitiBike expansion into underserved Brooklyn (and broader NYC borough) neighborhoods.

**This notebook covers:**
- Environment setup & library installation
- CitiBike live station feed (GBFS) ingestion
- CitiBike historical trip data download (2023–2024)
- MTA bus stop data ingestion
- NYC neighborhood boundary ingestion
- Census ACS demographic data ingestion
- DuckDB setup & initial table creation
- Folder structure setup for the full project

**Stack:** Python, DuckDB, Pandas, GeoPandas, Requests

In [3]:
# SESSION SETUP
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, warnings
import pandas as pd
import numpy as np
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── All project files live here — persists across sessions
PROJECT_ROOT = Path('/content/drive/MyDrive/citibike_project')

DIRS = {
    'raw_trips'    : PROJECT_ROOT / 'data' / 'raw' / 'trips',
    'raw_stations' : PROJECT_ROOT / 'data' / 'raw' / 'stations',
    'raw_mta'      : PROJECT_ROOT / 'data' / 'raw' / 'mta',
    'raw_census'   : PROJECT_ROOT / 'data' / 'raw' / 'census',
    'raw_geo'      : PROJECT_ROOT / 'data' / 'raw' / 'geo',
    'clean'        : PROJECT_ROOT / 'data' / 'clean',
    'exports'      : PROJECT_ROOT / 'data' / 'exports',
    'db'           : PROJECT_ROOT / 'db',
    'notebooks'    : PROJECT_ROOT / 'notebooks',
    'models'       : PROJECT_ROOT / 'models',
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("✅ Google Drive mounted.")
print(f"   Project root → {PROJECT_ROOT}")
print("\nFolder inventory:")
for name, path in DIRS.items():
    files = list(path.glob('*'))
    size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
    print(f"   {name:<15} {len(files):>3} file(s)  {size_mb:>7.1f} MB")

Mounted at /content/drive
✅ Google Drive mounted.
   Project root → /content/drive/MyDrive/citibike_project

Folder inventory:
   raw_trips         5 file(s)    783.7 MB
   raw_stations      1 file(s)      0.3 MB
   raw_mta           1 file(s)      0.9 MB
   raw_census        1 file(s)      0.4 MB
   raw_geo           6 file(s)     10.7 MB
   clean             0 file(s)      0.0 MB
   exports          10 file(s)      0.6 MB
   db                1 file(s)    416.3 MB
   notebooks         0 file(s)      0.0 MB
   models            0 file(s)      0.0 MB


In [4]:
!pip install duckdb geopandas census us folium mapclassify pyogrio --quiet

print("✅ Libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 13.8 MB/s eta 0:00:00
✅ Libraries installed.


In [5]:
import os
import json
import zipfile
import requests
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import duckdb
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"✅ All imports successful.")
print(f"   pandas     : {pd.__version__}")
print(f"   geopandas  : {gpd.__version__}")
print(f"   duckdb     : {duckdb.__version__}")

✅ All imports successful.
   pandas     : 2.2.2
   geopandas  : 1.1.3
   duckdb     : 1.3.2


---
## 2. Project Folder Structure
Set up a clean, consistent folder layout that all subsequent phases will use.

In [6]:
# ── Project root (adjust if using Google Drive)
# To persist files across Colab sessions, mount Drive and set:
# PROJECT_ROOT = Path('/content/drive/MyDrive/citibike_project')
PROJECT_ROOT = Path('/content/citibike_project')

# Subdirectory map
DIRS = {
    'raw_trips'    : PROJECT_ROOT / 'data' / 'raw' / 'trips',
    'raw_stations' : PROJECT_ROOT / 'data' / 'raw' / 'stations',
    'raw_mta'      : PROJECT_ROOT / 'data' / 'raw' / 'mta',
    'raw_census'   : PROJECT_ROOT / 'data' / 'raw' / 'census',
    'raw_geo'      : PROJECT_ROOT / 'data' / 'raw' / 'geo',
    'clean'        : PROJECT_ROOT / 'data' / 'clean',
    'exports'      : PROJECT_ROOT / 'data' / 'exports',   # → Tableau CSVs
    'db'           : PROJECT_ROOT / 'db',
    'notebooks'    : PROJECT_ROOT / 'notebooks',
    'models'       : PROJECT_ROOT / 'models',
}

for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)

print("✅ Project folder structure created:")
for name, path in DIRS.items():
    print(f"   {name:<15} → {path}")

✅ Project folder structure created:
   raw_trips       → /content/citibike_project/data/raw/trips
   raw_stations    → /content/citibike_project/data/raw/stations
   raw_mta         → /content/citibike_project/data/raw/mta
   raw_census      → /content/citibike_project/data/raw/census
   raw_geo         → /content/citibike_project/data/raw/geo
   clean           → /content/citibike_project/data/clean
   exports         → /content/citibike_project/data/exports
   db              → /content/citibike_project/db
   notebooks       → /content/citibike_project/notebooks
   models          → /content/citibike_project/models


---
## 3. CitiBike Live Station Feed (GBFS)

CitiBike publishes a **General Bikeshare Feed Specification (GBFS)** — a live JSON feed with every station's name, location, capacity, and real-time availability.

This gives us the **complete current network map** — the baseline we'll use to identify gap zones.

In [7]:
# GBFS endpoints
GBFS_STATION_INFO   = 'https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json'
GBFS_STATION_STATUS = 'https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_status.json'

def fetch_gbfs(url: str, label: str) -> pd.DataFrame:
    """Fetch a GBFS endpoint and return its stations array as a DataFrame."""
    print(f"   Fetching {label}...")
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    data = resp.json()['data']['stations']
    df = pd.DataFrame(data)
    print(f"   ✅ {len(df):,} records returned.")
    return df

print("Pulling GBFS station feeds...")
df_info   = fetch_gbfs(GBFS_STATION_INFO,   'station_information')
df_status = fetch_gbfs(GBFS_STATION_STATUS, 'station_status')

print("\nStation information columns:")
print(df_info.columns.tolist())

Pulling GBFS station feeds...
   Fetching station_information...
   ✅ 2,458 records returned.
   Fetching station_status...
   ✅ 2,458 records returned.

Station information columns:
['station_id', 'name', 'short_name', 'lon', 'lat', 'region_id', 'capacity', 'rental_uris']


In [8]:
INFO_COLS = ['station_id', 'name', 'short_name', 'lat', 'lon', 'capacity', 'region_id']
INFO_COLS = [c for c in INFO_COLS if c in df_info.columns]  # safe subset
df_info = df_info[INFO_COLS].copy()

# Key availability columns from station_status
STATUS_COLS = ['station_id', 'num_bikes_available', 'num_docks_available',
               'num_ebikes_available', 'is_installed', 'is_renting', 'last_reported']
STATUS_COLS = [c for c in STATUS_COLS if c in df_status.columns]
df_status = df_status[STATUS_COLS].copy()

df_stations = df_info.merge(df_status, on='station_id', how='left')
if 'last_reported' in df_stations.columns:
    df_stations['last_reported'] = pd.to_datetime(df_stations['last_reported'], unit='s')

# Borough labels based on lat/lon bounding boxes
def assign_borough(lat, lon):
    """Rough bounding-box borough assignment — will be refined with spatial join later."""
    if 40.495 <= lat <= 40.739 and -74.260 <= lon <= -73.833:
        return 'Brooklyn'
    elif 40.700 <= lat <= 40.800 and -74.020 <= lon <= -73.907:
        return 'Manhattan'
    elif 40.541 <= lat <= 40.670 and -73.833 <= lon <= -73.700:
        return 'Queens'
    elif 40.785 <= lat <= 40.915 and -73.933 <= lon <= -73.765:
        return 'Bronx'
    elif 40.495 <= lat <= 40.650 and -74.260 <= lon <= -74.034:
        return 'Staten Island'
    return 'Other'

df_stations['borough'] = df_stations.apply(
    lambda r: assign_borough(r['lat'], r['lon']), axis=1
)

# Save
out_path = DIRS['raw_stations'] / 'stations_live.csv'
df_stations.to_csv(out_path, index=False)

print(f"✅ Stations saved → {out_path}")
print(f"\nTotal stations  : {len(df_stations):,}")
print(f"\nBy borough:")
print(df_stations['borough'].value_counts().to_string())
print(f"\nSample rows:")
df_stations.head(3)

✅ Stations saved → /content/citibike_project/data/raw/stations/stations_live.csv

Total stations  : 2,458

By borough:
borough
Brooklyn     1252
Manhattan     468
Bronx         400
Other         338

Sample rows:


,station_id,name,short_name,lat,lon,capacity,region_id,num_bikes_available,num_docks_available,num_ebikes_available,is_installed,is_renting,last_reported,borough
0,2206782424530885178,Queens Blvd N & 78 Ave,5355.06,40.72,-73.83,24,71,0,0,0,0,0,1970-01-02 00:00:00,Other
1,2206781040135081610,Austin St & 76 Ave,5324.03,40.72,-73.84,16,71,0,0,0,0,0,1970-01-02 00:00:00,Brooklyn
2,66dd1f44-0aca-11e7-82f6-3863bb44ef7c,Greenpoint Ave & Manhattan Ave,5785.05,40.73,-73.95,27,71,0,0,0,0,0,2026-06-23 17:37:31,Brooklyn


---
## 4. CitiBike Historical Trip Data (2023–2024)

CitiBike publishes monthly trip CSVs at `https://s3.amazonaws.com/tripdata/`.

**Strategy:** We download months programmatically. Full 2023–2024 is ~24 files and several GB.
For your initial EDA run, we pull **Q4 2023 + Q1 2024** (6 months) — enough to capture seasonality. You can expand later.

> 💡 **Tip:** Set `SAMPLE_ONLY = True` below to download just 1 month for a fast first run.

In [9]:
# ── Configuration — flip SAMPLE_ONLY to False to pull the full date range
SAMPLE_ONLY = True   # True = 2 months only (fast); False = full range below

TRIP_BASE_URL = 'https://s3.amazonaws.com/tripdata/'

# Full target range: Oct 2023 – Mar 2024 (6 months, two seasons)
FULL_MONTHS = [
    '202310', '202311', '202312',   # Q4 2023
    '202401', '202402', '202403',   # Q1 2024
]

SAMPLE_MONTHS = ['202401', '202402']   # Jan–Feb 2024 only

TARGET_MONTHS = SAMPLE_MONTHS if SAMPLE_ONLY else FULL_MONTHS
print(f"Mode          : {'SAMPLE (2 months)' if SAMPLE_ONLY else 'FULL (6 months)'}")
print(f"Months to pull: {TARGET_MONTHS}")

Mode          : SAMPLE (2 months)
Months to pull: ['202401', '202402']


In [10]:
def download_trip_file(yyyymm: str, dest_dir: Path) -> Path | None:
    """
    Download one month of CitiBike trip data.
    CitiBike uses two filename formats — we try both.
    Returns the path to the extracted CSV, or None on failure.
    """
    # Two known filename patterns CitiBike has used
    candidates = [
        f'{yyyymm}-citibike-tripdata.csv.zip',
        f'{yyyymm}-citibike-tripdata.zip',
    ]

    for filename in candidates:
        url      = TRIP_BASE_URL + filename
        zip_path = dest_dir / filename
        csv_name = filename.replace('.zip', '')
        csv_path = dest_dir / csv_name

        if csv_path.exists():
            print(f"   ⏭  {csv_name} already exists — skipping download.")
            return csv_path

        print(f"   ↓  Trying {url}")
        try:
            resp = requests.get(url, stream=True, timeout=60)
            if resp.status_code == 404:
                continue   # try next pattern
            resp.raise_for_status()

            total = 0
            with open(zip_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)
                    total += len(chunk)
            print(f"      Downloaded {total / 1e6:.1f} MB")

            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(dest_dir)
            zip_path.unlink()   # remove zip to save space
            csvs = list(dest_dir.glob('*.csv'))
            if csvs:
                extracted = max(csvs, key=lambda p: p.stat().st_mtime)
                print(f"      ✅ Extracted → {extracted.name}")
                return extracted

        except Exception as e:
            print(f"      ⚠️  Error on {filename}: {e}")
            if zip_path.exists():
                zip_path.unlink()

    print(f"   ❌ Could not download data for {yyyymm}")
    return None


print("Starting trip data download...\n")
downloaded_csvs = []

for month in TARGET_MONTHS:
    print(f"── {month} ──")
    result = download_trip_file(month, DIRS['raw_trips'])
    if result:
        downloaded_csvs.append(result)

print(f"\n✅ Done. {len(downloaded_csvs)} file(s) ready for ingestion.")

Starting trip data download...

── 202401 ──
   ↓  Trying https://s3.amazonaws.com/tripdata/202401-citibike-tripdata.csv.zip
   ↓  Trying https://s3.amazonaws.com/tripdata/202401-citibike-tripdata.zip
      Downloaded 369.0 MB
      ✅ Extracted → 202401-citibike-tripdata_1.csv
── 202402 ──
   ↓  Trying https://s3.amazonaws.com/tripdata/202402-citibike-tripdata.csv.zip
   ↓  Trying https://s3.amazonaws.com/tripdata/202402-citibike-tripdata.zip
      Downloaded 414.7 MB
      ✅ Extracted → 202402-citibike-tripdata_3.csv

✅ Done. 2 file(s) ready for ingestion.


In [11]:
# Preview
if downloaded_csvs:
    sample_path = downloaded_csvs[0]
    print(f"Previewing: {sample_path.name}\n")
    df_sample = pd.read_csv(sample_path, nrows=5000)

    print(f"Shape (sample 5k rows): {df_sample.shape}")
    print(f"\nColumns:")
    for col in df_sample.columns:
        print(f"   {col:<35} {df_sample[col].dtype}")

    print(f"\nSample rows:")
    df_sample.head(3)

Previewing: 202401-citibike-tripdata_1.csv

Shape (sample 5k rows): (5000, 13)

Columns:
   ride_id                             object
   rideable_type                       object
   started_at                          object
   ended_at                            object
   start_station_name                  object
   start_station_id                    float64
   end_station_name                    object
   end_station_id                      float64
   start_lat                           float64
   start_lng                           float64
   end_lat                             float64
   end_lng                             float64
   member_casual                       object

Sample rows:


---
## 5. MTA Bus Stop Data

Bus stop locations let us identify **transit dependency** — areas where residents rely heavily on the bus to access the subway. These are our primary expansion targets.

Source: NYC Open Data (no API key required).

In [12]:
# SECTION 5 - MTA Bus Stop Data via GTFS
# Source: MTA Developer Portal (https://www.mta.info/developers)


import zipfile, io
MTA_GTFS_FEEDS = {
    'brooklyn'     : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_b.zip',
    'bronx'        : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_bx.zip',
    'manhattan'    : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_m.zip',
    'queens'       : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_q.zip',
    'staten_island': 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_si.zip',
    'mta_busco'    : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_busco.zip',
}

def fetch_gtfs_stops(borough_name: str, url: str) -> pd.DataFrame | None:
    """
    Download a GTFS zip from MTA S3, extract stops.txt,
    and return it as a DataFrame tagged with the borough name.
    """
    print(f"   Fetching {borough_name}...")
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            if 'stops.txt' not in zf.namelist():
                print(f"      ⚠️  No stops.txt found in {borough_name} zip")
                return None
            with zf.open('stops.txt') as f:
                df = pd.read_csv(f, dtype=str)

        df['borough_feed'] = borough_name
        print(f"      ✅ {len(df):,} stops")
        return df

    except Exception as e:
        print(f"      ❌ Failed: {e}")
        return None


print("Downloading MTA GTFS bus stop data...\n")
frames = []
for borough, url in MTA_GTFS_FEEDS.items():
    df = fetch_gtfs_stops(borough, url)
    if df is not None:
        frames.append(df)

df_bus_stops = pd.concat(frames, ignore_index=True) if frames else None

if df_bus_stops is not None:
    # Standardize
    df_bus_stops = df_bus_stops.rename(columns={
        'stop_id'  : 'stop_id',
        'stop_name': 'stop_name',
        'stop_lat' : 'lat',
        'stop_lon' : 'lon',
    })

    # Convert lat/lon to numeric
    df_bus_stops['lat'] = pd.to_numeric(df_bus_stops['lat'], errors='coerce')
    df_bus_stops['lon'] = pd.to_numeric(df_bus_stops['lon'], errors='coerce')

    # Drop duplicates as some stops appear in multiple borough feeds
    df_bus_stops = df_bus_stops.drop_duplicates(subset='stop_id')

    # Filter to NYC bounding box (removes any outliers/non-stop rows)
    df_bus_stops = df_bus_stops[
        df_bus_stops['lat'].between(40.45, 40.95) &
        df_bus_stops['lon'].between(-74.30, -73.65)
    ]

    out_path = DIRS['raw_mta'] / 'bus_stops.csv'
    df_bus_stops.to_csv(out_path, index=False)

    print(f"\n✅ Combined bus stop file saved → {out_path}")
    print(f"   Total unique stops : {len(df_bus_stops):,}")
    print(f"\nColumns available   : {df_bus_stops.columns.tolist()}")
    print(f"\nSample rows:")
    display(df_bus_stops[['stop_id','stop_name','lat','lon','borough_feed']].head(5))
else:
    print("❌ No MTA data loaded — check your internet connection.")


   Fetching brooklyn...
      ✅ 4,558 stops
   Fetching bronx...
      ✅ 1,884 stops
   Fetching manhattan...
      ✅ 1,825 stops
   Fetching queens...
      ✅ 1,398 stops
   Fetching staten_island...
      ✅ 1,940 stops
   Fetching mta_busco...
      ✅ 2,732 stops

✅ Combined bus stop file saved → /content/citibike_project/data/raw/mta/bus_stops.csv
   Total unique stops : 13,370

Columns available   : ['stop_id', 'stop_name', 'stop_desc', 'lat', 'lon', 'zone_id', 'stop_url', 'location_type', 'parent_station', 'borough_feed']

Sample rows:


,stop_id,stop_name,lat,lon,borough_feed
0,300000,ORIENTAL BLVD/MACKENZIE ST,40.58,-73.94,brooklyn
1,300002,ORIENTAL BLVD/JAFFRAY ST,40.58,-73.94,brooklyn
2,300003,ORIENTAL BLVD/HASTINGS ST,40.58,-73.94,brooklyn
3,300004,ORIENTAL BLVD/FALMOUTH ST,40.58,-73.95,brooklyn
4,300006,ORIENTAL BLVD/DOVER ST,40.58,-73.95,brooklyn


---
## 6. NYC Neighborhood & Borough Boundaries

We pull two boundary layers:
- **NTA (Neighborhood Tabulation Areas)** — neighborhood-level polygons used by NYC Planning, align with Census data
- **Borough boundaries** — for borough-level filtering and map aesthetics

In [13]:
# SECTION 6 — NYC Boundary Layers

import io, os, json, requests, warnings, subprocess
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely import wkt
from shapely.geometry import shape

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = Path('/content/drive/MyDrive/citibike_project')

DIRS = {
    'raw_trips'    : PROJECT_ROOT / 'data' / 'raw' / 'trips',
    'raw_stations' : PROJECT_ROOT / 'data' / 'raw' / 'stations',
    'raw_mta'      : PROJECT_ROOT / 'data' / 'raw' / 'mta',
    'raw_census'   : PROJECT_ROOT / 'data' / 'raw' / 'census',
    'raw_geo'      : PROJECT_ROOT / 'data' / 'raw' / 'geo',
    'clean'        : PROJECT_ROOT / 'data' / 'clean',
    'exports'      : PROJECT_ROOT / 'data' / 'exports',
    'db'           : PROJECT_ROOT / 'db',
    'notebooks'    : PROJECT_ROOT / 'notebooks',
    'models'       : PROJECT_ROOT / 'models',
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

geo_layers = {}

print("✅ Google Drive mounted.")
print(f"   Project root → {PROJECT_ROOT}")
print(f"   Geo folder   → {DIRS['raw_geo']}")

print("\n── Borough Boundaries ──")

borough_path = DIRS['raw_geo'] / 'borough_boundaries.geojson'

if borough_path.exists():
    gdf_boroughs = gpd.read_file(borough_path)
    geo_layers['borough_boundaries'] = gdf_boroughs
    print(f"   ✅ Loaded from Drive: {len(gdf_boroughs)} boroughs")

else:
    print("   Not cached — fetching from Census TIGER...")
    import zipfile

    TIGER_COUNTIES_URL = (
        'https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/'
        'tl_2023_us_county.zip'
    )
    NYC_BOROUGH_FIPS = {
        '36005': 'Bronx',
        '36047': 'Brooklyn',
        '36061': 'Manhattan',
        '36081': 'Queens',
        '36085': 'Staten Island',
    }

    try:
        resp = requests.get(TIGER_COUNTIES_URL, timeout=90)
        resp.raise_for_status()
        tmp_dir = DIRS['raw_geo'] / '_tmp_counties'
        tmp_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            zf.extractall(tmp_dir)
        shp = list(tmp_dir.glob('*.shp'))[0]
        gdf_all = gpd.read_file(shp).to_crs(epsg=4326)
        gdf_all['GEOID'] = gdf_all['STATEFP'] + gdf_all['COUNTYFP']
        gdf_boroughs = gdf_all[
            gdf_all['GEOID'].isin(NYC_BOROUGH_FIPS.keys())
        ].copy()
        gdf_boroughs['borough'] = gdf_boroughs['GEOID'].map(NYC_BOROUGH_FIPS)
        gdf_boroughs = gdf_boroughs[['GEOID','NAME','borough','geometry']]
        gdf_boroughs.to_file(borough_path, driver='GeoJSON')
        geo_layers['borough_boundaries'] = gdf_boroughs
        print(f"   ✅ Fetched & saved to Drive: {len(gdf_boroughs)} boroughs")
        import shutil
        shutil.rmtree(tmp_dir)
    except Exception as e:
        print(f"   ❌ Borough fetch failed: {e}")

print("\n── NTA Boundaries ──")

nta_geojson_path = DIRS['raw_geo'] / 'nta_boundaries.geojson'
nta_csv_path     = DIRS['raw_geo'] / 'nta_lookup.csv'

if nta_geojson_path.exists():
    gdf_nta = gpd.read_file(nta_geojson_path)
    geo_layers['nta_boundaries'] = gdf_nta
    print(f"   ✅ Loaded from Drive: {len(gdf_nta)} NTAs")

else:
    print("   Not cached — fetching NTA lookup from NYC Open Data...")

    NTA_LOOKUP_URL = (
        'https://data.cityofnewyork.us/api/views/9nt8-h7nd'
        '/rows.csv?accessType=DOWNLOAD'
    )

    try:
        resp = requests.get(NTA_LOOKUP_URL, timeout=45)
        resp.raise_for_status()
        df_nta = pd.read_csv(io.StringIO(resp.text))
        df_nta.to_csv(nta_csv_path, index=False)
        print(f"   Raw CSV saved → {nta_csv_path.name} ({len(df_nta)} rows)")

        sample = str(df_nta['the_geom'].dropna().iloc[0]).strip()

        if sample.startswith('{'):
            def parse_geom(val):
                try:
                    return shape(json.loads(val))
                except Exception:
                    return None
            df_nta['geometry'] = df_nta['the_geom'].apply(parse_geom)
        else:
            df_nta['geometry'] = df_nta['the_geom'].apply(
                lambda x: wkt.loads(str(x)) if pd.notna(x) else None
            )

        df_nta = df_nta.dropna(subset=['geometry'])
        gdf_nta = gpd.GeoDataFrame(df_nta, geometry='geometry', crs='EPSG:4326')
        if 'the_geom' in gdf_nta.columns:
            gdf_nta = gdf_nta.drop(columns='the_geom')

        # Residential NTAs only (NTAType == 0)
        if 'NTAType' in gdf_nta.columns:
            gdf_nta = gdf_nta[
                gdf_nta['NTAType'].astype(str) == '0'
            ].copy()

        gdf_nta.to_file(nta_geojson_path, driver='GeoJSON')
        nta_csv_out = gdf_nta.copy()
        nta_csv_out['centroid_lat'] = gdf_nta.geometry.centroid.y
        nta_csv_out['centroid_lon'] = gdf_nta.geometry.centroid.x
        nta_csv_out = nta_csv_out.drop(columns='geometry')
        nta_csv_out.to_csv(DIRS['raw_geo'] / 'nta_boundaries.csv', index=False)

        geo_layers['nta_boundaries'] = gdf_nta
        print(f"   ✅ Parsed & saved to Drive: {len(gdf_nta)} residential NTAs")

        if 'BoroName' in gdf_nta.columns:
            bk = gdf_nta[gdf_nta['BoroName'].str.lower() == 'brooklyn']
            print(f"   Brooklyn NTAs: {len(bk)}")

    except Exception as e:
        print(f"   ❌ NTA fetch/parse failed: {e}")
        import traceback; traceback.print_exc()


# ────────────────────────────────────────────────
# STEP 3: Census Tracts via pygris
# Load from Drive if cached, else fetch
# ────────────────────────────────────────────────
print("\n── Census Tracts ──")

tracts_path = DIRS['raw_geo'] / 'census_tracts_geo.geojson'

if tracts_path.exists():
    gdf_tracts = gpd.read_file(tracts_path)
    geo_layers['census_tracts'] = gdf_tracts
    print(f"   ✅ Loaded from Drive: {len(gdf_tracts):,} tracts")
    print(gdf_tracts['borough'].value_counts().to_string())

else:
    print("   Not cached — fetching via pygris...")
    subprocess.run(['pip', 'install', 'pygris', '--quiet'], capture_output=True)
    from pygris import tracts

    BOROUGH_BY_COUNTY = {
        '005': 'Bronx',
        '047': 'Brooklyn',
        '061': 'Manhattan',
        '081': 'Queens',
        '085': 'Staten Island',
    }

    tract_frames = []
    for county_fips, borough in BOROUGH_BY_COUNTY.items():
        try:
            print(f"   Fetching {borough}...")
            gdf_c = tracts(state='NY', county=county_fips, year=2022, cache=False)
            gdf_c = gdf_c.to_crs(epsg=4326)
            gdf_c['borough']     = borough
            gdf_c['county_fips'] = county_fips
            gdf_c['geoid']       = (gdf_c['STATEFP']
                                    + gdf_c['COUNTYFP']
                                    + gdf_c['TRACTCE'])
            tract_frames.append(gdf_c)
            print(f"      ✅ {len(gdf_c):,} tracts")
        except Exception as e:
            print(f"      ❌ {borough}: {e}")

    if tract_frames:
        gdf_tracts = pd.concat(tract_frames, ignore_index=True)
        if gdf_tracts.crs.to_epsg() != 4326:
            gdf_tracts = gdf_tracts.to_crs(epsg=4326)

        keep = ['geoid','STATEFP','COUNTYFP','TRACTCE','NAME',
                'borough','county_fips','ALAND','geometry']
        gdf_tracts = gdf_tracts[[c for c in keep if c in gdf_tracts.columns]]

        gdf_tracts.to_file(tracts_path, driver='GeoJSON')
        tracts_csv = gdf_tracts.copy()
        tracts_csv['centroid_lat'] = gdf_tracts.geometry.centroid.y
        tracts_csv['centroid_lon'] = gdf_tracts.geometry.centroid.x
        tracts_csv = tracts_csv.drop(columns='geometry')
        tracts_csv.to_csv(
            DIRS['raw_geo'] / 'census_tracts_geo.csv', index=False
        )

        geo_layers['census_tracts'] = gdf_tracts
        print(f"\n   ✅ Saved to Drive: {len(gdf_tracts):,} total tracts")
        print(gdf_tracts['borough'].value_counts().to_string())
    else:
        print("   ❌ All tract fetches failed.")



# FINAL SUMMARY
print("\n── Geo Layer Summary ──")
if geo_layers:
    for name, gdf in geo_layers.items():
        print(f"   ✅ {name:<25} {len(gdf):>6,} features | "
              f"CRS: EPSG:{gdf.crs.to_epsg()}")
else:
    print("   ⚠️  No layers loaded.")

print("\n✅ Section 6 complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted.
   Project root → /content/drive/MyDrive/citibike_project
   Geo folder   → /content/drive/MyDrive/citibike_project/data/raw/geo

── Borough Boundaries ──
   ✅ Loaded from Drive: 5 boroughs

── NTA Boundaries ──
   ✅ Loaded from Drive: 197 NTAs

── Census Tracts ──
   ✅ Loaded from Drive: 2,327 tracts
borough
Brooklyn         805
Queens           725
Bronx            361
Manhattan        310
Staten Island    126

── Geo Layer Summary ──
   ✅ borough_boundaries             5 features | CRS: EPSG:4326
   ✅ nta_boundaries               197 features | CRS: EPSG:4326
   ✅ census_tracts              2,327 features | CRS: EPSG:4326

✅ Section 6 complete.


---
## 7. US Census ACS Demographic Data

We pull **ACS 5-Year Estimates (2022)** at the **Census Tract level** for New York County + Kings County (Brooklyn).

**Variables we need:**
| Variable | ACS Code | Why it matters |
|---|---|---|
| Total population | B01003_001E | Population density base |
| Median household income | B19013_001E | Equity / affordability lens |
| Workers who commute by bike | B08301_018E | Latent demand signal |
| Workers who use public transit | B08301_010E | Bus dependency proxy |
| No vehicle available | B08201_002E | Car-free household rate |
| Total housing units | B25001_001E | Density denominator |

> 💡 **You need a free Census API key** → get one at https://api.census.gov/data/key_signup.html (takes ~1 min). Paste it below.

In [14]:
# Census API key
CENSUS_API_KEY = 'd7e5d34674387621d423f0cc572b2cdb1f4a9211'

ACS_VARIABLES = {
    'B01003_001E': 'total_population',
    'B19013_001E': 'median_household_income',
    'B08301_018E': 'commute_by_bike',
    'B08301_010E': 'commute_by_transit',
    'B08301_001E': 'total_commuters',
    'B08201_002E': 'hh_no_vehicle',
    'B08201_001E': 'total_households',
    'B25001_001E': 'total_housing_units',
}

# NYC county FIPS codes
# 36 = New York State
NYC_COUNTIES = {
    '047': 'Brooklyn',       # Kings County
    '081': 'Queens',
    '005': 'Bronx',
    '061': 'Manhattan',      # New York County
    '085': 'Staten Island',  # Richmond County
}

print("Census config ready.")
print(f"Variables to pull: {len(ACS_VARIABLES)}")
print(f"Counties         : {list(NYC_COUNTIES.values())}")

Census config ready.
Variables to pull: 8
Counties         : ['Brooklyn', 'Queens', 'Bronx', 'Manhattan', 'Staten Island']


In [15]:
def fetch_acs_county(state_fips: str, county_fips: str, borough_name: str,
                     variables: dict, api_key: str) -> pd.DataFrame | None:
    """Fetch ACS 5-Year 2022 data at census tract level for one county."""
    var_string = ','.join(variables.keys())
    url = (
        f'https://api.census.gov/data/2022/acs/acs5'
        f'?get=NAME,{var_string}'
        f'&for=tract:*'
        f'&in=state:{state_fips}+county:{county_fips}'
        f'&key={api_key}'
    )
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        rows  = resp.json()
        cols  = rows[0]
        df    = pd.DataFrame(rows[1:], columns=cols)
        df    = df.rename(columns=variables)
        df['borough']     = borough_name
        df['county_fips'] = county_fips
        df['geoid']       = state_fips + county_fips + df['tract']
        for new_col in variables.values():
            if new_col in df.columns:
                df[new_col] = pd.to_numeric(df[new_col], errors='coerce')
        return df
    except Exception as e:
        print(f"   ⚠️  Failed for {borough_name}: {e}")
        return None


if CENSUS_API_KEY == 'YOUR_CENSUS_API_KEY_HERE':
    print("⚠️  Census API key not set. Skipping live pull.")
    print("   → Get your free key at: https://api.census.gov/data/key_signup.html")
    print("   → Paste it into the CENSUS_API_KEY variable above and re-run.")
    df_census = None
else:
    print("Fetching ACS data for all NYC counties...\n")
    frames = []
    for county_fips, borough_name in NYC_COUNTIES.items():
        print(f"   {borough_name} (county {county_fips})...")
        df_c = fetch_acs_county('36', county_fips, borough_name,
                                ACS_VARIABLES, CENSUS_API_KEY)
        if df_c is not None:
            frames.append(df_c)
            print(f"   ✅ {len(df_c):,} tracts")

    df_census = pd.concat(frames, ignore_index=True) if frames else None

    if df_census is not None:
        df_census['transit_share']  = df_census['commute_by_transit'] / df_census['total_commuters'].replace(0, np.nan)
        df_census['bike_share'] = df_census['commute_by_bike'] / df_census['total_commuters'].replace(0, np.nan)
        df_census['no_vehicle_rate'] = df_census['hh_no_vehicle'] / df_census['total_households'].replace(0, np.nan)

        out_path = DIRS['raw_census'] / 'acs_nyc_tracts.csv'
        df_census.to_csv(out_path, index=False)
        print(f"\n✅ Census data saved → {out_path}")
        print(f"   Total tracts : {len(df_census):,}")
        print(f"\nBy borough:")
        print(df_census.groupby('borough')['total_population'].sum().apply(lambda x: f"{x:,.0f}").to_string())
        df_census.head(3)

Fetching ACS data for all NYC counties...

   Brooklyn (county 047)...
   ✅ 805 tracts
   Queens (county 081)...
   ✅ 725 tracts
   Bronx (county 005)...
   ✅ 361 tracts
   Manhattan (county 061)...
   ✅ 310 tracts
   Staten Island (county 085)...
   ✅ 126 tracts

✅ Census data saved → /content/drive/MyDrive/citibike_project/data/raw/census/acs_nyc_tracts.csv
   Total tracts : 2,327

By borough:
borough
Bronx            1,443,229
Brooklyn         2,679,620
Manhattan        1,645,867
Queens           2,360,826
Staten Island      492,925


---
## 8. DuckDB Setup & Initial Table Creation

Now we load everything into a persistent **DuckDB database** — this is what enables your SQL exploration in Phase 2.

DuckDB reads CSVs directly into tables with zero extra setup. You query them like a real database, and it's fast enough that 10M+ trip rows won't feel slow.

In [16]:
# SECTION 8 — DuckDB Setup & Table Creation

import io, zipfile, requests, duckdb
import pandas as pd
from pathlib import Path

DB_PATH = str(DIRS['db'] / 'citibike.duckdb')
con     = duckdb.connect(DB_PATH)
print(f"✅ DuckDB connected → {DB_PATH}\n")

def ensure_stations() -> str:
    """Return path to stations_live.csv, fetching from GBFS if missing."""
    path = DIRS['raw_stations'] / 'stations_live.csv'
    if path.exists():
        print(f"   ✅ stations_live.csv on Drive")
        return str(path)

    print("   stations_live.csv missing — fetching from GBFS...")
    GBFS_INFO   = 'https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json'
    GBFS_STATUS = 'https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_status.json'

    def fetch(url):
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        return pd.DataFrame(r.json()['data']['stations'])

    df_info   = fetch(GBFS_INFO)
    df_status = fetch(GBFS_STATUS)

    info_cols   = [c for c in ['station_id','name','short_name','lat','lon',
                                'capacity','region_id'] if c in df_info.columns]
    status_cols = [c for c in ['station_id','num_bikes_available',
                                'num_docks_available','num_ebikes_available',
                                'is_installed','is_renting','last_reported']
                   if c in df_status.columns]

    df = df_info[info_cols].merge(df_status[status_cols], on='station_id', how='left')
    if 'last_reported' in df.columns:
        df['last_reported'] = pd.to_datetime(df['last_reported'], unit='s')

    def assign_borough(lat, lon):
        if 40.495 <= lat <= 40.739 and -74.260 <= lon <= -73.833: return 'Brooklyn'
        if 40.700 <= lat <= 40.800 and -74.020 <= lon <= -73.907: return 'Manhattan'
        if 40.541 <= lat <= 40.670 and -73.833 <= lon <= -73.700: return 'Queens'
        if 40.785 <= lat <= 40.915 and -73.933 <= lon <= -73.765: return 'Bronx'
        if 40.495 <= lat <= 40.650 and -74.260 <= lon <= -74.034: return 'Staten Island'
        return 'Other'

    df['borough'] = df.apply(lambda r: assign_borough(r['lat'], r['lon']), axis=1)
    df.to_csv(path, index=False)
    print(f"   ✅ Fetched & saved: {len(df):,} stations")
    return str(path)


def ensure_trips() -> list[str]:
    """
    Return list of trip CSV paths on Drive.
    Downloads Jan–Feb 2024 if no CSVs found.
    Expand TARGET_MONTHS below to add more months.
    """
    existing = list(DIRS['raw_trips'].glob('*.csv'))
    if existing:
        print(f"   ✅ {len(existing)} trip file(s) on Drive:")
        for f in existing:
            print(f"      {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
        return [str(f) for f in existing]

    print("   No trip files on Drive — downloading now...")

    TARGET_MONTHS = ['202401', '202402']   # can expand as needed
    TRIP_BASE     = 'https://s3.amazonaws.com/tripdata/'
    downloaded    = []

    for yyyymm in TARGET_MONTHS:
        print(f"   Trying {yyyymm}...")
        candidates = [
            f'{yyyymm}-citibike-tripdata.csv.zip',
            f'{yyyymm}-citibike-tripdata.zip',
        ]
        success = False
        for filename in candidates:
            try:
                url  = TRIP_BASE + filename
                resp = requests.get(url, stream=True, timeout=120)
                if resp.status_code == 404:
                    continue
                resp.raise_for_status()

                zip_path = DIRS['raw_trips'] / filename
                with open(zip_path, 'wb') as f:
                    for chunk in resp.iter_content(chunk_size=1024*1024):
                        f.write(chunk)

                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(DIRS['raw_trips'])
                zip_path.unlink()

                csvs = sorted(
                    DIRS['raw_trips'].glob('*.csv'),
                    key=lambda p: p.stat().st_mtime
                )
                if csvs:
                    latest = csvs[-1]
                    downloaded.append(str(latest))
                    print(f"      ✅ {latest.name}  "
                          f"({latest.stat().st_size/1e6:.1f} MB)")
                success = True
                break

            except Exception as e:
                print(f"      ⚠️  {filename}: {e}")
                if zip_path.exists():
                    zip_path.unlink()

        if not success:
            print(f"      ❌ Could not download {yyyymm}")

    return downloaded


def ensure_bus_stops() -> str | None:
    """Return path to bus_stops.csv, fetching from MTA GTFS if missing."""
    path = DIRS['raw_mta'] / 'bus_stops.csv'
    if path.exists():
        print(f"   ✅ bus_stops.csv on Drive")
        return str(path)

    print("   bus_stops.csv missing — fetching from MTA GTFS...")

    MTA_FEEDS = {
        'brooklyn'     : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_b.zip',
        'bronx'        : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_bx.zip',
        'manhattan'    : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_m.zip',
        'queens'       : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_q.zip',
        'staten_island': 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_si.zip',
        'mta_busco'    : 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_busco.zip',
    }

    frames = []
    for borough, url in MTA_FEEDS.items():
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
                if 'stops.txt' not in zf.namelist():
                    continue
                with zf.open('stops.txt') as f:
                    df = pd.read_csv(f, dtype=str)
            df['borough_feed'] = borough
            frames.append(df)
            print(f"      ✅ {borough}: {len(df):,} stops")
        except Exception as e:
            print(f"      ❌ {borough}: {e}")

    if not frames:
        print("   ❌ All MTA feeds failed.")
        return None

    df_stops = pd.concat(frames, ignore_index=True)
    df_stops  = df_stops.rename(columns={'stop_lat':'lat','stop_lon':'lon'})
    df_stops['lat'] = pd.to_numeric(df_stops['lat'], errors='coerce')
    df_stops['lon'] = pd.to_numeric(df_stops['lon'], errors='coerce')
    df_stops  = df_stops.drop_duplicates(subset='stop_id')
    df_stops  = df_stops[
        df_stops['lat'].between(40.45, 40.95) &
        df_stops['lon'].between(-74.30, -73.65)
    ]
    df_stops.to_csv(path, index=False)
    print(f"   ✅ Saved: {len(df_stops):,} unique stops → {path.name}")
    return str(path)


def ensure_census() -> str | None:
    """Return path to acs_nyc_tracts.csv. Must be fetched via Census API — alerts if missing."""
    path = DIRS['raw_census'] / 'acs_nyc_tracts.csv'
    if path.exists():
        print(f"   ✅ acs_nyc_tracts.csv on Drive")
        return str(path)
    print("   ⚠️  acs_nyc_tracts.csv missing.")
    print("       Re-run Section 7 with your Census API key to generate it.")
    return None

# Stations
print("── Stations ──")
stations_csv = ensure_stations()
con.execute("DROP TABLE IF EXISTS stations")
con.execute(f"""
    CREATE TABLE stations AS
    SELECT * FROM read_csv_auto('{stations_csv}')
""")
n = con.execute("SELECT COUNT(*) AS n FROM stations").fetchdf()['n'][0]
print(f"   Loaded: {n:,} rows\n")

# Trips
print("── Trips ──")
trip_paths = ensure_trips()
if trip_paths:
    trips_glob = str(DIRS['raw_trips'] / '*.csv')
    con.execute("DROP TABLE IF EXISTS trips")
    con.execute(f"""
        CREATE TABLE trips AS
        SELECT * FROM read_csv_auto(
            '{trips_glob}',
            union_by_name = true,
            ignore_errors = true
        )
    """)
    n = con.execute("SELECT COUNT(*) AS n FROM trips").fetchdf()['n'][0]
    print(f"   Loaded: {n:,} rows\n")
else:
    print("   ❌ Trips not loaded.\n")

# Bus Stops
print("── Bus Stops ──")
bus_path = ensure_bus_stops()
if bus_path:
    con.execute("DROP TABLE IF EXISTS bus_stops")
    con.execute(f"""
        CREATE TABLE bus_stops AS
        SELECT * FROM read_csv_auto('{bus_path}')
    """)
    n = con.execute("SELECT COUNT(*) AS n FROM bus_stops").fetchdf()['n'][0]
    print(f"   Loaded: {n:,} rows\n")

# Census Tracts
print("── Census Tracts ──")
census_path = ensure_census()
if census_path:
    con.execute("DROP TABLE IF EXISTS census_tracts")
    con.execute(f"""
        CREATE TABLE census_tracts AS
        SELECT * FROM read_csv_auto('{census_path}')
    """)
    n = con.execute("SELECT COUNT(*) AS n FROM census_tracts").fetchdf()['n'][0]
    print(f"   Loaded: {n:,} rows\n")

# SUMMARY
print("=" * 50)
print("  SECTION 8 SUMMARY")
print("=" * 50)
all_tables = con.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'main'
""").fetchdf()['table_name'].tolist()

for t in ['stations', 'trips', 'bus_stops', 'census_tracts']:
    if t in all_tables:
        n = con.execute(f"SELECT COUNT(*) AS n FROM {t}").fetchdf()['n'][0]
        print(f"  ✅  {t:<20} {n:>10,} rows")
    else:
        print(f"  ❌  {t:<20} NOT LOADED")

con.close()
print(f"\n✅ DuckDB saved → {DB_PATH}")

✅ DuckDB connected → /content/drive/MyDrive/citibike_project/db/citibike.duckdb

── Stations ──
   ✅ stations_live.csv on Drive
   Loaded: 2,412 rows

── Trips ──
   ✅ 5 trip file(s) on Drive:
      202401-citibike-tripdata_2.csv  (173.7 MB)
      202401-citibike-tripdata_1.csv  (195.4 MB)
      202402-citibike-tripdata_1.csv  (195.6 MB)
      202402-citibike-tripdata_2.csv  (195.4 MB)
      202402-citibike-tripdata_3.csv  (23.7 MB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Loaded: 4,009,586 rows

── Bus Stops ──
   ✅ bus_stops.csv on Drive
   Loaded: 13,370 rows

── Census Tracts ──
   ✅ acs_nyc_tracts.csv on Drive
   Loaded: 2,327 rows

  SECTION 8 SUMMARY
  ✅  stations                  2,412 rows
  ✅  trips                 4,009,586 rows
  ✅  bus_stops                13,370 rows
  ✅  census_tracts             2,327 rows

✅ DuckDB saved → /content/drive/MyDrive/citibike_project/db/citibike.duckdb


---
## 9. Phase 1 Validation & Summary

Let's run a quick end-to-end check to confirm everything is in place before moving to Phase 2.

In [17]:
# SECTION 9 — Phase 1 Validation & Summary
import duckdb
from pathlib import Path

DB_PATH = str(DIRS['db'] / 'citibike.duckdb')
con     = duckdb.connect(DB_PATH)

print("=" * 55)
print("  PHASE 1 — INGESTION SUMMARY")
print("=" * 55)

expected = ['stations', 'trips', 'bus_stops', 'census_tracts']
all_tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
""").fetchdf()['table_name'].tolist()

print("\nTables:")
for t in expected:
    if t in all_tables:
        n = con.execute(f"SELECT COUNT(*) AS n FROM {t}").fetchdf()['n'][0]
        print(f"  ✅  {t:<20} {n:>10,} rows")
    else:
        print(f"  ❌  {t:<20} NOT LOADED")

print("\nDrive file inventory:")
for dir_name, dir_path in DIRS.items():
    files = [f for f in dir_path.glob('*') if f.is_file()]
    if files:
        total_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f"  {dir_name:<15} {len(files):>2} file(s)  {total_mb:>7.1f} MB")

print("\nTop 5 busiest start stations in loaded trip data:")
try:
    display(con.execute("""
        SELECT
            start_station_name,
            COUNT(*)                                        AS trip_count,
            ROUND(AVG(
                epoch(CAST(ended_at AS TIMESTAMP)
                    - CAST(started_at AS TIMESTAMP)) / 60.0
            ), 1)                                           AS avg_duration_min
        FROM trips
        WHERE start_station_name IS NOT NULL
        GROUP BY start_station_name
        ORDER BY trip_count DESC
        LIMIT 5
    """).fetchdf())
except Exception as e:
    print(f"  ⚠️  Teaser query failed: {e}")

print(f"\nDuckDB path : {DB_PATH}")
print("=" * 55)
print("\n🚀 Phase 1 complete. Ready for Phase 2: SQL Exploration.")

# ── Close cleanly — this is the ONE place we close in Phase 1
con.close()
print("✅ DuckDB connection closed.")

  PHASE 1 — INGESTION SUMMARY

Tables:
  ✅  stations                  2,412 rows
  ✅  trips                 4,009,586 rows
  ✅  bus_stops                13,370 rows
  ✅  census_tracts             2,327 rows

Drive file inventory:
  raw_trips        5 file(s)    783.7 MB
  raw_stations     1 file(s)      0.3 MB
  raw_mta          1 file(s)      0.9 MB
  raw_census       1 file(s)      0.4 MB
  raw_geo          6 file(s)     10.7 MB
  exports         10 file(s)      0.6 MB
  db               1 file(s)    416.3 MB

Top 5 busiest start stations in loaded trip data:


,start_station_name,trip_count,avg_duration_min
0,W 21 St & 6 Ave,17899,9.70
1,1 Ave & E 68 St,13937,12.80
2,8 Ave & W 31 St,13780,12.40
3,Lafayette St & E 8 St,13060,10.60
4,W 31 St & 7 Ave,12649,10.50



DuckDB path : /content/drive/MyDrive/citibike_project/db/citibike.duckdb

🚀 Phase 1 complete. Ready for Phase 2: SQL Exploration.
✅ DuckDB connection closed.


# 🚲 CitiBike Brooklyn Expansion Analysis
## Phase 2: SQL Exploration with DuckDB

**What this notebook does:**
This is your core analytical layer. Every query here serves a direct purpose in the final dashboard or downstream model — nothing is exploratory for its own sake. By the end, you'll have clean, exported tables ready for Phase 3 (gap scoring) and Phase 4 (financial modeling).

**DuckDB in Colab — the key things to know before you start:**
- DuckDB runs entirely in-process — no server, no connection string, no setup beyond `import duckdb`
- `.execute(sql).fetchdf()` runs a query and returns a pandas DataFrame — this is your main pattern
- You can mix Python variables into SQL using f-strings (shown throughout)
- DuckDB supports nearly all standard SQL including window functions, CTEs, QUALIFY, and PIVOT
- The `.duckdb` file written in Phase 1 persists your tables — you just reconnect to it

**Notebook sections:**
1. Reconnect to DuckDB & orient yourself
2. Schema validation & data quality checks
3. Station network analysis
4. Trip pattern analysis
5. Borough & neighborhood demand profiling
6. Transit dependency analysis (MTA + CitiBike overlap)
7. Coverage gap identification
8. Census demographic overlay
9. Sheepshead Bay deep dive
10. Export clean tables for Phase 3

In [18]:
import duckdb
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_rows', 60)

# ── Point to the same DB file written in Phase 1
# If you mounted Google Drive, adjust this path to match
PROJECT_ROOT = Path('/content/citibike_project')
DB_PATH      = str(PROJECT_ROOT / 'db' / 'citibike.duckdb')
EXPORTS_DIR  = PROJECT_ROOT / 'data' / 'exports'
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(DB_PATH)
print(f'✅ Connected to DuckDB → {DB_PATH}')

# ── Helper: run a query and display result with a label
# Use this throughout the notebook to keep things clean
def q(sql: str, label: str = '') -> pd.DataFrame:
    """Execute SQL and return a DataFrame. Prints label if provided."""
    if label:
        print(f'\n── {label} ──')
    df = con.execute(sql).fetchdf()
    return df

print('   Helper function q() registered.')

✅ Connected to DuckDB → /content/citibike_project/db/citibike.duckdb
   Helper function q() registered.


---
## 0. Session Setup
**Run this cell first, every time you open the notebook.**
Mounts Drive, rebuilds all path references, connects DuckDB, and loads every table.
On subsequent runs it skips re-fetching anything already in the DuckDB file.

In [19]:
# Install and imports
!pip install duckdb --quiet

import io, zipfile, warnings, requests
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_rows', 80)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Project paths — all on Drive so they survive restarts
PROJECT_ROOT = Path('/content/drive/MyDrive/citibike_project')
DIRS = {
    'raw_trips'   : PROJECT_ROOT / 'data' / 'raw' / 'trips',
    'raw_stations': PROJECT_ROOT / 'data' / 'raw' / 'stations',
    'raw_mta'     : PROJECT_ROOT / 'data' / 'raw' / 'mta',
    'raw_census'  : PROJECT_ROOT / 'data' / 'raw' / 'census',
    'raw_geo'     : PROJECT_ROOT / 'data' / 'raw' / 'geo',
    'clean'       : PROJECT_ROOT / 'data' / 'clean',
    'exports'     : PROJECT_ROOT / 'data' / 'exports',
    'db'          : PROJECT_ROOT / 'db',
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

EXPORTS_DIR = DIRS['exports']
DB_PATH     = str(DIRS['db'] / 'citibike.duckdb')

# Connect to DuckDB
# con stays open for the entire notebook
# Do NOT call con.close() until the very last cell
con = duckdb.connect(DB_PATH)
print(f'Connected to DuckDB: {DB_PATH}')

# Helper: run SQL and return a DataFrame with optional label
def q(sql: str, label: str = '') -> pd.DataFrame:
    if label:
        print(f'\n-- {label} --')
    return con.execute(sql).fetchdf()

# Table loading helpers
def table_exists(name: str) -> bool:
    tbls = con.execute("""
        SELECT table_name FROM information_schema.tables
        WHERE table_schema = 'main'
    """).fetchdf()['table_name'].tolist()
    return name in tbls

def load_csv_table(table: str, csv_path: Path, force: bool = False):
    if table_exists(table) and not force:
        n = con.execute(f'SELECT COUNT(*) AS n FROM {table}').fetchdf()['n'][0]
        print(f'   {table:<22} already loaded  {n:>8,} rows')
        return
    if not csv_path.exists():
        print(f'   {table:<22} CSV not found: {csv_path.name}')
        return
    con.execute(f'DROP TABLE IF EXISTS {table}')
    con.execute(f"""
        CREATE TABLE {table} AS
        SELECT * FROM read_csv_auto('{str(csv_path)}', ignore_errors=true)
    """)
    n = con.execute(f'SELECT COUNT(*) AS n FROM {table}').fetchdf()['n'][0]
    print(f'   {table:<22} loaded           {n:>8,} rows')

print('\nLoading tables...')

# Stations
load_csv_table('stations',      DIRS['raw_stations'] / 'stations_live.csv')

# Trips: glob across all monthly CSVs
if not table_exists('trips'):
    trip_files = list(DIRS['raw_trips'].glob('*.csv'))
    if trip_files:
        trips_glob = str(DIRS['raw_trips'] / '*.csv')
        con.execute('DROP TABLE IF EXISTS trips')
        con.execute(f"""
            CREATE TABLE trips AS
            SELECT * FROM read_csv_auto(
                '{trips_glob}',
                union_by_name = true,
                ignore_errors = true
            )
        """)
        n = con.execute('SELECT COUNT(*) AS n FROM trips').fetchdf()['n'][0]
        print(f'   {"trips":<22} loaded           {n:>8,} rows ({len(trip_files)} file(s))')
    else:
        print(f'   {"trips":<22} no CSV files found in raw_trips')
else:
    n = con.execute('SELECT COUNT(*) AS n FROM trips').fetchdf()['n'][0]
    print(f'   {"trips":<22} already loaded  {n:>8,} rows')

# Remaining tables
load_csv_table('bus_stops',     DIRS['raw_mta']    / 'bus_stops.csv')
load_csv_table('census_tracts', DIRS['raw_census'] / 'acs_nyc_tracts.csv')
load_csv_table('tracts_geo',    DIRS['raw_geo']    / 'census_tracts_geo.csv')
load_csv_table('nta_lookup',    DIRS['raw_geo']    / 'nta_boundaries.csv')

print('\nSession ready. Use q(sql) to run queries throughout the notebook.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected to DuckDB: /content/drive/MyDrive/citibike_project/db/citibike.duckdb

Loading tables...
   stations               already loaded     2,412 rows
   trips                  already loaded  4,009,586 rows
   bus_stops              already loaded    13,370 rows
   census_tracts          already loaded     2,327 rows
   tracts_geo             already loaded     2,327 rows
   nta_lookup             already loaded       197 rows

Session ready. Use q(sql) to run queries throughout the notebook.


---
## 1. Schema Orientation
Before writing queries, confirm exactly what columns exist and their inferred types.
DuckDB inferred all types from CSV headers and values — worth verifying before the heavy queries.

In [20]:
# What tables exist and their approximate sizes?
display(q("""
    SELECT table_name, estimated_size AS approx_rows
    FROM duckdb_tables()
    ORDER BY table_name
""", 'Tables in citibike.duckdb'))


-- Tables in citibike.duckdb --


,table_name,approx_rows
0,bus_stops,13370
1,census_tracts,2327
2,nta_lookup,197
3,stations,2412
4,tracts_geo,2327
5,trips,4009586


In [21]:
# DESCRIBE each table: column names and inferred types
# Pay attention to trips: CitiBike uses started_at/ended_at (strings) and start_lng
for tbl in ['stations', 'trips', 'bus_stops', 'census_tracts', 'tracts_geo']:
    try:
        print(f'\n-- {tbl} --')
        display(con.execute(f'DESCRIBE {tbl}').fetchdf()[['column_name','column_type']])
    except Exception as e:
        print(f'   Not loaded: {e}')


-- stations --


,column_name,column_type
0,station_id,VARCHAR
1,name,VARCHAR
2,short_name,VARCHAR
3,lat,DOUBLE
4,lon,DOUBLE
5,capacity,BIGINT
6,region_id,BIGINT
7,num_bikes_available,BIGINT
8,num_docks_available,BIGINT
9,num_ebikes_available,BIGINT



-- trips --


,column_name,column_type
0,ride_id,VARCHAR
1,rideable_type,VARCHAR
2,started_at,TIMESTAMP
3,ended_at,TIMESTAMP
4,start_station_name,VARCHAR
5,start_station_id,VARCHAR
6,end_station_name,VARCHAR
7,end_station_id,VARCHAR
8,start_lat,DOUBLE
9,start_lng,DOUBLE



-- bus_stops --


,column_name,column_type
0,stop_id,BIGINT
1,stop_name,VARCHAR
2,stop_desc,VARCHAR
3,lat,DOUBLE
4,lon,DOUBLE
5,zone_id,VARCHAR
6,stop_url,VARCHAR
7,location_type,BIGINT
8,parent_station,VARCHAR
9,borough_feed,VARCHAR



-- census_tracts --


,column_name,column_type
0,NAME,VARCHAR
1,total_population,BIGINT
2,median_household_income,BIGINT
3,commute_by_bike,BIGINT
4,commute_by_transit,BIGINT
5,total_commuters,BIGINT
6,hh_no_vehicle,BIGINT
7,total_households,BIGINT
8,total_housing_units,BIGINT
9,state,BIGINT



-- tracts_geo --


,column_name,column_type
0,geoid,BIGINT
1,STATEFP,BIGINT
2,COUNTYFP,VARCHAR
3,TRACTCE,VARCHAR
4,NAME,DOUBLE
5,borough,VARCHAR
6,county_fips,VARCHAR
7,ALAND,BIGINT
8,centroid_lat,DOUBLE
9,centroid_lon,DOUBLE


In [22]:
# Sample 3 rows from trips to confirm live schema before writing any trip queries
display(q('SELECT * FROM trips LIMIT 3', 'Trip data sample'))


-- Trip data sample --


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,8E865410DBDE0CA9,electric_bike,2024-01-01 13:00:04.563,2024-01-01 13:04:04.652,3 St & 3 Ave,4028.03,Carroll St & Smith St,4225.14,40.68,-73.99,40.68,-73.99,casual
1,0403D0B3FC9CA77D,electric_bike,2024-01-08 19:36:43.520,2024-01-08 19:53:16.266,Franklin Ave & St Marks Ave,4107.05,Bedford Ave & Bergen St,4066.15,40.68,-73.96,40.68,-73.95,casual
2,F6DE7BB42FF550BE,electric_bike,2024-01-12 15:00:41.580,2024-01-12 15:36:29.622,W 67 St & Broadway,7116.04,Central Park W & W 103 St,7577.27,40.77,-73.98,40.80,-73.96,casual


---
## 2. Data Quality and Clean View

Every downstream query uses `trips_clean` -- a DuckDB VIEW that filters bad rows
and casts columns to the right types once, so we never repeat that logic.

**VIEW vs TABLE in DuckDB:** A VIEW stores the query definition, not the data.
Every time you query `trips_clean`, DuckDB re-runs the SQL against `trips`.
No extra storage, always reflects source data, negligible performance cost at this scale.

In [23]:
# Null audit on the columns used in every downstream query
# COUNT(*) FILTER (WHERE ...) is standard SQL, cleaner than COUNT(CASE WHEN ... END)
# Supported in DuckDB, PostgreSQL, and BigQuery
display(q("""
    SELECT
        COUNT(*)                                                        AS total_rows,
        COUNT(*) FILTER (WHERE ride_id          IS NULL)                AS null_ride_id,
        COUNT(*) FILTER (WHERE started_at       IS NULL)                AS null_started_at,
        COUNT(*) FILTER (WHERE ended_at         IS NULL)                AS null_ended_at,
        COUNT(*) FILTER (WHERE start_station_id IS NULL
                            OR start_station_id = '')                   AS null_start_station,
        COUNT(*) FILTER (WHERE member_casual    IS NULL)                AS null_member_casual,
        ROUND(
            COUNT(*) FILTER (WHERE start_station_id IS NULL
                                OR start_station_id = '')
            * 100.0 / COUNT(*)
        , 2)                                                            AS pct_null_start_station
    FROM trips
""", 'Trips null audit'))


-- Trips null audit --


,total_rows,null_ride_id,null_started_at,null_ended_at,null_start_station,null_member_casual,pct_null_start_station
0,4009586,0,0,0,3026,0,0.08


In [24]:
# Trip duration distribution
# CitiBike stores started_at and ended_at as timestamp strings
# epoch() converts a DuckDB interval to seconds; divide by 60 gives minutes
# PERCENTILE_CONT is standard SQL, works in DuckDB, Postgres, Snowflake
display(q("""
    WITH dur AS (
        SELECT
            epoch(
                CAST(ended_at AS TIMESTAMP)
              - CAST(started_at AS TIMESTAMP)
            ) / 60.0 AS duration_min
        FROM trips
        WHERE started_at IS NOT NULL AND ended_at IS NOT NULL
    )
    SELECT
        COUNT(*)                                                AS total_trips,
        ROUND(MIN(duration_min), 2)                             AS min_min,
        ROUND(MAX(duration_min), 2)                             AS max_min,
        ROUND(AVG(duration_min), 2)                             AS avg_min,
        ROUND(PERCENTILE_CONT(0.5)
            WITHIN GROUP (ORDER BY duration_min), 2)            AS median_min,
        COUNT(*) FILTER (WHERE duration_min <  0)               AS negative,
        COUNT(*) FILTER (WHERE duration_min <  1)               AS under_1_min,
        COUNT(*) FILTER (WHERE duration_min >  60)              AS over_60_min,
        COUNT(*) FILTER (WHERE duration_min >  1440)            AS over_24_hrs
    FROM dur
""", 'Trip duration distribution'))
print('\nunder_1_min = likely dock errors. over_24_hrs = lost bikes. Both filtered in trips_clean.')


-- Trip duration distribution --


,total_trips,min_min,max_min,avg_min,median_min,negative,under_1_min,over_60_min,over_24_hrs
0,4009586,0.41,1500.49,11.59,7.85,0,1,29280,1267



under_1_min = likely dock errors. over_24_hrs = lost bikes. Both filtered in trips_clean.


In [25]:
# Create trips_clean VIEW
# This is the table every downstream query uses.
# Note: CitiBike uses start_lng not start_lon

con.execute('DROP VIEW IF EXISTS trips_clean')
con.execute("""
    CREATE VIEW trips_clean AS
    SELECT
        ride_id,
        rideable_type,
        CAST(started_at AS TIMESTAMP)                           AS started_at,
        CAST(ended_at   AS TIMESTAMP)                           AS ended_at,
        epoch(
            CAST(ended_at   AS TIMESTAMP)
          - CAST(started_at AS TIMESTAMP)
        ) / 60.0                                                AS duration_min,
        start_station_id,
        start_station_name,
        end_station_id,
        end_station_name,
        start_lat,
        start_lng,
        end_lat,
        end_lng,
        member_casual,
        EXTRACT(year  FROM CAST(started_at AS TIMESTAMP))       AS trip_year,
        EXTRACT(month FROM CAST(started_at AS TIMESTAMP))       AS trip_month,
        EXTRACT(dow   FROM CAST(started_at AS TIMESTAMP))       AS trip_dow,
        EXTRACT(hour  FROM CAST(started_at AS TIMESTAMP))       AS trip_hour
    FROM trips
    WHERE
        started_at IS NOT NULL
        AND ended_at IS NOT NULL
        AND epoch(
                CAST(ended_at   AS TIMESTAMP)
              - CAST(started_at AS TIMESTAMP)
            ) / 60.0 BETWEEN 1 AND 1440
        AND start_station_id IS NOT NULL
        AND start_station_id != ''
""")

raw   = con.execute('SELECT COUNT(*) AS n FROM trips').fetchdf()['n'][0]
clean = con.execute('SELECT COUNT(*) AS n FROM trips_clean').fetchdf()['n'][0]
print(f'trips_clean VIEW created')
print(f'  Raw rows    : {raw:>10,}')
print(f'  Clean rows  : {clean:>10,}')
print(f'  Filtered    : {raw-clean:>10,}  ({(raw-clean)/raw*100:.1f}%)')

trips_clean VIEW created
  Raw rows    :  4,009,586
  Clean rows  :  4,005,293
  Filtered    :      4,293  (0.1%)


In [26]:
# Null audit on the columns used in every downstream query
# COUNT(*) FILTER (WHERE ...) is standard SQL, cleaner than COUNT(CASE WHEN ... END)
# Supported in DuckDB, PostgreSQL, and BigQuery
display(q("""
    SELECT
        COUNT(*)                                                        AS total_rows,
        COUNT(*) FILTER (WHERE ride_id          IS NULL)                AS null_ride_id,
        COUNT(*) FILTER (WHERE started_at       IS NULL)                AS null_started_at,
        COUNT(*) FILTER (WHERE ended_at         IS NULL)                AS null_ended_at,
        COUNT(*) FILTER (WHERE start_station_id IS NULL
                            OR start_station_id = '')                   AS null_start_station,
        COUNT(*) FILTER (WHERE member_casual    IS NULL)                AS null_member_casual,
        ROUND(
            COUNT(*) FILTER (WHERE start_station_id IS NULL
                                OR start_station_id = '')
            * 100.0 / COUNT(*)
        , 2)                                                            AS pct_null_start_station
    FROM trips
""", 'Trips null audit'))


-- Trips null audit --


,total_rows,null_ride_id,null_started_at,null_ended_at,null_start_station,null_member_casual,pct_null_start_station
0,4009586,0,0,0,3026,0,0.08


In [27]:
# Trip duration distribution
# CitiBike stores started_at and ended_at as timestamp strings
# epoch() converts a DuckDB interval to seconds; divide by 60 gives minutes
# PERCENTILE_CONT is standard SQL, works in DuckDB, Postgres, Snowflake
display(q("""
    WITH dur AS (
        SELECT
            epoch(
                CAST(ended_at AS TIMESTAMP)
              - CAST(started_at AS TIMESTAMP)
            ) / 60.0 AS duration_min
        FROM trips
        WHERE started_at IS NOT NULL AND ended_at IS NOT NULL
    )
    SELECT
        COUNT(*)                                                AS total_trips,
        ROUND(MIN(duration_min), 2)                             AS min_min,
        ROUND(MAX(duration_min), 2)                             AS max_min,
        ROUND(AVG(duration_min), 2)                             AS avg_min,
        ROUND(PERCENTILE_CONT(0.5)
            WITHIN GROUP (ORDER BY duration_min), 2)            AS median_min,
        COUNT(*) FILTER (WHERE duration_min <  0)               AS negative,
        COUNT(*) FILTER (WHERE duration_min <  1)               AS under_1_min,
        COUNT(*) FILTER (WHERE duration_min >  60)              AS over_60_min,
        COUNT(*) FILTER (WHERE duration_min >  1440)            AS over_24_hrs
    FROM dur
""", 'Trip duration distribution'))
print('\nunder_1_min = likely dock errors. over_24_hrs = lost bikes. Both filtered in trips_clean.')


-- Trip duration distribution --


,total_trips,min_min,max_min,avg_min,median_min,negative,under_1_min,over_60_min,over_24_hrs
0,4009586,0.41,1500.49,11.59,7.85,0,1,29280,1267



under_1_min = likely dock errors. over_24_hrs = lost bikes. Both filtered in trips_clean.


In [28]:
# Create trips_clean VIEW
# This is the table every downstream query uses.
# Note: CitiBike uses start_lng not start_lon

con.execute('DROP VIEW IF EXISTS trips_clean')
con.execute("""
    CREATE VIEW trips_clean AS
    SELECT
        ride_id,
        rideable_type,
        CAST(started_at AS TIMESTAMP)                           AS started_at,
        CAST(ended_at   AS TIMESTAMP)                           AS ended_at,
        epoch(
            CAST(ended_at   AS TIMESTAMP)
          - CAST(started_at AS TIMESTAMP)
        ) / 60.0                                                AS duration_min,
        start_station_id,
        start_station_name,
        end_station_id,
        end_station_name,
        start_lat,
        start_lng,
        end_lat,
        end_lng,
        member_casual,
        EXTRACT(year  FROM CAST(started_at AS TIMESTAMP))       AS trip_year,
        EXTRACT(month FROM CAST(started_at AS TIMESTAMP))       AS trip_month,
        EXTRACT(dow   FROM CAST(started_at AS TIMESTAMP))       AS trip_dow,
        EXTRACT(hour  FROM CAST(started_at AS TIMESTAMP))       AS trip_hour
    FROM trips
    WHERE
        started_at IS NOT NULL
        AND ended_at IS NOT NULL
        AND epoch(
                CAST(ended_at   AS TIMESTAMP)
              - CAST(started_at AS TIMESTAMP)
            ) / 60.0 BETWEEN 1 AND 1440
        AND start_station_id IS NOT NULL
        AND start_station_id != ''
""")

raw   = con.execute('SELECT COUNT(*) AS n FROM trips').fetchdf()['n'][0]
clean = con.execute('SELECT COUNT(*) AS n FROM trips_clean').fetchdf()['n'][0]
print(f'trips_clean VIEW created')
print(f'  Raw rows    : {raw:>10,}')
print(f'  Clean rows  : {clean:>10,}')
print(f'  Filtered    : {raw-clean:>10,}  ({(raw-clean)/raw*100:.1f}%)')

trips_clean VIEW created
  Raw rows    :  4,009,586
  Clean rows  :  4,005,293
  Filtered    :      4,293  (0.1%)


In [29]:
# ── Create trips_with_station VIEW - new edit
# Root cause: trips.start_station_id uses the short_name format (e.g. "5156.05")
# NOT the station_id format (UUIDs / large integers) from the live GBFS feed.
# This view resolves that once so every downstream query just uses it directly.

con.execute("DROP VIEW IF EXISTS trips_with_station")
con.execute("""
    CREATE VIEW trips_with_station AS
    SELECT
        t.*,
        s.station_id  AS matched_station_id,
        s.name        AS station_name_live,
        s.borough,
        s.lat         AS station_lat,
        s.lon         AS station_lon,
        s.capacity
    FROM trips_clean t
    LEFT JOIN stations s
        ON TRIM(t.start_station_id) = TRIM(s.short_name)
""")

total   = con.execute("SELECT COUNT(*) AS n FROM trips_with_station").fetchdf()['n'][0]
matched = con.execute("""
    SELECT COUNT(*) AS n FROM trips_with_station WHERE borough IS NOT NULL
""").fetchdf()['n'][0]
unmatched = total - matched

print(f"trips_with_station VIEW created")
print(f"  Total clean trips     : {total:>10,}")
print(f"  Matched to a station  : {matched:>10,}  ({matched/total*100:.1f}%)")
print(f"  Unmatched (retired)   : {unmatched:>10,}  ({unmatched/total*100:.1f}%)")
print(f"\n  Borough breakdown of matched trips:")
display(con.execute("""
    SELECT borough, COUNT(*) AS trips
    FROM trips_with_station
    WHERE borough IS NOT NULL
    GROUP BY borough
    ORDER BY trips DESC
""").fetchdf())

trips_with_station VIEW created
  Total clean trips     :  4,005,293
  Matched to a station  :  3,723,258  (93.0%)
  Unmatched (retired)   :    282,035  (7.0%)

  Borough breakdown of matched trips:


,borough,trips
0,Brooklyn,1877815
1,Manhattan,1517573
2,Other,209478
3,Bronx,118392


---
## 3. Station Network Analysis
Before looking for gaps, understand the existing network.
Key question: is Brooklyn well-served, or just geographically spread thin?

In [30]:
# Borough-level summary: count, capacity, size distribution
# avg_docks_per_station reveals commitment level
# Small stations in an area = CitiBike was testing, not fully investing
display(q("""
    SELECT
        borough,
        COUNT(*)                               AS station_count,
        SUM(capacity)                          AS total_docks,
        ROUND(AVG(capacity), 1)                AS avg_docks_per_station,
        MIN(capacity)                          AS min_docks,
        MAX(capacity)                          AS max_docks,
        COUNT(*) FILTER (WHERE capacity < 15)  AS small_stations,
        COUNT(*) FILTER (WHERE capacity >= 25) AS large_stations
    FROM stations
    WHERE capacity > 0 AND is_installed = true AND is_renting = true
    GROUP BY borough
    ORDER BY station_count DESC
""", 'Active station summary by borough'))


-- Active station summary by borough --


,borough,station_count,total_docks,avg_docks_per_station,min_docks,max_docks,small_stations,large_stations
0,Brooklyn,1153,34611.00,30.00,6,120,62,534
1,Manhattan,458,19398.00,42.40,15,123,0,358
2,Bronx,393,8478.00,21.60,12,39,10,72
3,Other,329,7883.00,24.00,14,58,3,103


In [31]:
# Top 20 stations by capacity using RANK() and QUALIFY
# QUALIFY filters on window function results without needing a subquery
# Equivalent to: SELECT * FROM (...subquery...) WHERE capacity_rank <= 20
display(q("""
    SELECT
        RANK() OVER (ORDER BY capacity DESC)  AS capacity_rank,
        name                                  AS station_name,
        borough,
        capacity,
        lat,
        lon
    FROM stations
    WHERE is_installed = true AND is_renting = true AND capacity > 0
    QUALIFY capacity_rank <= 20
    ORDER BY capacity_rank
""", 'Top 20 stations by capacity (RANK + QUALIFY)'))


-- Top 20 stations by capacity (RANK + QUALIFY) --


,capacity_rank,station_name,borough,capacity,lat,lon
0,1,E 40 St & Park Ave,Manhattan,123,40.75,-73.98
1,1,FDR Drive & E 35 St,Manhattan,123,40.74,-73.97
2,3,Gansevoort St & Hudson St,Manhattan,120,40.74,-74.01
3,3,Centre St & Worth St,Brooklyn,120,40.71,-74.00
4,5,W 59 St & 10 Ave,Manhattan,117,40.77,-73.99
5,5,W 43 St & 10 Ave,Manhattan,117,40.76,-73.99
6,5,Allen St & Hester St,Brooklyn,117,40.72,-73.99
7,8,Broadway & E 14 St,Brooklyn,114,40.73,-73.99
8,9,Park Ave & E 41 St,Manhattan,109,40.75,-73.98
9,9,Broadway & W 25 St,Manhattan,109,40.74,-73.99


In [32]:
# Brooklyn stations with capacity percentile and quartile
# PERCENT_RANK: where each station sits in the distribution (0.0 to 1.0)
# NTILE(4): splits into 4 equal buckets, useful as a Tableau color scale
display(q("""
    SELECT
        name                                                   AS station_name,
        capacity,
        ROUND(PERCENT_RANK() OVER (ORDER BY capacity) * 100, 1) AS capacity_pct_rank,
        NTILE(4) OVER (ORDER BY capacity)                       AS capacity_quartile
    FROM stations
    WHERE borough = 'Brooklyn' AND is_installed = true
      AND is_renting = true AND capacity > 0
    ORDER BY capacity DESC
    LIMIT 30
""", 'Brooklyn stations: capacity percentile and quartile (top 30)'))


-- Brooklyn stations: capacity percentile and quartile (top 30) --


,station_name,capacity,capacity_pct_rank,capacity_quartile
0,Centre St & Worth St,120,100.00,4
1,Allen St & Hester St,117,99.90,4
2,Broadway & E 14 St,114,99.80,4
3,E 10 St & Ave A,107,99.70,4
4,Broadway & E 19 St,99,99.50,4
5,E 11 St & Broadway,99,99.50,4
6,W Broadway & Spring St,99,99.50,4
7,E 6 St & Ave B,95,99.40,4
8,1 Ave & E 6 St,94,99.30,4
9,Old Slip & South St,93,99.20,4


---
## 4. Trip Pattern Analysis
When do people ride, how long, who rides, and what bike type?
These patterns feed directly into the Phase 4 financial model.

In [33]:
# Monthly volume: seasonality baseline
display(q("""
    SELECT
        trip_year,
        trip_month,
        COUNT(*)                                              AS total_trips,
        COUNT(*) FILTER (WHERE member_casual = 'member')     AS member_trips,
        COUNT(*) FILTER (WHERE member_casual = 'casual')     AS casual_trips,
        ROUND(
            COUNT(*) FILTER (WHERE member_casual = 'member')
            * 100.0 / COUNT(*)
        , 1)                                                  AS member_pct,
        ROUND(AVG(duration_min), 2)                           AS avg_duration_min
    FROM trips_clean
    GROUP BY trip_year, trip_month
    ORDER BY trip_year, trip_month
""", 'Monthly trip volume and member mix'))


-- Monthly trip volume and member mix --


,trip_year,trip_month,total_trips,member_trips,casual_trips,member_pct,avg_duration_min
0,2023,12,374,177,197,47.30,95.06
1,2024,1,1886146,1678479,207667,89.00,11.01
2,2024,2,2118773,1866386,252387,88.10,11.15


In [34]:
# Hourly distribution with LAG() to show ramp-up into peak hours
# Two peaks at 8am and 5-6pm = strong commuter signal
# LAG(n) looks back n rows in the ordered result -- here, the previous hour
hourly = q("""
    SELECT
        trip_hour,
        COUNT(*)                                               AS trips,
        COUNT(*) FILTER (WHERE member_casual = 'member')      AS member_trips,
        COUNT(*) FILTER (WHERE member_casual = 'casual')      AS casual_trips,
        LAG(COUNT(*)) OVER (ORDER BY trip_hour)                AS prev_hour_trips,
        COUNT(*) - LAG(COUNT(*)) OVER (ORDER BY trip_hour)    AS delta_from_prev
    FROM trips_clean
    GROUP BY trip_hour
    ORDER BY trip_hour
""", 'Hourly distribution with LAG()')

display(hourly)
peak = hourly.loc[hourly['trips'].idxmax()]
print(f"Peak hour: {int(peak['trip_hour']):02d}:00  ({int(peak['trips']):,} trips)")


-- Hourly distribution with LAG() --


,trip_hour,trips,member_trips,casual_trips,prev_hour_trips,delta_from_prev
0,0,45134,36404,8730,<NA>,<NA>
1,1,27211,21424,5787,45134,-17923
2,2,17659,13746,3913,27211,-9552
3,3,11901,9392,2509,17659,-5758
4,4,12322,10097,2225,11901,421
5,5,32667,29711,2956,12322,20345
6,6,89530,82587,6943,32667,56863
7,7,187636,174351,13285,89530,98106
8,8,291946,272624,19322,187636,104310
9,9,229468,209843,19625,291946,-62478


Peak hour: 17:00  (365,580 trips)


In [35]:
# Day-of-week patterns
# Weekday-heavy = commuter use case (strongest expansion argument for deep Brooklyn)
# Weekend-heavy = leisure/tourist (weaker argument)
dow_map = {0:'Sun',1:'Mon',2:'Tue',3:'Wed',4:'Thu',5:'Fri',6:'Sat'}

dow = q("""
    SELECT
        trip_dow,
        COUNT(*)                                              AS total_trips,
        ROUND(AVG(duration_min), 2)                           AS avg_duration_min,
        ROUND(
            COUNT(*) FILTER (WHERE member_casual = 'member')
            * 100.0 / COUNT(*)
        , 1)                                                  AS member_pct
    FROM trips_clean
    GROUP BY trip_dow
    ORDER BY trip_dow
""", 'Day-of-week patterns')

dow['day'] = dow['trip_dow'].map(dow_map)
display(dow[['day','total_trips','avg_duration_min','member_pct']])


-- Day-of-week patterns --


,day,total_trips,avg_duration_min,member_pct
0,Sun,397583,11.40,84.90
1,Mon,605111,10.94,89.00
2,Tue,547971,10.74,90.80
3,Wed,651769,10.80,91.00
4,Thu,709549,10.81,90.10
5,Fri,592003,11.01,88.10
6,Sat,501307,12.28,83.10


In [36]:
# E-bike vs classic split
# SUM(COUNT(*)) OVER () = grand total across all rows -- computes share in one pass
# E-bike trips matter for financial model: per-minute pricing, favored for longer commutes
display(q("""
    SELECT
        rideable_type,
        COUNT(*)                                               AS trips,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)    AS pct_of_total,
        ROUND(AVG(duration_min), 2)                            AS avg_duration_min,
        ROUND(
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY duration_min)
        , 2)                                                   AS median_duration_min
    FROM trips_clean
    GROUP BY rideable_type
    ORDER BY trips DESC
""", 'Trip volume and duration by bike type'))


-- Trip volume and duration by bike type --


,rideable_type,trips,pct_of_total,avg_duration_min,median_duration_min
0,electric_bike,2579306,64.40,10.83,7.92
1,classic_bike,1425987,35.60,11.56,7.71


---
## 5. Demand Profiling by Station and Borough
Join trips back to stations to understand where demand concentrates
and where it is absent. Foundation of the gap analysis.

In [37]:
# new fix
station_demand = q("""
    WITH st AS (
        SELECT
            start_station_id,
            start_station_name,
            borough,
            capacity,
            COUNT(*)                                              AS trip_count,
            ROUND(AVG(duration_min), 2)                           AS avg_duration_min,
            ROUND(
                COUNT(*) FILTER (WHERE member_casual = 'member')
                * 100.0 / COUNT(*)
            , 1)                                                  AS member_pct
        FROM trips_with_station
        GROUP BY start_station_id, start_station_name, borough, capacity
    )
    SELECT
        *,
        RANK() OVER (ORDER BY trip_count DESC)                    AS demand_rank,
        ROUND(
            SUM(trip_count) OVER (ORDER BY trip_count DESC)
            * 100.0 / SUM(trip_count) OVER ()
        , 2)                                                      AS cumulative_pct,
        ROUND(trip_count * 1.0 / NULLIF(capacity, 0), 1)         AS trips_per_dock
    FROM st
    ORDER BY trip_count DESC
""", "All-station demand ranking")

display(station_demand.head(25))
top10_n     = max(1, len(station_demand) // 10)
top10_trips = station_demand.head(top10_n)['trip_count'].sum()
total_trips = station_demand['trip_count'].sum()
print(f"Top 10% of stations ({top10_n}) drive {top10_trips/total_trips*100:.1f}% of all trips")


-- All-station demand ranking --


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_station_id,start_station_name,borough,capacity,trip_count,avg_duration_min,member_pct,demand_rank,cumulative_pct,trips_per_dock
0,6140.05,W 21 St & 6 Ave,Manhattan,74,17896,9.48,93.50,1,0.45,241.80
1,6822.09,1 Ave & E 68 St,Manhattan,63,13935,12.55,94.00,2,0.79,221.20
2,6450.05,8 Ave & W 31 St,Manhattan,97,13773,11.67,89.80,3,1.14,142.00
3,5788.13,Lafayette St & E 8 St,Brooklyn,0,13052,9.72,91.50,4,1.46,NaN
4,6331.01,W 31 St & 7 Ave,Manhattan,84,12647,10.24,89.20,5,1.78,150.60
5,6173.08,Broadway & W 25 St,Manhattan,109,12638,10.76,90.20,6,2.10,115.90
6,5779.11,Ave A & E 14 St,Brooklyn,41,12509,7.21,92.10,7,2.41,305.10
7,6197.08,E 33 St & 1 Ave,Manhattan,83,11926,10.42,91.40,8,2.71,143.70
8,6602.03,W 41 St & 8 Ave,Manhattan,105,11668,10.43,91.60,9,3.00,111.10
9,5905.14,University Pl & E 14 St,Brooklyn,61,11315,9.27,90.30,10,3.28,185.50


Top 10% of stations (225) drive 40.4% of all trips


In [38]:
# Brooklyn station demand ranking - new edit
# Uses trips_with_station -- join already resolved via short_name
bk_demand = q("""
    WITH st AS (
        SELECT
            start_station_id,
            start_station_name,
            station_lat  AS lat,
            station_lon  AS lon,
            capacity,
            COUNT(*)                                              AS trip_count,
            ROUND(AVG(duration_min), 2)                           AS avg_duration_min,
            ROUND(
                COUNT(*) FILTER (WHERE member_casual = 'member')
                * 100.0 / COUNT(*)
            , 1)                                                  AS member_pct
        FROM trips_with_station
        WHERE borough = 'Brooklyn'
        GROUP BY start_station_id, start_station_name,
                 station_lat, station_lon, capacity
    )
    SELECT
        RANK() OVER (ORDER BY trip_count DESC)            AS brooklyn_rank,
        *,
        ROUND(trip_count * 1.0 / NULLIF(capacity, 0), 1) AS trips_per_dock
    FROM st
    ORDER BY trip_count DESC
""", "Brooklyn station demand ranking")

display(bk_demand.head(30))
bk_demand.to_csv(EXPORTS_DIR / 'brooklyn_station_demand.csv', index=False)
print(f"Exported: brooklyn_station_demand.csv  ({len(bk_demand):,} stations)")


-- Brooklyn station demand ranking --


,brooklyn_rank,start_station_id,start_station_name,lat,lon,capacity,trip_count,avg_duration_min,member_pct,trips_per_dock
0,1,5788.13,Lafayette St & E 8 St,40.73,-73.99,0,13052,9.72,91.50,NaN
1,2,5779.11,Ave A & E 14 St,40.73,-73.98,41,12509,7.21,92.10,305.10
2,3,5905.14,University Pl & E 14 St,40.73,-73.99,61,11315,9.27,90.30,185.50
3,4,5905.12,Broadway & E 14 St,40.73,-73.99,114,11260,9.51,89.20,98.80
4,5,5303.06,Clinton St & Grand St,40.72,-73.99,54,10709,9.37,94.60,198.30
5,6,5374.01,Norfolk St & Broome St,40.72,-73.99,59,9350,10.38,89.30,158.50
6,7,5886.13,E 20 St & FDR Dr,40.73,-73.98,75,8878,9.90,91.80,118.40
7,8,5303.08,Canal St & Rutgers St,40.71,-73.99,45,8814,10.30,90.40,195.90
8,9,5382.06,Grand St & Elizabeth St,40.72,-74.00,45,8807,11.53,92.00,195.70
9,10,5382.07,Forsyth St & Grand St,40.72,-73.99,63,8776,9.31,93.40,139.30


Exported: brooklyn_station_demand.csv  (978 stations)


In [39]:
display(q("""
    WITH yearly AS (
        SELECT borough, trip_year, COUNT(*) AS trips
        FROM trips_with_station
        WHERE borough IS NOT NULL
        GROUP BY borough, trip_year
    )
    SELECT
        borough, trip_year, trips,
        LAG(trips) OVER (PARTITION BY borough ORDER BY trip_year) AS prior_year_trips,
        ROUND(
            (trips - LAG(trips) OVER (PARTITION BY borough ORDER BY trip_year))
            * 100.0
            / NULLIF(LAG(trips) OVER (PARTITION BY borough ORDER BY trip_year), 0)
        , 1) AS yoy_growth_pct
    FROM yearly
    ORDER BY borough, trip_year
""", "YoY growth by borough"))


-- YoY growth by borough --


,borough,trip_year,trips,prior_year_trips,yoy_growth_pct
0,Bronx,2023,15,<NA>,NaN
1,Bronx,2024,118377,15,789080.00
2,Brooklyn,2023,164,<NA>,NaN
3,Brooklyn,2024,1877651,164,1144809.10
4,Manhattan,2023,159,<NA>,NaN
5,Manhattan,2024,1517414,159,954248.40
6,Other,2023,20,<NA>,NaN
7,Other,2024,209458,20,1047190.00


---
## 6. Transit Dependency Analysis
Areas with many bus stops but zero CitiBike = strongest expansion candidates.
Bus dependency means people need transit to get anywhere -- natural bike share users.

In [40]:
# Bus stop count by borough
display(q("""
    SELECT borough_feed AS borough, COUNT(*) AS bus_stop_count
    FROM bus_stops
    WHERE lat IS NOT NULL AND lon IS NOT NULL
    GROUP BY borough_feed
    ORDER BY bus_stop_count DESC
""", 'Bus stops by borough (MTA GTFS)'))


-- Bus stops by borough (MTA GTFS) --


,borough,bus_stop_count
0,brooklyn,4558
1,mta_busco,1945
2,bronx,1884
3,staten_island,1862
4,manhattan,1773
5,queens,1348


In [41]:
# Bus stops with vs. without a nearby CitiBike station
# Bounding-box approximation: 0.0045 degrees ~ 500m at NYC latitude
# True spatial join with polygon buffers happens in Phase 3 with GeoPandas
BUFFER = 0.0045

display(q(f"""
    WITH active AS (
        SELECT
            lat - {BUFFER} AS lat_min,  lat + {BUFFER} AS lat_max,
            lon - {BUFFER} AS lon_min,  lon + {BUFFER} AS lon_max
        FROM stations
        WHERE is_installed=true AND is_renting=true AND capacity>0
    ),
    flagged AS (
        SELECT
            b.borough_feed,
            MAX(CASE
                WHEN CAST(b.lat AS DOUBLE) BETWEEN s.lat_min AND s.lat_max
                 AND CAST(b.lon AS DOUBLE) BETWEEN s.lon_min AND s.lon_max
                THEN 1 ELSE 0
            END) AS has_citibike
        FROM bus_stops b
        LEFT JOIN active s ON 1=1
        WHERE b.lat IS NOT NULL AND b.lon IS NOT NULL
        GROUP BY b.stop_id, b.borough_feed
    )
    SELECT
        borough_feed                                         AS borough,
        COUNT(*)                                             AS total_bus_stops,
        SUM(has_citibike)                                    AS stops_with_citibike,
        COUNT(*) - SUM(has_citibike)                        AS stops_without_citibike,
        ROUND(SUM(has_citibike) * 100.0 / COUNT(*), 1)      AS pct_covered
    FROM flagged
    GROUP BY borough_feed
    ORDER BY pct_covered ASC
""", 'Bus stop CitiBike coverage -- lowest coverage first'))
print('Low pct_covered = many bus-dependent stops with no CitiBike nearby = gap zone')


-- Bus stop CitiBike coverage -- lowest coverage first --


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,borough,total_bus_stops,stops_with_citibike,stops_without_citibike,pct_covered
0,staten_island,1862,105.00,1757.00,5.60
1,queens,1348,81.00,1267.00,6.00
2,mta_busco,1945,744.00,1201.00,38.30
3,brooklyn,4558,2542.00,2016.00,55.80
4,bronx,1884,1069.00,815.00,56.70
5,manhattan,1773,1771.00,2.00,99.90


Low pct_covered = many bus-dependent stops with no CitiBike nearby = gap zone


---
## 7. Coverage Gap Identification
Flag every census tract as Covered or Gap Zone, then aggregate to borough level.
This produces your headline dashboard statistic.

We join `census_tracts` (ACS demographics) with `tracts_geo` (centroid coordinates)
on `geoid` because the two came from different sources:
- `census_tracts` from the Census ACS API (demographics only)
- `tracts_geo` from pygris (geometry and centroids)

In [42]:
# Confirm join key and centroid columns exist in both tables before heavy queries
for tbl in ['census_tracts', 'tracts_geo']:
    try:
        cols = con.execute(f'DESCRIBE {tbl}').fetchdf()['column_name'].tolist()
        print(f'{tbl}')
        print(f'  geoid:        {"geoid" in cols}')
        print(f'  centroid_lat: {"centroid_lat" in cols}')
        print(f'  centroid_lon: {"centroid_lon" in cols}')
        print(f'  all columns:  {cols}')
    except Exception as e:
        print(f'{tbl} not loaded: {e}')

census_tracts
  geoid:        True
  centroid_lat: False
  centroid_lon: False
  all columns:  ['NAME', 'total_population', 'median_household_income', 'commute_by_bike', 'commute_by_transit', 'total_commuters', 'hh_no_vehicle', 'total_households', 'total_housing_units', 'state', 'county', 'tract', 'borough', 'county_fips', 'geoid', 'transit_share', 'bike_share', 'no_vehicle_rate']
tracts_geo
  geoid:        True
  centroid_lat: True
  centroid_lon: True
  all columns:  ['geoid', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'NAME', 'borough', 'county_fips', 'ALAND', 'centroid_lat', 'centroid_lon']


In [43]:
# Census tracts flagged as Covered or Gap Zone
# Join census_tracts + tracts_geo on geoid, then check bounding box overlap
BUFFER = 0.0045

gap_tracts = q(f"""
    WITH active AS (
        SELECT
            lat - {BUFFER} AS lat_min,  lat + {BUFFER} AS lat_max,
            lon - {BUFFER} AS lon_min,  lon + {BUFFER} AS lon_max
        FROM stations
        WHERE is_installed=true AND is_renting=true AND capacity>0
    ),
    tract_centroids AS (
        SELECT
            ct.geoid, ct.borough,
            CAST(ct.total_population        AS DOUBLE) AS population,
            CAST(ct.median_household_income AS DOUBLE) AS median_income,
            CAST(ct.no_vehicle_rate         AS DOUBLE) AS no_vehicle_rate,
            CAST(ct.transit_share           AS DOUBLE) AS transit_share,
            CAST(g.centroid_lat             AS DOUBLE) AS clat,
            CAST(g.centroid_lon             AS DOUBLE) AS clon
        FROM census_tracts ct
        INNER JOIN tracts_geo g ON ct.geoid = g.geoid
        WHERE CAST(ct.total_population AS DOUBLE) > 0
    ),
    flagged AS (
        SELECT
            tc.*,
            MAX(CASE
                WHEN tc.clat BETWEEN s.lat_min AND s.lat_max
                 AND tc.clon BETWEEN s.lon_min AND s.lon_max
                THEN 1 ELSE 0
            END) AS has_citibike
        FROM tract_centroids tc
        LEFT JOIN active s ON 1=1
        WHERE tc.clat IS NOT NULL AND tc.clon IS NOT NULL
        GROUP BY tc.geoid, tc.borough, tc.population, tc.median_income,
                 tc.no_vehicle_rate, tc.transit_share, tc.clat, tc.clon
    )
    SELECT
        geoid, borough,
        ROUND(population, 0)                        AS population,
        ROUND(median_income, 0)                     AS median_income,
        ROUND(no_vehicle_rate  * 100, 1)            AS no_vehicle_pct,
        ROUND(transit_share    * 100, 1)            AS transit_commute_pct,
        has_citibike,
        CASE WHEN has_citibike = 0 THEN 'Gap Zone' ELSE 'Covered' END AS coverage_status,
        clat AS centroid_lat,
        clon AS centroid_lon
    FROM flagged
    ORDER BY has_citibike ASC, population DESC
""", 'Census tracts: Covered vs Gap Zone')

summary = (
    gap_tracts
    .groupby('coverage_status')
    .agg(tracts=('geoid','count'), population=('population','sum'),
         avg_no_vehicle=('no_vehicle_pct','mean'), avg_transit=('transit_commute_pct','mean'))
    .round(1).reset_index()
)
display(summary)
print()
display(gap_tracts[gap_tracts['coverage_status']=='Gap Zone'].head(20))


-- Census tracts: Covered vs Gap Zone --


,coverage_status,tracts,population,avg_no_vehicle,avg_transit
0,Covered,1115,4787857.00,64.40,53.20
1,Gap Zone,1127,3834610.00,32.40,39.40


,geoid,borough,population,median_income,no_vehicle_pct,transit_commute_pct,has_citibike,coverage_status,centroid_lat,centroid_lon
0,36081088400,Queens,9257.00,102727.00,11.40,19.50,0,Gap Zone,40.66,-73.83
1,36081099801,Queens,9129.00,69621.00,33.80,38.50,0,Gap Zone,40.60,-73.76
2,36061016200,Manhattan,9015.00,26520.00,86.20,59.30,0,Gap Zone,40.78,-73.94
3,36081089201,Queens,8806.00,157845.00,3.00,19.90,0,Gap Zone,40.66,-73.84
4,36081157101,Queens,8501.00,102465.00,11.70,20.80,0,Gap Zone,40.74,-73.72
5,36005038600,Bronx,8233.00,62340.00,56.40,41.40,0,Gap Zone,40.88,-73.85
6,36081122703,Queens,8202.00,69417.00,12.50,38.90,0,Gap Zone,40.74,-73.81
7,36005004200,Bronx,8179.00,52423.00,37.60,44.60,0,Gap Zone,40.82,-73.86
8,36081103202,Queens,8163.00,85561.00,25.90,23.20,0,Gap Zone,40.61,-73.75
9,36081080900,Queens,8110.00,60014.00,22.80,32.20,0,Gap Zone,40.73,-73.82


In [44]:
# Gap population by borough -- headline dashboard stat
BUFFER = 0.0045

gap_borough = q(f"""
    WITH active AS (
        SELECT
            lat-{BUFFER} AS lat_min, lat+{BUFFER} AS lat_max,
            lon-{BUFFER} AS lon_min, lon+{BUFFER} AS lon_max
        FROM stations
        WHERE is_installed=true AND is_renting=true AND capacity>0
    ),
    flagged AS (
        SELECT
            ct.borough,
            CAST(ct.total_population  AS DOUBLE) AS pop,
            CAST(ct.no_vehicle_rate   AS DOUBLE) AS no_veh,
            CAST(ct.transit_share     AS DOUBLE) AS transit,
            MAX(CASE
                WHEN CAST(g.centroid_lat AS DOUBLE) BETWEEN s.lat_min AND s.lat_max
                 AND CAST(g.centroid_lon AS DOUBLE) BETWEEN s.lon_min AND s.lon_max
                THEN 1 ELSE 0
            END) AS has_citibike
        FROM census_tracts ct
        INNER JOIN tracts_geo g  ON ct.geoid = g.geoid
        LEFT JOIN  active s ON 1=1
        WHERE CAST(ct.total_population AS DOUBLE) > 0
          AND g.centroid_lat IS NOT NULL
        GROUP BY ct.borough, ct.geoid, ct.total_population,
                 ct.no_vehicle_rate, ct.transit_share, g.centroid_lat, g.centroid_lon
    )
    SELECT
        borough,
        COUNT(*)                                                           AS total_tracts,
        COUNT(*) FILTER (WHERE has_citibike = 0)                          AS gap_tracts,
        ROUND(SUM(pop), 0)                                                 AS total_population,
        ROUND(SUM(CASE WHEN has_citibike=0 THEN pop ELSE 0 END), 0)       AS gap_population,
        ROUND(
            SUM(CASE WHEN has_citibike=0 THEN pop ELSE 0 END)
            * 100.0 / NULLIF(SUM(pop),0)
        , 1)                                                               AS pct_pop_in_gap,
        ROUND(AVG(CASE WHEN has_citibike=0 THEN no_veh END)*100, 1)       AS gap_no_vehicle_pct,
        ROUND(AVG(CASE WHEN has_citibike=0 THEN transit END)*100, 1)      AS gap_transit_pct
    FROM flagged
    GROUP BY borough
    ORDER BY pct_pop_in_gap DESC
""", 'Gap population by borough')

display(gap_borough)
gap_borough.to_csv(EXPORTS_DIR / 'gap_zone_borough_summary.csv', index=False)
print('Exported: gap_zone_borough_summary.csv')


-- Gap population by borough --


,borough,total_tracts,gap_tracts,total_population,gap_population,pct_pop_in_gap,gap_no_vehicle_pct,gap_transit_pct
0,Staten Island,122,122,492925.00,492925.00,100.00,16.70,25.10
1,Queens,689,455,2360826.00,1526931.00,64.70,24.70,36.50
2,Brooklyn,779,380,2679620.00,1195051.00,44.60,41.60,44.80
3,Bronx,348,163,1443229.00,591598.00,41.00,42.80,45.40
4,Manhattan,304,7,1645867.00,28105.00,1.70,76.70,42.40


Exported: gap_zone_borough_summary.csv


---
## 8. Census Demographic Overlay
The equity argument: gap zones are not just geographically distant --
they are disproportionately lower-income and transit-dependent.
This turns a mapping exercise into urban policy analysis.

In [45]:
# Income bracket distribution: covered vs gap tracts
# CASE bucketing + aggregation in one query
# The '1.' prefix on bracket labels preserves sort order in Tableau
BUFFER = 0.0045

income_dist = q(f"""
    WITH active AS (
        SELECT lat-{BUFFER} AS lat_min, lat+{BUFFER} AS lat_max,
               lon-{BUFFER} AS lon_min, lon+{BUFFER} AS lon_max
        FROM stations
        WHERE is_installed=true AND is_renting=true AND capacity>0
    ),
    flagged AS (
        SELECT
            CAST(ct.median_household_income AS DOUBLE) AS income,
            CAST(ct.total_population        AS DOUBLE) AS pop,
            CAST(ct.no_vehicle_rate         AS DOUBLE) AS no_veh,
            CAST(ct.transit_share           AS DOUBLE) AS transit,
            MAX(CASE
                WHEN CAST(g.centroid_lat AS DOUBLE) BETWEEN s.lat_min AND s.lat_max
                 AND CAST(g.centroid_lon AS DOUBLE) BETWEEN s.lon_min AND s.lon_max
                THEN 1 ELSE 0
            END) AS has_citibike
        FROM census_tracts ct
        INNER JOIN tracts_geo g ON ct.geoid = g.geoid
        LEFT JOIN  active s ON 1=1
        WHERE CAST(ct.total_population AS DOUBLE) > 0
          AND CAST(ct.median_household_income AS DOUBLE) > 0
        GROUP BY ct.geoid, ct.median_household_income, ct.total_population,
                 ct.no_vehicle_rate, ct.transit_share, g.centroid_lat, g.centroid_lon
    )
    SELECT
        CASE
            WHEN income <  30000 THEN '1. Under $30k'
            WHEN income <  50000 THEN '2. $30k to $50k'
            WHEN income <  75000 THEN '3. $50k to $75k'
            WHEN income < 100000 THEN '4. $75k to $100k'
            ELSE                      '5. Over $100k'
        END                                                     AS income_bracket,
        COUNT(*) FILTER (WHERE has_citibike=1)                  AS covered_tracts,
        COUNT(*) FILTER (WHERE has_citibike=0)                  AS gap_tracts,
        ROUND(SUM(CASE WHEN has_citibike=1 THEN pop ELSE 0 END),0) AS covered_pop,
        ROUND(SUM(CASE WHEN has_citibike=0 THEN pop ELSE 0 END),0) AS gap_pop,
        ROUND(AVG(no_veh)  * 100, 1)                            AS avg_no_vehicle_pct,
        ROUND(AVG(transit) * 100, 1)                            AS avg_transit_pct
    FROM flagged
    GROUP BY income_bracket
    ORDER BY income_bracket
""", 'Income distribution: Covered vs Gap tracts')

display(income_dist)
income_dist.to_csv(EXPORTS_DIR / 'income_coverage_distribution.csv', index=False)
print('Exported: income_coverage_distribution.csv')


-- Income distribution: Covered vs Gap tracts --


,income_bracket,covered_tracts,gap_tracts,covered_pop,gap_pop,avg_no_vehicle_pct,avg_transit_pct
0,1. Under $30k,86,36,395810.00,149379.00,77.70,60.20
1,2. $30k to $50k,166,104,818203.00,422966.00,66.80,55.40
2,3. $50k to $75k,269,323,1144036.00,1145395.00,50.40,50.00
3,4. $75k to $100k,230,359,904732.00,1131994.00,38.40,44.60
4,5. Over $100k,348,280,1505987.00,957046.00,42.50,37.70


Exported: income_coverage_distribution.csv


In [46]:
# Full tract-level export for the Tableau map layer
# One row per census tract with everything Tableau needs to color and filter
BUFFER = 0.0045

tract_export = q(f"""
    WITH active AS (
        SELECT lat-{BUFFER} AS lat_min, lat+{BUFFER} AS lat_max,
               lon-{BUFFER} AS lon_min, lon+{BUFFER} AS lon_max
        FROM stations
        WHERE is_installed=true AND is_renting=true AND capacity>0
    )
    SELECT
        ct.geoid, ct.borough,
        CAST(g.centroid_lat              AS DOUBLE)             AS centroid_lat,
        CAST(g.centroid_lon              AS DOUBLE)             AS centroid_lon,
        CAST(ct.total_population         AS DOUBLE)             AS population,
        CAST(ct.median_household_income  AS DOUBLE)             AS median_income,
        ROUND(CAST(ct.no_vehicle_rate    AS DOUBLE)*100, 1)    AS no_vehicle_pct,
        ROUND(CAST(ct.transit_share      AS DOUBLE)*100, 1)    AS transit_commute_pct,
        ROUND(CAST(ct.bike_share         AS DOUBLE)*100, 2)    AS bike_commute_pct,
        MAX(CASE
            WHEN CAST(g.centroid_lat AS DOUBLE) BETWEEN s.lat_min AND s.lat_max
             AND CAST(g.centroid_lon AS DOUBLE) BETWEEN s.lon_min AND s.lon_max
            THEN 1 ELSE 0
        END)                                                    AS has_citibike,
        CASE WHEN MAX(CASE
            WHEN CAST(g.centroid_lat AS DOUBLE) BETWEEN s.lat_min AND s.lat_max
             AND CAST(g.centroid_lon AS DOUBLE) BETWEEN s.lon_min AND s.lon_max
            THEN 1 ELSE 0 END) = 0
        THEN 'Gap Zone' ELSE 'Covered' END                      AS coverage_status
    FROM census_tracts ct
    INNER JOIN tracts_geo g ON ct.geoid = g.geoid
    LEFT JOIN  active s ON 1=1
    WHERE CAST(ct.total_population AS DOUBLE) > 0
      AND g.centroid_lat IS NOT NULL
    GROUP BY ct.geoid, ct.borough, g.centroid_lat, g.centroid_lon,
             ct.total_population, ct.median_household_income,
             ct.no_vehicle_rate, ct.transit_share, ct.bike_share
    ORDER BY ct.borough, has_citibike, population DESC
""", 'Full tract demographic export')

tract_export.to_csv(EXPORTS_DIR / 'tract_demographics_coverage.csv', index=False)
print(f'Exported: tract_demographics_coverage.csv  ({len(tract_export):,} rows)')
display(tract_export.head(10))


-- Full tract demographic export --
Exported: tract_demographics_coverage.csv  (2,242 rows)


,geoid,borough,centroid_lat,centroid_lon,population,median_income,no_vehicle_pct,transit_commute_pct,bike_commute_pct,has_citibike,coverage_status
0,36005038600,Bronx,40.88,-73.85,8233.00,62340.00,56.40,41.40,0.00,0,Gap Zone
1,36005004200,Bronx,40.82,-73.86,8179.00,52423.00,37.60,44.60,0.00,0,Gap Zone
2,36005021002,Bronx,40.84,-73.86,7925.00,71397.00,69.30,55.80,0.00,0,Gap Zone
3,36005021001,Bronx,40.84,-73.86,7895.00,46269.00,65.40,60.30,0.00,0,Gap Zone
4,36005035800,Bronx,40.87,-73.84,7781.00,85804.00,33.40,42.80,0.00,0,Gap Zone
5,36005046203,Bronx,40.88,-73.82,7762.00,41371.00,40.00,56.60,0.00,0,Gap Zone
6,36005013200,Bronx,40.81,-73.82,7249.00,104651.00,18.20,27.50,0.00,0,Gap Zone
7,36005042600,Bronx,40.89,-73.84,6844.00,87763.00,23.50,40.90,0.00,0,Gap Zone
8,36005030000,Bronx,40.85,-73.83,6702.00,90307.00,30.10,33.00,0.00,0,Gap Zone
9,36005046207,Bronx,40.88,-73.83,6671.00,69637.00,45.50,41.20,0.00,0,Gap Zone


---
## 9. Sheepshead Bay Deep Dive
Your anchor neighborhood. Quantify exactly how far the nearest station is
and what the community looks like demographically.

In [47]:
# Nearest CitiBike stations using flat-earth Haversine approximation
# Accurate within 1% for distances under 50km -- sufficient for NYC
# Formula: dist_km = sqrt((dlat)^2 + (dlon * cos(lat))^2) * 111.32
SHB_LAT = 40.585
SHB_LON = -73.944

nearest = q(f"""
    SELECT
        name AS station_name, borough, capacity, lat, lon,
        ROUND(
            SQRT(
                POWER(lat - {SHB_LAT}, 2)
              + POWER((lon - {SHB_LON}) * COS(RADIANS({SHB_LAT})), 2)
            ) * 111.32
        , 2) AS distance_km
    FROM stations
    WHERE is_installed=true AND is_renting=true AND capacity>0
    ORDER BY distance_km ASC
    LIMIT 15
""", 'Nearest stations to Sheepshead Bay')

display(nearest)
c = nearest.iloc[0]
print(f"Closest: {c['station_name']}")
print(f"Distance: {c['distance_km']} km ({c['distance_km']*0.621:.1f} miles)")
print(f"Cycling at 15km/h: {c['distance_km']/15*60:.0f} min")
print(f"Walking  at  5km/h: {c['distance_km']/5*60:.0f} min")


-- Nearest stations to Sheepshead Bay --


,station_name,borough,capacity,lat,lon,distance_km
0,Ditmas Ave & McDonald Ave,Brooklyn,23,40.64,-73.98,6.32
1,Cortelyou Rd & E 8 St,Brooklyn,16,40.64,-73.97,6.39
2,Cortelyou Rd & Stratford Rd,Brooklyn,21,40.64,-73.97,6.42
3,Cortelyou Rd & Argyle Rd,Brooklyn,25,40.64,-73.97,6.44
4,E 16 St & Cortelyou Rd,Brooklyn,26,40.64,-73.96,6.53
5,E 3 St & Cortelyou Rd,Brooklyn,14,40.64,-73.98,6.54
6,Cortelyou Rd & E 19 St,Brooklyn,23,40.64,-73.96,6.58
7,Cortelyou Rd & E 34 St,Brooklyn,19,40.64,-73.94,6.63
8,Cortelyou Rd & Albany Ave,Brooklyn,21,40.64,-73.94,6.68
9,Beverley Rd & Nostrand Ave,Brooklyn,25,40.65,-73.95,6.70


Closest: Ditmas Ave & McDonald Ave
Distance: 6.32 km (3.9 miles)
Cycling at 15km/h: 25 min
Walking  at  5km/h: 76 min


In [48]:
# Bus stops in Sheepshead Bay bounding box
SHB_LAT_MIN, SHB_LAT_MAX = 40.570, 40.600
SHB_LON_MIN, SHB_LON_MAX = -73.970, -73.920

shbay_buses = q(f"""
    SELECT stop_id, stop_name,
           CAST(lat AS DOUBLE) AS lat,
           CAST(lon AS DOUBLE) AS lon
    FROM bus_stops
    WHERE CAST(lat AS DOUBLE) BETWEEN {SHB_LAT_MIN} AND {SHB_LAT_MAX}
      AND CAST(lon AS DOUBLE) BETWEEN {SHB_LON_MIN} AND {SHB_LON_MAX}
    ORDER BY stop_name
""", 'Bus stops in Sheepshead Bay')

print(f'Bus stops in area : {len(shbay_buses):,}')
print(f'CitiBike stations : 0  <-- the gap')
display(shbay_buses.head(20))


-- Bus stops in Sheepshead Bay --
Bus stops in area : 225
CitiBike stations : 0  <-- the gap


,stop_id,stop_name,lat,lon
0,300206,AVENUE U/BEDFORD AV,40.60,-73.95
1,300261,AVENUE U/BEDFORD AV,40.60,-73.95
2,300200,AVENUE U/CONEY ISLAND AV,40.60,-73.96
3,300267,AVENUE U/CONEY ISLAND AV,40.60,-73.96
4,300201,AVENUE U/E 13 ST,40.60,-73.96
5,300266,AVENUE U/E 13 ST,40.60,-73.96
6,308545,AVENUE U/E 16 ST,40.60,-73.96
7,300265,AVENUE U/E 16 ST,40.60,-73.96
8,300203,AVENUE U/E 18 ST,40.60,-73.95
9,300264,AVENUE U/E 18 ST,40.60,-73.95


In [49]:
# Census profile for Sheepshead Bay tracts
shbay_census = q(f"""
    SELECT
        ct.geoid,
        CAST(ct.total_population        AS DOUBLE)              AS population,
        CAST(ct.median_household_income AS DOUBLE)              AS median_income,
        ROUND(CAST(ct.no_vehicle_rate   AS DOUBLE)*100, 1)     AS no_vehicle_pct,
        ROUND(CAST(ct.transit_share     AS DOUBLE)*100, 1)     AS transit_commute_pct,
        ROUND(CAST(ct.bike_share        AS DOUBLE)*100, 2)     AS bike_commute_pct,
        CAST(g.centroid_lat             AS DOUBLE)              AS lat,
        CAST(g.centroid_lon             AS DOUBLE)              AS lon
    FROM census_tracts ct
    INNER JOIN tracts_geo g ON ct.geoid = g.geoid
    WHERE CAST(g.centroid_lat AS DOUBLE) BETWEEN {SHB_LAT_MIN} AND {SHB_LAT_MAX}
      AND CAST(g.centroid_lon AS DOUBLE) BETWEEN {SHB_LON_MIN} AND {SHB_LON_MAX}
      AND CAST(ct.total_population AS DOUBLE) > 0
    ORDER BY population DESC
""", 'Census tracts in Sheepshead Bay')

display(shbay_census)

if len(shbay_census) > 0:
    print(f"\nSheepshead Bay aggregate profile")
    print(f"  Total population           : {shbay_census['population'].sum():,.0f}")
    print(f"  Avg median income          : ${shbay_census['median_income'].mean():,.0f}")
    print(f"  Avg no-vehicle rate        : {shbay_census['no_vehicle_pct'].mean():.1f}%")
    print(f"  Avg transit commute share  : {shbay_census['transit_commute_pct'].mean():.1f}%")
    print(f"  Current bike commute share : {shbay_census['bike_commute_pct'].mean():.2f}%")
    print(f"")
    print(f"  Near-zero bike commute = suppressed demand from lack of infrastructure,")
    print(f"  not lack of interest. Key point for your dashboard narrative.")


-- Census tracts in Sheepshead Bay --


,geoid,population,median_income,no_vehicle_pct,transit_commute_pct,bike_commute_pct,lat,lon
0,36047060000,6509.00,94482.00,29.60,47.60,0.00,40.59,-73.95
1,36047061004,5636.00,38137.00,71.40,40.00,0.00,40.58,-73.96
2,36047059402,5617.00,77500.00,24.00,33.10,1.56,40.59,-73.94
3,36047057200,5520.00,40330.00,67.40,45.20,1.99,40.60,-73.94
4,36047061002,5248.00,78468.00,46.50,57.40,0.00,40.57,-73.96
5,36047062800,4808.00,108227.00,11.80,23.90,0.00,40.59,-73.93
6,36047037401,4602.00,56667.00,39.80,44.70,0.00,40.59,-73.97
7,36047059404,4487.00,76179.00,45.50,43.90,0.00,40.59,-73.95
8,36047036002,4321.00,38047.00,78.90,37.10,0.00,40.58,-73.96
9,36047037000,4184.00,79688.00,22.20,49.00,0.00,40.59,-73.96



Sheepshead Bay aggregate profile
  Total population           : 140,930
  Avg median income          : $70,865
  Avg no-vehicle rate        : 41.9%
  Avg transit commute share  : 41.1%
  Current bike commute share : 0.76%

  Near-zero bike commute = suppressed demand from lack of infrastructure,
  not lack of interest. Key point for your dashboard narrative.


---
## 10. Export Clean Tables for Phase 3
Produce denormalized, Tableau-ready CSVs. No joins needed in Tableau downstream.

In [50]:
stations_export = q("""
    WITH st AS (
        SELECT
            matched_station_id,
            COUNT(*)                                              AS total_trips,
            ROUND(AVG(duration_min), 2)                           AS avg_duration_min,
            COUNT(*) FILTER (WHERE member_casual = 'member')      AS member_trips,
            ROUND(
                COUNT(*) FILTER (WHERE member_casual = 'member')
                * 100.0 / COUNT(*)
            , 1)                                                  AS member_pct
        FROM trips_with_station
        WHERE matched_station_id IS NOT NULL
        GROUP BY matched_station_id
    )
    SELECT
        s.station_id,
        s.name        AS station_name,
        s.borough,
        s.lat,
        s.lon,
        s.capacity,
        COALESCE(t.total_trips,      0) AS total_trips,
        COALESCE(t.avg_duration_min, 0) AS avg_duration_min,
        COALESCE(t.member_trips,     0) AS member_trips,
        COALESCE(t.member_pct,       0) AS member_pct,
        ROUND(
            COALESCE(t.total_trips, 0) * 1.0 / NULLIF(s.capacity, 0)
        , 2)                            AS trips_per_dock
    FROM stations s
    LEFT JOIN st t ON s.station_id = t.matched_station_id
    WHERE s.is_installed = true
      AND s.is_renting   = true
      AND s.capacity     > 0
    ORDER BY s.borough, total_trips DESC
""", "Final stations export (fixed join)")

stations_export.to_csv(EXPORTS_DIR / 'stations_with_demand.csv', index=False)
print(f"Exported: stations_with_demand.csv  ({len(stations_export):,} stations)")
print(f"Stations with trips > 0: {(stations_export['total_trips'] > 0).sum():,}")


-- Final stations export (fixed join) --
Exported: stations_with_demand.csv  (2,333 stations)
Stations with trips > 0: 2,006


In [51]:
# Sheepshead Bay detail exports
shbay_census.to_csv(EXPORTS_DIR / 'sheepshead_bay_profile.csv',          index=False)
nearest.to_csv(EXPORTS_DIR       / 'sheepshead_bay_nearest_stations.csv', index=False)
shbay_buses.to_csv(EXPORTS_DIR   / 'sheepshead_bay_bus_stops.csv',        index=False)
print('Exported: sheepshead_bay_profile.csv')
print('Exported: sheepshead_bay_nearest_stations.csv')
print('Exported: sheepshead_bay_bus_stops.csv')

Exported: sheepshead_bay_profile.csv
Exported: sheepshead_bay_nearest_stations.csv
Exported: sheepshead_bay_bus_stops.csv


In [53]:
# Final export summary and close connection
print('=' * 60)
print('  PHASE 2 EXPORT SUMMARY')
print('=' * 60)
for f in sorted(EXPORTS_DIR.glob('*.csv')):
    rows = sum(1 for _ in open(f)) - 1
    kb   = f.stat().st_size / 1024
    print(f'  {f.name:<45}  {rows:>6,} rows  {kb:>7.1f} KB')
print(f'\nLocation: {EXPORTS_DIR}')
print('Download from Drive or connect Tableau directly to your Drive folder.')

# Close connection -- ONLY here, at the very end of the notebook
con.close()
print('\nDuckDB connection closed.')
print('Phase 2 complete. Ready for Phase 3: Gap Scoring and Demand Model.')

  PHASE 2 EXPORT SUMMARY
  brooklyn_station_demand.csv                       978 rows     77.6 KB
  gap_zone_borough_summary.csv                        5 rows      0.4 KB
  income_coverage_distribution.csv                    5 rows      0.3 KB
  nta_demand_scores.csv                             197 rows     62.7 KB
  sheepshead_bay_bus_stops.csv                      225 rows     11.1 KB
  sheepshead_bay_nearest_stations.csv                15 rows      0.9 KB
  sheepshead_bay_profile.csv                         42 rows      3.3 KB
  stations_with_demand.csv                        2,333 rows    257.7 KB
  top15_expansion_candidates.csv                     15 rows      2.4 KB
  tract_demographics_coverage.csv                 2,242 rows    213.8 KB

Location: /content/drive/MyDrive/citibike_project/data/exports
Download from Drive or connect Tableau directly to your Drive folder.

DuckDB connection closed.
Phase 2 complete. Ready for Phase 3: Gap Scoring and Demand Model.


# Citibike Borough Expansion Analysis
## Phase 3: Gap Scoring and Demand Model

This phase produces the ranked neighborhood demand scores that justify the expansion recommendation.

**What is built here:**
- True spatial coverage analysis using GeoPandas 0.5-mile buffers (replaces the bounding-box approximation from Phase 2)
- NTA (neighborhood) level aggregation of census demographics
- Bus stop density per neighborhood as a transit dependency signal
- A composite Demand Proxy Score for every gap-zone neighborhood
- Ranked expansion candidate list with proposed station counts
- Exports for Phase 4 (financial model) and Tableau

**Sections:**
- **0** Session setup
- **1** Load spatial layers
- **2** True coverage analysis with GeoPandas buffers
- **3** NTA-level aggregation
- **4** Bus stop density per NTA
- **5** Build the Demand Proxy Score
- **6** Rank gap-zone neighborhoods
- **7** Proposed expansion locations
- **8** Export for Phase 4 and Tableau

---
## 0. Session Setup
Run this cell first every session. Mounts Drive, loads libraries, connects DuckDB.

In [54]:
!pip install duckdb geopandas pygris --quiet

import warnings, json
import numpy as np
import pandas as pd
import geopandas as gpd
import duckdb
from pathlib import Path
from shapely.geometry import Point
from shapely.ops import unary_union

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = Path('/content/drive/MyDrive/citibike_project')
DIRS = {
    'raw_stations': PROJECT_ROOT / 'data' / 'raw' / 'stations',
    'raw_mta'     : PROJECT_ROOT / 'data' / 'raw' / 'mta',
    'raw_census'  : PROJECT_ROOT / 'data' / 'raw' / 'census',
    'raw_geo'     : PROJECT_ROOT / 'data' / 'raw' / 'geo',
    'exports'     : PROJECT_ROOT / 'data' / 'exports',
    'db'          : PROJECT_ROOT / 'db',
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

EXPORTS_DIR = DIRS['exports']
DB_PATH     = str(DIRS['db'] / 'citibike.duckdb')

con = duckdb.connect(DB_PATH)

def q(sql, label=''):
    if label: print(f'\n-- {label} --')
    return con.execute(sql).fetchdf()

# Confirm tables are available
tables = con.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'main'
""").fetchdf()['table_name'].tolist()
print('Tables in DuckDB:', tables)
print('\nSession ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tables in DuckDB: ['bus_stops', 'census_tracts', 'nta_lookup', 'stations', 'tracts_geo', 'trips', 'trips_clean', 'trips_with_station']

Session ready.


---
## 1. Load Spatial Layers

We need three GeoDataFrames for the spatial analysis:
- **stations_gdf** -- existing CitiBike station points
- **nta_gdf** -- NTA neighborhood polygons (our unit of analysis)
- **tracts_gdf** -- census tract polygons (for demographic joins)

All layers are reprojected to **EPSG:2263** (NY State Plane, feet) for accurate
distance calculations. WGS84 (lat/lon degrees) distorts distances at NYC latitude.
We reproject back to WGS84 for export to Tableau.

In [55]:
# Load station points from the stations table in DuckDB
df_stations = q("""
    SELECT station_id, name, borough, lat, lon, capacity,
           is_installed, is_renting
    FROM stations
    WHERE is_installed = true
      AND is_renting   = true
      AND capacity     > 0
      AND lat IS NOT NULL
""")

stations_gdf = gpd.GeoDataFrame(
    df_stations,
    geometry=gpd.points_from_xy(df_stations['lon'], df_stations['lat']),
    crs='EPSG:4326'
)

# Reproject to NY State Plane (feet) for accurate distance buffers
stations_proj = stations_gdf.to_crs('EPSG:2263')

print(f'Stations loaded : {len(stations_proj):,}')
print(f'CRS             : {stations_proj.crs}')
display(stations_proj[['name','borough','capacity','lat','lon']].head(5))

Stations loaded : 2,333
CRS             : EPSG:2263


,name,borough,capacity,lat,lon
0,68 Dr & 112 St,Brooklyn,12,40.73,-73.84
1,83 St & Narrows Ave,Brooklyn,12,40.63,-74.04
2,36 St & 3 Ave,Brooklyn,30,40.66,-74.01
3,Grand Concourse & E 164 St,Bronx,22,40.83,-73.92
4,Cooper Ave & 65 Pl,Brooklyn,23,40.70,-73.89


In [56]:
# Load NTA boundaries
# Primary: GeoJSON saved from Phase 1 Section 6
# Fallback: rebuild from the nta_lookup CSV which has geometry in the_geom column

nta_geojson_path = DIRS['raw_geo'] / 'nta_boundaries.geojson'
nta_csv_path     = DIRS['raw_geo'] / 'nta_boundaries.csv'

if nta_geojson_path.exists():
    nta_gdf = gpd.read_file(nta_geojson_path)
    print(f'NTA GeoJSON loaded from Drive: {len(nta_gdf):,} NTAs')
elif nta_csv_path.exists():
    # Rebuild from centroid CSV -- less precise but workable for scoring
    df_nta = pd.read_csv(nta_csv_path)
    nta_gdf = gpd.GeoDataFrame(
        df_nta,
        geometry=gpd.points_from_xy(df_nta['centroid_lon'], df_nta['centroid_lat']),
        crs='EPSG:4326'
    )
    print(f'NTA centroids loaded from CSV: {len(nta_gdf):,} NTAs (point fallback)')
else:
    print('NTA file not found -- re-run Phase 1 Section 6')
    nta_gdf = None

if nta_gdf is not None:
    nta_proj = nta_gdf.to_crs('EPSG:2263')
    print(f'Columns: {nta_gdf.columns.tolist()}')
    display(nta_gdf.head(3))

NTA GeoJSON loaded from Drive: 197 NTAs
Columns: ['BoroCode', 'BoroName', 'CountyFIPS', 'NTA2020', 'NTAName', 'NTAAbbrev', 'NTAType', 'CDTA2020', 'CDTAName', 'Shape_Length', 'Shape_Area', 'geometry']


,BoroCode,BoroName,CountyFIPS,NTA2020,NTAName,NTAAbbrev,NTAType,CDTA2020,CDTAName,Shape_Length,Shape_Area,geometry
0,3,Brooklyn,47,BK0101,Greenpoint,Grnpt,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),28919.56,35321808.44,"MULTIPOLYGON (((-73.93213 40.72816, -73.93238 ..."
1,3,Brooklyn,47,BK0102,Williamsburg,Wllmsbrg,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),28134.08,28852852.89,"MULTIPOLYGON (((-73.95814 40.7244, -73.95772 4..."
2,3,Brooklyn,47,BK0103,South Williamsburg,SWllmsbrg,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),18250.28,15208960.64,"MULTIPOLYGON (((-73.95024 40.70547, -73.94984 ..."


In [57]:
# Load census tract boundaries (geometry from pygris GeoJSON)
tracts_geojson = DIRS['raw_geo'] / 'census_tracts_geo.geojson'

if tracts_geojson.exists():
    tracts_gdf = gpd.read_file(tracts_geojson)
    tracts_proj = tracts_gdf.to_crs('EPSG:2263')
    print(f'Census tracts loaded: {len(tracts_gdf):,}')
    print(f'Columns: {tracts_gdf.columns.tolist()}')
else:
    print('census_tracts_geo.geojson not found -- re-run Phase 1 Section 6')
    tracts_gdf = None

Census tracts loaded: 2,327
Columns: ['geoid', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'NAME', 'borough', 'county_fips', 'ALAND', 'geometry']


---
## 2. True Coverage Analysis with GeoPandas Buffers

Phase 2 used a bounding-box approximation (~0.5km squares). Here we use proper
circular 0.5-mile (2,640 foot) buffers in projected coordinates, unioned into a
single coverage polygon, then spatially joined to NTAs.

**0.5 miles** is the standard last-mile walk tolerance used in transit planning
and in CitiBike's own service area documentation.

In [58]:
# Build 0.5-mile coverage polygon
# 1 mile = 5,280 feet; 0.5 mile = 2,640 feet (EPSG:2263 unit is feet)
BUFFER_FEET = 2640

print(f'Building {BUFFER_FEET:,}-foot ({BUFFER_FEET/5280:.1f} mile) buffers around {len(stations_proj):,} stations...')

# Buffer each station point
station_buffers = stations_proj.copy()
station_buffers['geometry'] = stations_proj.geometry.buffer(BUFFER_FEET)

# Union all buffers into a single coverage polygon
# unary_union merges overlapping circles into one continuous shape
coverage_union = unary_union(station_buffers['geometry'])

coverage_gdf = gpd.GeoDataFrame(
    {'label': ['CitiBike Coverage Zone']},
    geometry=[coverage_union],
    crs='EPSG:2263'
)

area_sqmi = coverage_union.area / (5280 ** 2)
print(f'Coverage zone area : {area_sqmi:.1f} square miles')
print(f'NYC total area     : ~302 square miles')
print(f'Coverage pct       : {area_sqmi/302*100:.1f}% of NYC by area')

Building 2,640-foot (0.5 mile) buffers around 2,333 stations...
Coverage zone area : 128.5 square miles
NYC total area     : ~302 square miles
Coverage pct       : 42.6% of NYC by area


In [59]:
# Spatial join: which NTAs intersect the coverage zone?
# predicate='intersects' -- NTA is 'covered' if ANY part of it overlaps coverage
# For a stricter definition, use predicate='within' (NTA centroid inside coverage)

if nta_gdf is not None and nta_gdf.geometry.geom_type.unique().tolist() != ['Point']:
    # Polygon NTAs -- use centroid for coverage check (more meaningful than any intersection)
    nta_proj_centroids = nta_proj.copy()
    nta_proj_centroids['geometry'] = nta_proj.geometry.centroid

    nta_coverage = gpd.sjoin(
        nta_proj_centroids,
        coverage_gdf,
        how='left',
        predicate='within'
    )
    nta_coverage['covered'] = nta_coverage['index_right'].notna()

else:
    # Point fallback -- check if centroid point is within coverage
    nta_coverage = nta_proj.copy()
    nta_coverage['covered'] = nta_coverage.geometry.within(coverage_union)

covered_count = nta_coverage['covered'].sum()
total_count   = len(nta_coverage)
print(f'NTA coverage summary:')
print(f'  Total NTAs  : {total_count}')
print(f'  Covered     : {covered_count}  ({covered_count/total_count*100:.1f}%)')
print(f'  Gap zones   : {total_count - covered_count}  ({(total_count-covered_count)/total_count*100:.1f}%)')

# Tag NTA dataframe with coverage status
nta_coverage['coverage_status'] = nta_coverage['covered'].map(
    {True: 'Covered', False: 'Gap Zone'}
)
display(nta_coverage.groupby('coverage_status').size().reset_index(name='count'))

NTA coverage summary:
  Total NTAs  : 197
  Covered     : 101  (51.3%)
  Gap zones   : 96  (48.7%)


,coverage_status,count
0,Covered,101
1,Gap Zone,96


In [60]:
# Spatial join: which census tracts are in gap zones?
# This is the refined version of the Phase 2 bounding-box gap detection

if tracts_gdf is not None:
    tracts_centroids = tracts_proj.copy()
    tracts_centroids['geometry'] = tracts_proj.geometry.centroid

    tracts_coverage = gpd.sjoin(
        tracts_centroids,
        coverage_gdf,
        how='left',
        predicate='within'
    )
    tracts_coverage['covered']         = tracts_coverage['index_right'].notna()
    tracts_coverage['coverage_status'] = tracts_coverage['covered'].map(
        {True: 'Covered', False: 'Gap Zone'}
    )

    print('Census tract coverage (refined spatial join):')
    display(tracts_coverage.groupby(['borough','coverage_status']).size()
            .reset_index(name='tract_count')
            .sort_values(['borough','coverage_status']))

Census tract coverage (refined spatial join):


,borough,coverage_status,tract_count
0,Bronx,Covered,205
1,Bronx,Gap Zone,156
2,Brooklyn,Covered,437
3,Brooklyn,Gap Zone,368
4,Manhattan,Covered,309
5,Manhattan,Gap Zone,1
6,Queens,Covered,261
7,Queens,Gap Zone,464
8,Staten Island,Gap Zone,126


---
## 3. NTA-Level Aggregation

Census data is at the tract level. We join it up to NTAs (neighborhoods) for
the demand score -- neighborhood is the right unit for a CitiBike expansion
recommendation, matching how CitiBike itself plans and announces expansions.

Population-weighted averaging ensures large tracts don't skew neighborhood metrics.

In [61]:
# Load ACS census data from DuckDB
df_census = q("""
    SELECT
        ct.geoid,
        ct.borough,
        CAST(ct.total_population        AS DOUBLE) AS population,
        CAST(ct.median_household_income AS DOUBLE) AS median_income,
        CAST(ct.no_vehicle_rate         AS DOUBLE) AS no_vehicle_rate,
        CAST(ct.transit_share           AS DOUBLE) AS transit_share,
        CAST(ct.bike_share              AS DOUBLE) AS bike_share,
        CAST(g.centroid_lat             AS DOUBLE) AS centroid_lat,
        CAST(g.centroid_lon             AS DOUBLE) AS centroid_lon
    FROM census_tracts ct
    INNER JOIN tracts_geo g ON ct.geoid = g.geoid
    WHERE CAST(ct.total_population AS DOUBLE) > 0
      AND g.centroid_lat IS NOT NULL
""")

print(f'Census records loaded: {len(df_census):,} tracts')
print(f'Columns: {df_census.columns.tolist()}')
display(df_census.head(3))

Census records loaded: 2,242 tracts
Columns: ['geoid', 'borough', 'population', 'median_income', 'no_vehicle_rate', 'transit_share', 'bike_share', 'centroid_lat', 'centroid_lon']


,geoid,borough,population,median_income,no_vehicle_rate,transit_share,bike_share,centroid_lat,centroid_lon
0,36047000100,Brooklyn,4974.00,165188.00,0.50,0.44,0.02,40.70,-73.99
1,36047000200,Brooklyn,1170.00,78942.00,0.61,0.56,0.00,40.65,-74.01
2,36047000301,Brooklyn,4057.00,193158.00,0.61,0.46,0.03,40.70,-74.00


In [62]:
# Spatially join census tracts to NTAs to get the NTA name for each tract
# This gives us the bridge between tract-level ACS data and neighborhood names

# Build census GeoDataFrame
census_gdf = gpd.GeoDataFrame(
    df_census,
    geometry=gpd.points_from_xy(df_census['centroid_lon'], df_census['centroid_lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2263')

if nta_gdf is not None and nta_gdf.geometry.geom_type.unique().tolist() != ['Point']:
    # Identify the NTA name column
    nta_name_col = next((c for c in ['NTAName','nta_name','NTAname'] if c in nta_proj.columns), None)
    nta_code_col = next((c for c in ['NTA2020','nta_code','NTA2010'] if c in nta_proj.columns), None)
    boro_col     = next((c for c in ['BoroName','borough','BoroName'] if c in nta_proj.columns), None)

    print(f'NTA name col  : {nta_name_col}')
    print(f'NTA code col  : {nta_code_col}')
    print(f'Borough col   : {boro_col}')

    # Join each census tract centroid to the NTA polygon it falls within
    join_cols = ['geometry']
    if nta_name_col: join_cols.append(nta_name_col)
    if nta_code_col: join_cols.append(nta_code_col)
    if boro_col:     join_cols.append(boro_col)

    census_with_nta = gpd.sjoin(
        census_gdf,
        nta_proj[join_cols],
        how='left',
        predicate='within'
    )
    # Rename for consistency
    rename_map = {}
    if nta_name_col: rename_map[nta_name_col] = 'nta_name'
    if nta_code_col: rename_map[nta_code_col] = 'nta_code'
    if boro_col and boro_col != 'borough': rename_map[boro_col] = 'nta_borough'
    census_with_nta = census_with_nta.rename(columns=rename_map)

    matched = census_with_nta['nta_name'].notna().sum()
    print(f'\nTracts matched to an NTA: {matched:,} / {len(census_with_nta):,}')
else:
    # Fallback: use borough as grouping unit
    print('NTA polygon join not available -- using borough as grouping unit')
    census_with_nta = census_gdf.copy()
    census_with_nta['nta_name'] = census_with_nta['borough']
    census_with_nta['nta_code'] = census_with_nta['borough']

NTA name col  : NTAName
NTA code col  : NTA2020
Borough col   : BoroName

Tracts matched to an NTA: 2,143 / 2,242


In [63]:
# Attach coverage status to each tract
# Fix: cast geoid to string on both sides before merging
# Root cause: DuckDB stores geoid as int64, pygris stores it as string

census_with_nta['geoid'] = census_with_nta['geoid'].astype(str).str.zfill(11)

if 'tracts_coverage' in dir() and tracts_coverage is not None:
    coverage_flag = tracts_coverage[['geoid','coverage_status']].copy()
    coverage_flag['geoid'] = coverage_flag['geoid'].astype(str).str.zfill(11)

    census_with_nta = census_with_nta.merge(
        coverage_flag, on='geoid', how='left'
    )
    census_with_nta['coverage_status'] = (
        census_with_nta['coverage_status'].fillna('Gap Zone')
    )

else:
    # Fallback from Phase 2 CSV
    phase2_tracts = pd.read_csv(EXPORTS_DIR / 'tract_demographics_coverage.csv')
    phase2_tracts['geoid'] = phase2_tracts['geoid'].astype(str).str.zfill(11)

    census_with_nta = census_with_nta.merge(
        phase2_tracts[['geoid','coverage_status']], on='geoid', how='left'
    )
    census_with_nta['coverage_status'] = (
        census_with_nta['coverage_status'].fillna('Gap Zone')
    )

print('Coverage status attached:')
display(census_with_nta['coverage_status'].value_counts().reset_index())

# Sanity check: confirm geoid format looks right
print(f'\nSample geoid values after cast:')
print(census_with_nta['geoid'].head(5).tolist())
print(f'Expected format: 11-digit string like "36047XXXXXX"')

Coverage status attached:


,coverage_status,count
0,Covered,1174
1,Gap Zone,1068



Sample geoid values after cast:
['36047000100', '36047000200', '36047000301', '36047000501', '36047000502']
Expected format: 11-digit string like "36047XXXXXX"


In [64]:
# Aggregate from tract level to NTA level
# Population-weighted average for rate columns
# This ensures a large low-income tract doesn't get equal weight to a small one

def wavg(group, col, weight_col='population'):
    """Population-weighted average."""
    w = group[weight_col]
    v = group[col]
    mask = w.notna() & v.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return (v[mask] * w[mask]).sum() / w[mask].sum()

nta_agg_rows = []

for (nta_name, borough), group in census_with_nta.groupby(['nta_name','borough']):
    if pd.isna(nta_name):
        continue

    # A neighborhood is a gap zone if MOST of its population is uncovered
    gap_pop     = group.loc[group['coverage_status']=='Gap Zone', 'population'].sum()
    total_pop   = group['population'].sum()
    pct_gap     = gap_pop / total_pop if total_pop > 0 else 0
    nta_status  = 'Gap Zone' if pct_gap >= 0.5 else 'Covered'

    nta_agg_rows.append({
        'nta_name'         : nta_name,
        'borough'          : borough,
        'nta_code'         : group['nta_code'].iloc[0] if 'nta_code' in group.columns else '',
        'tract_count'      : len(group),
        'population'       : total_pop,
        'gap_population'   : gap_pop,
        'pct_gap_pop'      : round(pct_gap * 100, 1),
        'coverage_status'  : nta_status,
        'median_income'    : wavg(group, 'median_income'),
        'no_vehicle_rate'  : wavg(group, 'no_vehicle_rate'),
        'transit_share'    : wavg(group, 'transit_share'),
        'bike_share'       : wavg(group, 'bike_share'),
        'centroid_lat'     : group['centroid_lat'].mean(),
        'centroid_lon'     : group['centroid_lon'].mean(),
    })

df_nta_agg = pd.DataFrame(nta_agg_rows)
print(f'NTAs aggregated: {len(df_nta_agg):,}')
print(f'Gap zone NTAs  : {(df_nta_agg["coverage_status"]=="Gap Zone").sum():,}')
display(df_nta_agg.head(5))

NTAs aggregated: 197
Gap zone NTAs  : 95


,nta_name,borough,nta_code,tract_count,population,gap_population,pct_gap_pop,coverage_status,median_income,no_vehicle_rate,transit_share,bike_share,centroid_lat,centroid_lon
0,Allerton,Bronx,BX1104,9,35581.00,30576.00,85.90,Gap Zone,46623.31,0.58,0.60,0.01,40.86,-73.86
1,Annadale-Huguenot-Prince's Bay-Woodrow,Staten Island,SI0304,7,42756.00,42756.00,100.00,Gap Zone,118500.42,0.06,0.19,0.00,40.53,-74.20
2,Arden Heights-Rossville,Staten Island,SI0303,7,31559.00,31559.00,100.00,Gap Zone,104843.07,0.05,0.14,0.00,40.55,-74.19
3,Astoria (Central),Queens,QN0103,11,46503.00,0.00,0.00,Covered,103838.60,0.68,0.60,0.01,40.76,-73.92
4,Astoria (East)-Woodside (North),Queens,QN0104,14,37239.00,0.00,0.00,Covered,79364.65,0.59,0.58,0.02,40.76,-73.91


---
## 4. Bus Stop Density per NTA

Bus stop density is one of the strongest signals for transit dependency.
A neighborhood with many bus stops but no CitiBike is a neighborhood where
residents rely on buses to get anywhere -- exactly the last-mile gap CitiBike fills.

We compute stops per square kilometer to normalize for neighborhood size.

In [65]:
# Load bus stop coordinates
df_bus = q("""
    SELECT stop_id,
           CAST(lat AS DOUBLE) AS lat,
           CAST(lon AS DOUBLE) AS lon,
           borough_feed
    FROM bus_stops
    WHERE lat IS NOT NULL AND lon IS NOT NULL
""")

bus_gdf = gpd.GeoDataFrame(
    df_bus,
    geometry=gpd.points_from_xy(df_bus['lon'], df_bus['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2263')

print(f'Bus stops loaded: {len(bus_gdf):,}')

Bus stops loaded: 13,370


In [66]:
# Count bus stops per NTA using spatial join
# Each bus stop point is joined to the NTA polygon it falls within

if nta_gdf is not None and nta_gdf.geometry.geom_type.unique().tolist() != ['Point']:
    # Polygon NTAs -- proper spatial join
    join_cols = ['geometry']
    if 'NTAName' in nta_proj.columns: join_cols.append('NTAName')
    if 'NTA2020' in nta_proj.columns: join_cols.append('NTA2020')

    bus_nta = gpd.sjoin(
        bus_gdf,
        nta_proj[join_cols].rename(columns={'NTAName':'nta_name','NTA2020':'nta_code'}),
        how='left',
        predicate='within'
    )

    bus_counts = (
        bus_nta[bus_nta['nta_name'].notna()]
        .groupby('nta_name')
        .size()
        .reset_index(name='bus_stop_count')
    )

    # Also compute NTA area in sq km for density
    nta_areas = nta_proj.copy()
    nta_areas['area_sqkm'] = nta_proj.geometry.area / 1_000_000 * (0.3048 ** 2) * 1_000_000
    # EPSG:2263 is in feet; area in sq feet -> sq km: divide by (3280.84^2)
    nta_areas['area_sqkm'] = nta_proj.geometry.area / (3280.84 ** 2)

    name_col = 'NTAName' if 'NTAName' in nta_areas.columns else 'nta_name'
    nta_area_df = nta_areas[[name_col, 'area_sqkm']].rename(columns={name_col:'nta_name'})

    # Merge counts and area into nta_agg
    df_nta_agg = df_nta_agg.merge(bus_counts, on='nta_name', how='left')
    df_nta_agg = df_nta_agg.merge(nta_area_df, on='nta_name', how='left')
    df_nta_agg['bus_stop_count'] = df_nta_agg['bus_stop_count'].fillna(0)
    df_nta_agg['area_sqkm']      = df_nta_agg['area_sqkm'].fillna(df_nta_agg['area_sqkm'].median())
    df_nta_agg['bus_stop_density'] = (
        df_nta_agg['bus_stop_count'] / df_nta_agg['area_sqkm'].replace(0, np.nan)
    ).fillna(0)

    print('Bus stop density added:')
    display(df_nta_agg[['nta_name','borough','bus_stop_count',
                         'area_sqkm','bus_stop_density']]
            .sort_values('bus_stop_density', ascending=False).head(10))

else:
    # Bounding box fallback per borough
    bus_borough_counts = df_bus.groupby('borough_feed').size().reset_index(name='bus_stop_count')
    bus_borough_counts = bus_borough_counts.rename(columns={'borough_feed':'borough'})
    df_nta_agg = df_nta_agg.merge(bus_borough_counts, on='borough', how='left')
    df_nta_agg['bus_stop_count']   = df_nta_agg['bus_stop_count'].fillna(0)
    df_nta_agg['bus_stop_density'] = df_nta_agg['bus_stop_count']  # no area available
    df_nta_agg['area_sqkm']        = np.nan
    print('Bus stop counts added at borough level (fallback).')

Bus stop density added:


,nta_name,borough,bus_stop_count,area_sqkm,bus_stop_density
113,Midtown-Times Square,Manhattan,182,2.28,79.79
112,Midtown South-Flatiron-Union Square,Manhattan,80,1.38,57.87
62,Financial District-Battery Park City,Manhattan,100,1.79,55.98
83,Harlem (South),Manhattan,73,1.34,54.47
73,Gramercy,Manhattan,37,0.70,52.92
185,Washington Heights (South),Manhattan,95,1.95,48.63
181,Upper West Side-Lincoln Square,Manhattan,71,1.47,48.38
53,East Midtown-Turtle Bay,Manhattan,58,1.22,47.52
82,Harlem (North),Manhattan,103,2.21,46.60
122,Murray Hill-Kips Bay,Manhattan,68,1.47,46.41


---
## 5. Build the Demand Proxy Score

The Demand Proxy Score estimates how much a gap-zone neighborhood *would* use
CitiBike if stations existed there. It combines five components:

| Component | Weight | Rationale |
|---|---|---|
| Transit commute share | 35% | Strongest signal: bus commuters are the natural CitiBike user |
| No-vehicle household rate | 25% | Car-free = dependent on alternatives |
| Population density | 20% | More people = more potential trips |
| Bus stop density | 10% | Dense bus network = transit infrastructure already there |
| Inverse income | 10% | Equity weight: lower income areas are most underserved |

All components are min-max normalized to 0-1 before weighting,
so no single metric dominates due to scale differences.

In [67]:
def minmax(series):
    """Min-max normalize a series to 0-1. Handles NaN gracefully."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

# Compute population density (people per sq km)
if 'area_sqkm' in df_nta_agg.columns and df_nta_agg['area_sqkm'].notna().any():
    df_nta_agg['pop_density'] = (
        df_nta_agg['population'] / df_nta_agg['area_sqkm'].replace(0, np.nan)
    ).fillna(0)
else:
    # Fallback: use raw population as density proxy
    df_nta_agg['pop_density'] = df_nta_agg['population']

# Normalize each component across ALL NTAs (not just gap zones)
# so gap zones are scored relative to the full city
df_nta_agg['norm_transit']     = minmax(df_nta_agg['transit_share'])
df_nta_agg['norm_no_vehicle']  = minmax(df_nta_agg['no_vehicle_rate'])
df_nta_agg['norm_density']     = minmax(df_nta_agg['pop_density'])
df_nta_agg['norm_bus_density'] = minmax(df_nta_agg['bus_stop_density'])
# Inverse income: lower income = higher score
df_nta_agg['norm_inv_income']  = 1 - minmax(
    df_nta_agg['median_income'].fillna(df_nta_agg['median_income'].median())
)

# Weights
W_TRANSIT     = 0.35
W_NO_VEHICLE  = 0.25
W_DENSITY     = 0.20
W_BUS_DENSITY = 0.10
W_INV_INCOME  = 0.10

df_nta_agg['demand_score'] = (
    df_nta_agg['norm_transit']     * W_TRANSIT
  + df_nta_agg['norm_no_vehicle']  * W_NO_VEHICLE
  + df_nta_agg['norm_density']     * W_DENSITY
  + df_nta_agg['norm_bus_density'] * W_BUS_DENSITY
  + df_nta_agg['norm_inv_income']  * W_INV_INCOME
)

# Round for readability
df_nta_agg['demand_score'] = df_nta_agg['demand_score'].round(4)

print('Demand Proxy Score computed for all NTAs')
print(f'Score range: {df_nta_agg["demand_score"].min():.3f} - {df_nta_agg["demand_score"].max():.3f}')
print(f'Score mean : {df_nta_agg["demand_score"].mean():.3f}')

Demand Proxy Score computed for all NTAs
Score range: 0.020 - 0.752
Score mean : 0.415


In [68]:
# Score component breakdown for inspection
# Lets you verify the weights are producing sensible results
component_cols = ['nta_name','borough','coverage_status','demand_score',
                  'norm_transit','norm_no_vehicle','norm_density',
                  'norm_bus_density','norm_inv_income']
component_cols = [c for c in component_cols if c in df_nta_agg.columns]

print('Top 15 NTAs by demand score (all coverage statuses):')
display(
    df_nta_agg[component_cols]
    .sort_values('demand_score', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

Top 15 NTAs by demand score (all coverage statuses):


,nta_name,borough,coverage_status,demand_score,norm_transit,norm_no_vehicle,norm_density,norm_bus_density,norm_inv_income
0,Mount Hope,Bronx,Covered,0.75,1.00,0.85,0.75,0.39,0.00
1,Fordham Heights,Bronx,Covered,0.75,0.90,0.87,0.91,0.31,0.00
2,Harlem (North),Manhattan,Covered,0.72,0.87,0.86,0.72,0.55,0.00
3,Mount Eden-Claremont (West),Bronx,Covered,0.71,0.88,0.84,0.78,0.37,0.00
4,Washington Heights (South),Manhattan,Covered,0.69,0.82,0.94,0.56,0.58,0.00
5,Hamilton Heights-Sugar Hill,Manhattan,Covered,0.69,0.91,0.89,0.48,0.48,0.00
6,East Harlem (North),Manhattan,Covered,0.69,0.89,0.95,0.48,0.40,0.00
7,Concourse-Concourse Village,Bronx,Covered,0.68,0.91,0.85,0.51,0.25,0.24
8,Upper East Side-Yorkville,Manhattan,Covered,0.68,0.63,0.83,1.00,0.47,0.00
9,Upper West Side-Manhattan Valley,Manhattan,Covered,0.67,0.71,0.87,0.80,0.49,0.00


---
## 6. Rank Gap-Zone Neighborhoods

Filter to gap zones only and rank by demand score.
These are the neighborhoods CitiBike should prioritize for expansion.

In [69]:
# Gap zone neighborhoods ranked by demand score
gap_zones = (
    df_nta_agg[df_nta_agg['coverage_status'] == 'Gap Zone']
    .copy()
    .sort_values('demand_score', ascending=False)
    .reset_index(drop=True)
)
gap_zones['expansion_rank'] = gap_zones.index + 1

display_cols = [
    'expansion_rank','nta_name','borough','demand_score',
    'population','median_income','no_vehicle_rate',
    'transit_share','bus_stop_count'
]
display_cols = [c for c in display_cols if c in gap_zones.columns]

print(f'Gap zone neighborhoods: {len(gap_zones):,}')
print(f'\nTop 20 expansion candidates by demand score:')
display(gap_zones[display_cols].head(20))

Gap zone neighborhoods: 95

Top 20 expansion candidates by demand score:


,expansion_rank,nta_name,borough,demand_score,population,median_income,no_vehicle_rate,transit_share,bus_stop_count
0,1,Parkchester,Bronx,0.67,30221.00,59699.77,0.69,0.64,37
1,2,Brownsville,Brooklyn,0.62,54529.00,33493.55,0.71,0.67,89
2,3,East Flatbush-Remsen Village,Brooklyn,0.57,38460.00,57610.92,0.60,0.61,63
3,4,Soundview-Bruckner-Bronx River,Bronx,0.55,67145.00,41358.61,0.64,0.61,54
4,5,Allerton,Bronx,0.54,35581.00,46623.31,0.58,0.60,31
5,6,Cypress Hills,Brooklyn,0.53,40786.00,-100598282.53,0.53,0.52,47
6,7,East New York-City Line,Brooklyn,0.52,49066.00,50859.72,0.60,0.59,82
7,8,Jamaica,Queens,0.49,58368.00,61787.50,0.52,0.57,174
8,9,East New York-New Lots,Brooklyn,0.48,53067.00,58086.81,0.57,0.57,109
9,10,East New York (North),Brooklyn,0.47,44062.00,57841.43,0.52,0.56,59


In [70]:
# Borough breakdown of top 15 candidates
top15 = gap_zones.head(15)
print('Top 15 gap zones by borough:')
display(top15.groupby('borough').agg(
    neighborhoods=('nta_name','count'),
    avg_demand_score=('demand_score','mean'),
    total_gap_population=('gap_population','sum')
).round(3).reset_index().sort_values('avg_demand_score', ascending=False))

# Highlight Sheepshead Bay specifically
shbay_rows = gap_zones[gap_zones['nta_name'].str.contains('Sheepshead|sheepshead', case=False, na=False)]
if len(shbay_rows) > 0:
    print(f'\nSheepshead Bay rank:')
    display(shbay_rows[display_cols])
else:
    print('\nNote: Sheepshead Bay may appear under a different NTA name.')
    print('NTA names containing "Bay":')
    print(gap_zones[gap_zones['nta_name'].str.contains('Bay|bay', na=False)]['nta_name'].tolist())

Top 15 gap zones by borough:


,borough,neighborhoods,avg_demand_score,total_gap_population
1,Brooklyn,7,0.52,257074.00
0,Bronx,7,0.50,234024.00
2,Queens,1,0.49,58368.00



Sheepshead Bay rank:


,expansion_rank,nta_name,borough,demand_score,population,median_income,no_vehicle_rate,transit_share,bus_stop_count
41,42,Sheepshead Bay-Manhattan Beach-Gerritsen Beach,Brooklyn,0.30,66402.00,79582.20,0.36,0.39,130


In [71]:
# Score sensitivity check
# Shows how the top 10 ranking changes if we adjust the transit weight
# This demonstrates your methodology is robust, not cherry-picked

print('Sensitivity check: top 10 ranking under two alternative weight schemes')
print()

# Alternative 1: Equity-heavy (weight income more)
gap_zones['score_equity'] = (
    gap_zones['norm_transit']     * 0.25
  + gap_zones['norm_no_vehicle']  * 0.25
  + gap_zones['norm_density']     * 0.20
  + gap_zones['norm_bus_density'] * 0.10
  + gap_zones['norm_inv_income']  * 0.20
).round(4)

# Alternative 2: Density-heavy (weight population more)
gap_zones['score_density'] = (
    gap_zones['norm_transit']     * 0.30
  + gap_zones['norm_no_vehicle']  * 0.20
  + gap_zones['norm_density']     * 0.35
  + gap_zones['norm_bus_density'] * 0.10
  + gap_zones['norm_inv_income']  * 0.05
).round(4)

top10_base    = set(gap_zones.nlargest(10,'demand_score')['nta_name'])
top10_equity  = set(gap_zones.nlargest(10,'score_equity')['nta_name'])
top10_density = set(gap_zones.nlargest(10,'score_density')['nta_name'])

consistent = top10_base & top10_equity & top10_density
print(f'In top 10 under ALL three weight schemes ({len(consistent)} neighborhoods):')
for n in sorted(consistent):
    print(f'  {n}')
print(f'\nThese are your most defensible expansion recommendations.')

Sensitivity check: top 10 ranking under two alternative weight schemes

In top 10 under ALL three weight schemes (8 neighborhoods):
  Allerton
  Brownsville
  Cypress Hills
  East Flatbush-Remsen Village
  East New York-City Line
  Jamaica
  Parkchester
  Soundview-Bruckner-Bronx River

These are your most defensible expansion recommendations.


---
## 7. Proposed Expansion Locations

For the top 15 gap-zone neighborhoods, estimate how many stations each needs
and generate a proposed station count. This feeds Phase 4's financial model.

**Station count estimation:**
One station per 0.3 square miles of gap area, capped at 8 and floored at 2.
This matches CitiBike's observed station spacing in covered neighborhoods.

In [72]:
# Estimate proposed station count for top 15 gap zones
STATIONS_PER_SQMI = 1 / 0.3   # 1 station per 0.3 sq miles
SQM_PER_SQMI      = 2_589_988  # square meters per square mile

top15 = gap_zones.head(15).copy()

if 'area_sqkm' in top15.columns and top15['area_sqkm'].notna().any():
    # area_sqkm to sq miles: 1 sqkm = 0.386 sqmi
    top15['area_sqmi']          = top15['area_sqkm'] * 0.386
    top15['proposed_stations']  = (
        (top15['area_sqmi'] * STATIONS_PER_SQMI)
        .clip(lower=2, upper=8)
        .round(0)
        .astype(int)
    )
else:
    # Fallback: scale by population
    top15['proposed_stations'] = (
        (top15['population'] / 5000)
        .clip(lower=2, upper=8)
        .round(0)
        .astype(int)
    )

top15['proposed_docks'] = top15['proposed_stations'] * 23  # avg 23 docks per station

summary_cols = [
    'expansion_rank','nta_name','borough','demand_score',
    'population','proposed_stations','proposed_docks',
    'centroid_lat','centroid_lon'
]
summary_cols = [c for c in summary_cols if c in top15.columns]

print(f'Top 15 expansion recommendations:')
display(top15[summary_cols])

total_stations = top15['proposed_stations'].sum()
total_docks    = top15['proposed_docks'].sum()
print(f'\nTotal proposed stations : {total_stations}')
print(f'Total proposed docks    : {total_docks}')
print(f'These figures feed directly into the Phase 4 financial model.')

Top 15 expansion recommendations:


,expansion_rank,nta_name,borough,demand_score,population,proposed_stations,proposed_docks,centroid_lat,centroid_lon
0,1,Parkchester,Bronx,0.67,30221.00,2,46,40.84,-73.86
1,2,Brownsville,Brooklyn,0.62,54529.00,4,92,40.66,-73.91
2,3,East Flatbush-Remsen Village,Brooklyn,0.57,38460.00,2,46,40.66,-73.92
3,4,Soundview-Bruckner-Bronx River,Bronx,0.55,67145.00,4,92,40.83,-73.87
4,5,Allerton,Bronx,0.54,35581.00,2,46,40.86,-73.86
5,6,Cypress Hills,Brooklyn,0.53,40786.00,3,69,40.68,-73.88
6,7,East New York-City Line,Brooklyn,0.52,49066.00,4,92,40.67,-73.87
7,8,Jamaica,Queens,0.49,58368.00,6,138,40.70,-73.80
8,9,East New York-New Lots,Brooklyn,0.48,53067.00,6,138,40.66,-73.89
9,10,East New York (North),Brooklyn,0.47,44062.00,3,69,40.67,-73.89



Total proposed stations : 50
Total proposed docks    : 1150
These figures feed directly into the Phase 4 financial model.


In [73]:
# Sheepshead Bay specific recommendation
print('=== SHEEPSHEAD BAY EXPANSION PROPOSAL ===')

shbay = top15[top15['nta_name'].str.contains('Sheepshead|sheepshead', case=False, na=False)]

if len(shbay) == 0:
    # Try all gap zones if not in top 15
    shbay = gap_zones[gap_zones['nta_name'].str.contains('Sheepshead|sheepshead', case=False, na=False)]

if len(shbay) > 0:
    r = shbay.iloc[0]
    print(f'Neighborhood    : {r["nta_name"]}')
    print(f'Borough         : {r["borough"]}')
    print(f'Expansion rank  : #{int(r["expansion_rank"])}')
    print(f'Demand score    : {r["demand_score"]:.4f}')
    print(f'Population      : {r["population"]:,.0f}')
    if 'median_income' in r:
        print(f'Median income   : ${r["median_income"]:,.0f}')
    if 'no_vehicle_rate' in r:
        print(f'No-vehicle rate : {r["no_vehicle_rate"]*100:.1f}%')
    if 'transit_share' in r:
        print(f'Transit commute : {r["transit_share"]*100:.1f}%')
    if 'bus_stop_count' in r:
        print(f'Bus stops       : {int(r["bus_stop_count"]):,}')
    if 'proposed_stations' in r:
        print(f'Proposed stations: {int(r["proposed_stations"])}')
        print(f'Proposed docks  : {int(r["proposed_docks"])}')
else:
    print('Sheepshead Bay NTA name may differ -- check gap_zones nta_name column')
    print('Available southern Brooklyn NTAs:')
    bk_gap = gap_zones[gap_zones['borough']=='Brooklyn']
    print(bk_gap[bk_gap['centroid_lat'] < 40.61][['nta_name','expansion_rank','demand_score']].to_string())

=== SHEEPSHEAD BAY EXPANSION PROPOSAL ===
Neighborhood    : Sheepshead Bay-Manhattan Beach-Gerritsen Beach
Borough         : Brooklyn
Expansion rank  : #42
Demand score    : 0.3007
Population      : 66,402
Median income   : $79,582
No-vehicle rate : 36.2%
Transit commute : 38.6%
Bus stops       : 130


---
## 8. Export for Phase 4 and Tableau

In [74]:
# Full NTA scoring table (all neighborhoods, covered and gap)
export_cols = [
    'nta_name','nta_code','borough','coverage_status','demand_score','expansion_rank',
    'population','gap_population','pct_gap_pop',
    'median_income','no_vehicle_rate','transit_share','bike_share',
    'bus_stop_count','bus_stop_density','area_sqkm',
    'norm_transit','norm_no_vehicle','norm_density','norm_bus_density','norm_inv_income',
    'centroid_lat','centroid_lon'
]
export_cols = [c for c in export_cols if c in gap_zones.columns]

# Add expansion_rank to all gap zones
all_nta_export = df_nta_agg.copy().sort_values('demand_score', ascending=False)
gap_mask = all_nta_export['coverage_status'] == 'Gap Zone'
all_nta_export.loc[gap_mask, 'expansion_rank'] = range(1, gap_mask.sum() + 1)
all_nta_export['expansion_rank'] = all_nta_export['expansion_rank'].fillna(0).astype(int)

export_cols_all = [c for c in export_cols if c in all_nta_export.columns]
all_nta_export[export_cols_all].to_csv(
    EXPORTS_DIR / 'nta_demand_scores.csv', index=False
)
print(f'Exported: nta_demand_scores.csv  ({len(all_nta_export):,} neighborhoods)')

Exported: nta_demand_scores.csv  (197 neighborhoods)


In [75]:
# Top 15 expansion candidates (for Phase 4 financial model)
top15_export_cols = [
    'expansion_rank','nta_name','borough','demand_score',
    'population','gap_population','median_income',
    'no_vehicle_rate','transit_share','bus_stop_count',
    'proposed_stations','proposed_docks',
    'centroid_lat','centroid_lon'
]
top15_export_cols = [c for c in top15_export_cols if c in top15.columns]
top15[top15_export_cols].to_csv(EXPORTS_DIR / 'top15_expansion_candidates.csv', index=False)
print(f'Exported: top15_expansion_candidates.csv  ({len(top15)} neighborhoods)')

Exported: top15_expansion_candidates.csv  (15 neighborhoods)


In [76]:
# Phase 3 summary
print('=' * 60)
print('  PHASE 3 EXPORT SUMMARY')
print('=' * 60)
for f in sorted(EXPORTS_DIR.glob('*.csv')):
    rows = sum(1 for _ in open(f)) - 1
    kb   = f.stat().st_size / 1024
    print(f'  {f.name:<45}  {rows:>6,} rows  {kb:>7.1f} KB')

print()
print('Key files for downstream phases:')
print('  nta_demand_scores.csv          --> Tableau: neighborhood scorecard + map')
print('  top15_expansion_candidates.csv --> Phase 4: financial model inputs')

con.close()
print('\nDuckDB connection closed.')
print('Phase 3 complete. Ready for Phase 4: Financial Projection Model.')

  PHASE 3 EXPORT SUMMARY
  brooklyn_station_demand.csv                       978 rows     77.6 KB
  gap_zone_borough_summary.csv                        5 rows      0.4 KB
  income_coverage_distribution.csv                    5 rows      0.3 KB
  nta_demand_scores.csv                             197 rows     62.7 KB
  sheepshead_bay_bus_stops.csv                      225 rows     11.1 KB
  sheepshead_bay_nearest_stations.csv                15 rows      0.9 KB
  sheepshead_bay_profile.csv                         42 rows      3.3 KB
  stations_with_demand.csv                        2,333 rows    257.7 KB
  top15_expansion_candidates.csv                     15 rows      2.4 KB
  tract_demographics_coverage.csv                 2,242 rows    213.8 KB

Key files for downstream phases:
  nta_demand_scores.csv          --> Tableau: neighborhood scorecard + map
  top15_expansion_candidates.csv --> Phase 4: financial model inputs

DuckDB connection closed.
Phase 3 complete. Ready for Phase 4: Fin

# CitiBike Borough Expansion Analysis
## Phase 4: Financial Projection Model

This phase builds the business case for expansion: what does it cost,
what does it earn, and what ridership volume makes each neighborhood viable?

**What is built here:**
- Baseline revenue analysis from existing stations (what a station actually earns)
- Fare model using published CitiBike/Lyft pricing
- Per-station cost model (installation + annual operations)
- Break-even analysis: trips needed to cover costs
- Low/mid/high ridership scenarios tied to Phase 3 demand scores
- NPV and payback period per neighborhood
- Sheepshead Bay financial case
- Exports for Tableau financial model tab

**Sections:**
- **0** Session setup
- **1** Load Phase 3 inputs
- **2** Baseline revenue from existing stations
- **3** Fare model and financial assumptions
- **4** Per-station revenue projection
- **5** Cost model
- **6** Break-even analysis
- **7** Scenario modeling tied to demand scores
- **8** Neighborhood ROI projections
- **9** Sheepshead Bay financial case
- **10** Export for Tableau

---
## 0. Session Setup
Run this cell first every session.

In [77]:
!pip install duckdb --quiet

import warnings
import numpy as np
import pandas as pd
import duckdb
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = Path('/content/drive/MyDrive/citibike_project')
DIRS = {
    'exports' : PROJECT_ROOT / 'data' / 'exports',
    'models'  : PROJECT_ROOT / 'models',
    'db'      : PROJECT_ROOT / 'db',
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

EXPORTS_DIR = DIRS['exports']
MODELS_DIR  = DIRS['models']
DB_PATH     = str(DIRS['db'] / 'citibike.duckdb')

con = duckdb.connect(DB_PATH)

def q(sql, label=''):
    if label: print(f'\n-- {label} --')
    return con.execute(sql).fetchdf()

tables = con.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'main'
""").fetchdf()['table_name'].tolist()

print('DuckDB tables:', tables)
print('\nExport files available:')
for f in sorted(EXPORTS_DIR.glob('*.csv')):
    print(f'  {f.name}')
print('\nSession ready.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DuckDB tables: ['bus_stops', 'census_tracts', 'nta_lookup', 'stations', 'tracts_geo', 'trips', 'trips_clean', 'trips_with_station']

Export files available:
  brooklyn_station_demand.csv
  gap_zone_borough_summary.csv
  income_coverage_distribution.csv
  nta_demand_scores.csv
  sheepshead_bay_bus_stops.csv
  sheepshead_bay_nearest_stations.csv
  sheepshead_bay_profile.csv
  stations_with_demand.csv
  top15_expansion_candidates.csv
  tract_demographics_coverage.csv

Session ready.


---
## 1. Load Phase 3 Inputs
The two key files from Phase 3 that drive this entire model.

In [78]:
# Top 15 expansion candidates with proposed station counts
top15 = pd.read_csv(EXPORTS_DIR / 'top15_expansion_candidates.csv')
print(f'Top 15 candidates loaded: {len(top15)} neighborhoods')
display(top15[[
    'expansion_rank','nta_name','borough','demand_score',
    'population','proposed_stations','proposed_docks'
]].head(15))

Top 15 candidates loaded: 15 neighborhoods


,expansion_rank,nta_name,borough,demand_score,population,proposed_stations,proposed_docks
0,1,Parkchester,Bronx,0.67,"30,221.00",2,46
1,2,Brownsville,Brooklyn,0.62,"54,529.00",4,92
2,3,East Flatbush-Remsen Village,Brooklyn,0.57,"38,460.00",2,46
3,4,Soundview-Bruckner-Bronx River,Bronx,0.55,"67,145.00",4,92
4,5,Allerton,Bronx,0.54,"35,581.00",2,46
5,6,Cypress Hills,Brooklyn,0.53,"40,786.00",3,69
6,7,East New York-City Line,Brooklyn,0.52,"49,066.00",4,92
7,8,Jamaica,Queens,0.49,"58,368.00",6,138
8,9,East New York-New Lots,Brooklyn,0.48,"53,067.00",6,138
9,10,East New York (North),Brooklyn,0.47,"44,062.00",3,69


In [79]:
# Full NTA demand scores (all neighborhoods)
nta_scores = pd.read_csv(EXPORTS_DIR / 'nta_demand_scores.csv')
print(f'NTA scores loaded: {len(nta_scores)} neighborhoods')

# Existing station demand (Phase 2 output)
stations_demand = pd.read_csv(EXPORTS_DIR / 'stations_with_demand.csv')
print(f'Station demand loaded: {len(stations_demand):,} stations')

# Quick check: how many stations have non-zero trip data?
active = stations_demand[stations_demand['total_trips'] > 0]
print(f'Stations with trip data: {len(active):,}')
print(f'\nStation demand sample:')
display(stations_demand[stations_demand['total_trips'] > 0].head(5))

NTA scores loaded: 197 neighborhoods
Station demand loaded: 2,333 stations
Stations with trip data: 2,006

Station demand sample:


,station_id,station_name,borough,lat,lon,capacity,total_trips,avg_duration_min,member_trips,member_pct,trips_per_dock
0,3db799ba-b574-4e5f-95e5-c767bec2acb6,Melrose Ave & E 150 St,Bronx,40.82,-73.92,31,2090,9.54,1820,87.10,67.42
1,4b67c6e0-d603-4fbc-afbe-0b8952f4f7af,E 161 St & River Ave,Bronx,40.83,-73.93,38,1346,11.65,1088,80.80,35.42
2,1f8f6714-42a7-45d5-86cd-a093c940b2c9,Grand Concourse & E 161 St,Bronx,40.83,-73.92,27,1256,12.18,1065,84.80,46.52
3,11a8d465-b4eb-466b-a6ca-055f73994231,Lincoln Ave & E 138 St,Bronx,40.81,-73.93,26,1233,9.49,1067,86.50,47.42
4,dfe2c398-ff5c-4a8d-83c8-fa18801540d8,Plaza Dr & W 170 St,Bronx,40.84,-73.92,30,1195,9.58,1060,88.70,39.83


---
## 2. Baseline Revenue from Existing Stations

Before projecting revenue for new stations, we need to understand what
existing stations actually generate. This anchors the financial model
in real observed behavior rather than pure assumptions.

We estimate revenue from our trip data using published CitiBike fare structure.
This is an estimate, not audited financials -- but it is directionally accurate
and is the standard approach in market-entry feasibility analyses.

In [80]:
# Revenue estimation from trip data
# CitiBike pricing (as of 2024, sourced from citibikenyc.com/pricing):
#
# MEMBERS ($205/year annual pass):
#   Classic bike  : included in membership (unlimited 45-min rides)
#   E-bike surcharge: $0.19/min for members
#   Revenue proxy : $205 / avg trips per member per year
#   Simpler proxy : $3.50 per member trip (industry standard for revenue allocation)
#
# CASUAL (non-member):
#   Single ride   : $4.99 for 30 min classic, $4.99 + $0.30/min e-bike
#   Day pass      : $19 for unlimited 30-min rides
#   Revenue proxy : $5.50 per casual trip (blended single ride + day pass)
#
# E-BIKE SURCHARGE (applied on top of above for electric bike trips):
#   Members: $0.19/min | Casual: $0.30/min
#   Avg e-bike trip duration ~14 min → avg surcharge ~$3.00

FARE = {
    'member_per_trip'         : 3.50,   # membership revenue allocated per trip
    'casual_per_trip'         : 5.50,   # blended casual fare
    'ebike_surcharge_member'  : 0.19,   # per minute
    'ebike_surcharge_casual'  : 0.30,   # per minute
    'avg_ebike_duration_min'  : 14.0,   # observed median from Phase 2
}

# E-bike share from Phase 2 trip analysis
# Pull from DuckDB if trips_with_station exists, else use a reasonable default
try:
    ebike_share = con.execute("""
        SELECT
            ROUND(COUNT(*) FILTER (WHERE rideable_type = 'electric_bike')
                  * 100.0 / COUNT(*), 1) AS ebike_pct
        FROM trips_clean
    """).fetchdf()['ebike_pct'][0]
    print(f'E-bike share from trip data: {ebike_share:.1f}%')
except Exception:
    ebike_share = 45.0
    print(f'trips_clean not available -- using default e-bike share: {ebike_share}%')

EBIKE_SHARE = ebike_share / 100

# Effective revenue per trip (blended across bike type and rider type)
# Formula: weighted average of member + casual fares, plus e-bike surcharge where applicable
def effective_revenue_per_trip(member_pct: float) -> float:
    """
    Estimate blended revenue per trip given a member percentage.
    member_pct: fraction of trips by members (0.0 to 1.0)
    """
    casual_pct = 1 - member_pct

    base = (
        member_pct * FARE['member_per_trip']
      + casual_pct * FARE['casual_per_trip']
    )

    # E-bike surcharge on top
    ebike_surcharge = EBIKE_SHARE * (
        member_pct * FARE['ebike_surcharge_member'] * FARE['avg_ebike_duration_min']
      + casual_pct * FARE['ebike_surcharge_casual'] * FARE['avg_ebike_duration_min']
    )

    return round(base + ebike_surcharge, 4)

# Test at typical member rates
for pct in [0.60, 0.70, 0.80]:
    rev = effective_revenue_per_trip(pct)
    print(f'  {pct*100:.0f}% member mix → ${rev:.2f} revenue per trip')

E-bike share from trip data: 64.4%
  60% member mix → $6.41 revenue per trip
  70% member mix → $6.11 revenue per trip
  80% member mix → $5.81 revenue per trip


In [81]:
# Compute estimated annual revenue per existing station
# trips_with_station covers Jan-Feb 2024 only (2 months)
# Annualize using a seasonal multiplier
#
# Jan-Feb are the two lowest-volume months (winter)
# Multiplier = 12 / (Jan + Feb share of annual trips)
# Industry data: Jan+Feb account for roughly 10-12% of annual CitiBike trips
# We use 11% --> multiplier = 12/0.11 = 10.9x

try:
    monthly_vol = q("""
        SELECT trip_month, COUNT(*) AS trips
        FROM trips_clean
        GROUP BY trip_month
        ORDER BY trip_month
    """)
    print('Monthly volumes from loaded data:')
    display(monthly_vol)
    # If we only have Jan+Feb, apply the multiplier
    months_loaded = len(monthly_vol)
    if months_loaded <= 3:
        SEASONAL_MULTIPLIER = 10.9
        print(f'\nOnly {months_loaded} month(s) loaded -- applying {SEASONAL_MULTIPLIER}x seasonal multiplier to annualize')
    else:
        SEASONAL_MULTIPLIER = 12 / months_loaded
        print(f'\n{months_loaded} months loaded -- annualizing with {SEASONAL_MULTIPLIER:.1f}x multiplier')
except Exception:
    SEASONAL_MULTIPLIER = 10.9
    print(f'Using default seasonal multiplier: {SEASONAL_MULTIPLIER}x')

print(f'\nInterpretation: multiply observed trip counts by {SEASONAL_MULTIPLIER:.1f} to estimate full-year volume')

Monthly volumes from loaded data:


,trip_month,trips
0,1,1886146
1,2,2118773
2,12,374



Only 3 month(s) loaded -- applying 10.9x seasonal multiplier to annualize

Interpretation: multiply observed trip counts by 10.9 to estimate full-year volume


In [82]:
# Compute per-station baseline revenue
active_stations = stations_demand[stations_demand['total_trips'] > 0].copy()

# Annualize trip counts
active_stations['annual_trips_est']   = active_stations['total_trips'] * SEASONAL_MULTIPLIER

# Apply effective revenue per trip using each station's member mix
active_stations['rev_per_trip']       = active_stations['member_pct'].apply(
    lambda p: effective_revenue_per_trip(p / 100)
)
active_stations['annual_revenue_est'] = (
    active_stations['annual_trips_est'] * active_stations['rev_per_trip']
).round(2)

print('Per-station baseline revenue summary:')
summary = active_stations.groupby('borough').agg(
    stations          = ('station_id', 'count'),
    avg_annual_trips  = ('annual_trips_est', 'mean'),
    avg_rev_per_trip  = ('rev_per_trip', 'mean'),
    avg_annual_rev    = ('annual_revenue_est', 'mean'),
    median_annual_rev = ('annual_revenue_est', 'median'),
    total_rev_est     = ('annual_revenue_est', 'sum'),
).round(2).reset_index()

display(summary)

overall_avg_rev   = active_stations['annual_revenue_est'].mean()
overall_med_rev   = active_stations['annual_revenue_est'].median()
overall_avg_trips = active_stations['annual_trips_est'].mean()

print(f'\nNetwork-wide baseline:')
print(f'  Avg annual trips per station    : {overall_avg_trips:,.0f}')
print(f'  Avg annual revenue per station  : ${overall_avg_rev:,.0f}')
print(f'  Median annual revenue per station: ${overall_med_rev:,.0f}')
print(f'\nThese figures anchor the "mid" ridership scenario for new stations.')

Per-station baseline revenue summary:


,borough,stations,avg_annual_trips,avg_rev_per_trip,avg_annual_rev,median_annual_rev,total_rev_est
0,Bronx,348,"3,662.74",5.77,"20,904.49","16,898.69","7,274,761.54"
1,Brooklyn,943,"20,214.33",5.68,"112,731.75","67,880.69","106,306,038.28"
2,Manhattan,418,"38,207.19",5.55,"211,009.55","165,144.95","88,201,992.63"
3,Other,297,"7,616.79",5.76,"42,664.30","24,552.55","12,671,297.23"



Network-wide baseline:
  Avg annual trips per station    : 19,227
  Avg annual revenue per station  : $106,906
  Median annual revenue per station: $47,586

These figures anchor the "mid" ridership scenario for new stations.


---
## 3. Financial Assumptions

All cost and revenue assumptions are defined here as named constants.
This makes the model transparent and easy to update if inputs change.
Tableau sliders in the final dashboard will let viewers adjust these.

In [83]:
# ── COST ASSUMPTIONS
# Sources:
#   Installation: NYC DOT bike share infrastructure reports, industry benchmarks
#   Operations: Lyft 10-K operational cost disclosures, scaled to station level

COSTS = {
    # One-time capital cost per station (dock hardware + installation + permitting)
    'install_per_station'    : 75_000,

    # Annual operating cost per station (maintenance, rebalancing, staffing allocation)
    'opex_per_station_yr'    : 20_000,

    # Average docks per new station (from Phase 3 proposed_docks / proposed_stations)
    'avg_docks_per_station'  : 23,

    # Discount rate for NPV calculation (Lyft's weighted average cost of capital ~8%)
    'discount_rate'          : 0.08,

    # Analysis horizon in years
    'years'                  : 10,
}

# ── RIDERSHIP SCENARIO ASSUMPTIONS
# Anchored to network baseline from Section 2
# Low scenario: 50% of network average (conservative for a new market)
# Mid scenario: 70% of network average (reasonable ramp-up by year 3)
# High scenario: 90% of network average (strong adoption, comparable to inner Brooklyn)

SCENARIOS = {
    'low'  : {'multiplier': 0.50, 'label': 'Conservative (50% of network avg)'},
    'mid'  : {'multiplier': 0.70, 'label': 'Base Case (70% of network avg)'},
    'high' : {'multiplier': 0.90, 'label': 'Optimistic (90% of network avg)'},
}

# ── RAMP-UP CURVE
# New stations don't hit full ridership in Year 1
# Based on CitiBike expansion data from Bronx and outer Brooklyn launches:
# Year 1: 40% of steady-state | Year 2: 70% | Year 3+: 100%
RAMP_UP = {1: 0.40, 2: 0.70}
# Years 3-10 default to 1.0 (handled in model)

# ── MEMBER MIX ASSUMPTION FOR NEW STATIONS
# New outer-borough stations typically start with lower member share
# We use 60% (vs ~73% network average) for new stations in gap zones
NEW_STATION_MEMBER_PCT = 0.60

REV_PER_TRIP_NEW = effective_revenue_per_trip(NEW_STATION_MEMBER_PCT)

print('Financial model assumptions:')
print(f'\nCost assumptions:')
for k, v in COSTS.items():
    print(f'  {k:<30} : {v}')
print(f'\nRevenue per trip (new stations): ${REV_PER_TRIP_NEW:.2f}')
print(f'  (based on {NEW_STATION_MEMBER_PCT*100:.0f}% member mix + {EBIKE_SHARE*100:.0f}% e-bike share)')
print(f'\nRidership scenarios (% of network average {overall_avg_trips:,.0f} trips/station/yr):')
for k, v in SCENARIOS.items():
    trips = overall_avg_trips * v['multiplier']
    print(f'  {k:<6}: {trips:>7,.0f} trips/yr  {v["label"]}')
print(f'\nRamp-up curve: Year 1={RAMP_UP[1]*100:.0f}%  Year 2={RAMP_UP[2]*100:.0f}%  Year 3+={100}%')

Financial model assumptions:

Cost assumptions:
  install_per_station            : 75000
  opex_per_station_yr            : 20000
  avg_docks_per_station          : 23
  discount_rate                  : 0.08
  years                          : 10

Revenue per trip (new stations): $6.41
  (based on 60% member mix + 64% e-bike share)

Ridership scenarios (% of network average 19,227 trips/station/yr):
  low   :   9,614 trips/yr  Conservative (50% of network avg)
  mid   :  13,459 trips/yr  Base Case (70% of network avg)
  high  :  17,304 trips/yr  Optimistic (90% of network avg)

Ramp-up curve: Year 1=40%  Year 2=70%  Year 3+=100%


---
## 4. Per-Station Revenue Projection

Build the annual revenue curve for a single new station under each scenario.
This is the building block the neighborhood model scales up from.

In [84]:
def project_station_revenue(
    annual_trips_steady: float,
    rev_per_trip: float = REV_PER_TRIP_NEW,
    years: int = COSTS['years'],
    ramp_up: dict = RAMP_UP
) -> pd.DataFrame:
    """
    Project revenue for a single station over N years.

    Parameters:
        annual_trips_steady : trips at full ridership (steady-state)
        rev_per_trip        : blended revenue per trip
        years               : projection horizon
        ramp_up             : dict of {year: fraction_of_steady_state}

    Returns:
        DataFrame with year-by-year trips and revenue
    """
    rows = []
    for yr in range(1, years + 1):
        utilization = ramp_up.get(yr, 1.0)
        trips       = annual_trips_steady * utilization
        revenue     = trips * rev_per_trip
        rows.append({
            'year'          : yr,
            'utilization'   : utilization,
            'trips'         : round(trips),
            'gross_revenue' : round(revenue, 2),
        })
    return pd.DataFrame(rows)


# Show revenue curves for all three scenarios
print('Single-station revenue projection by scenario:\n')
for scenario, params in SCENARIOS.items():
    trips_steady = overall_avg_trips * params['multiplier']
    df_rev = project_station_revenue(trips_steady)
    total_10yr = df_rev['gross_revenue'].sum()
    print(f"{scenario.upper()} -- {params['label']}")
    print(f"  Steady-state trips/yr  : {trips_steady:>8,.0f}")
    print(f"  Year 1 revenue         : ${df_rev.loc[0,'gross_revenue']:>10,.0f}")
    print(f"  Year 3+ revenue/yr     : ${df_rev.loc[2,'gross_revenue']:>10,.0f}")
    print(f"  10-year gross revenue  : ${total_10yr:>10,.0f}")
    print()

Single-station revenue projection by scenario:

LOW -- Conservative (50% of network avg)
  Steady-state trips/yr  :    9,614
  Year 1 revenue         : $    24,648
  Year 3+ revenue/yr     : $    61,620
  10-year gross revenue  : $   560,741

MID -- Base Case (70% of network avg)
  Steady-state trips/yr  :   13,459
  Year 1 revenue         : $    34,507
  Year 3+ revenue/yr     : $    86,268
  10-year gross revenue  : $   785,038

HIGH -- Optimistic (90% of network avg)
  Steady-state trips/yr  :   17,304
  Year 1 revenue         : $    44,366
  Year 3+ revenue/yr     : $   110,916
  10-year gross revenue  : $ 1,009,335



---
## 5. Cost Model

Total cost per station: one-time capital + annual operating costs over the horizon.

In [85]:
def project_station_costs(
    n_stations: int = 1,
    years: int = COSTS['years']
) -> pd.DataFrame:
    """
    Project costs for a given number of stations over N years.

    Year 0: capital installation cost (one-time)
    Year 1+: annual operating costs
    """
    capex      = COSTS['install_per_station'] * n_stations
    opex_yr    = COSTS['opex_per_station_yr'] * n_stations

    rows = [{'year': 0, 'capex': capex, 'opex': 0, 'total_cost': capex}]
    for yr in range(1, years + 1):
        rows.append({'year': yr, 'capex': 0, 'opex': opex_yr, 'total_cost': opex_yr})

    return pd.DataFrame(rows)


# Single station cost structure
cost_1 = project_station_costs(n_stations=1)
print('Single-station cost structure:')
display(cost_1)
print(f'\n10-year total cost (1 station): ${cost_1["total_cost"].sum():,.0f}')
print(f'  Capital (Year 0): ${COSTS["install_per_station"]:,.0f}')
print(f'  Operating (Yr 1-10): ${COSTS["opex_per_station_yr"]:,.0f}/yr x 10 = ${COSTS["opex_per_station_yr"]*10:,.0f}')

Single-station cost structure:


,year,capex,opex,total_cost
0,0,75000,0,75000
1,1,0,20000,20000
2,2,0,20000,20000
3,3,0,20000,20000
4,4,0,20000,20000
5,5,0,20000,20000
6,6,0,20000,20000
7,7,0,20000,20000
8,8,0,20000,20000
9,9,0,20000,20000



10-year total cost (1 station): $275,000
  Capital (Year 0): $75,000
  Operating (Yr 1-10): $20,000/yr x 10 = $200,000


---
## 6. Break-Even Analysis

The break-even question: how many trips per year does a station need
to cover its costs? And how many years until cumulative revenue exceeds
cumulative costs?

This is the core financial viability argument for the dashboard.

In [86]:
def compute_breakeven(
    annual_trips_steady: float,
    rev_per_trip: float = REV_PER_TRIP_NEW,
    n_stations: int = 1,
    years: int = COSTS['years'],
) -> dict:
    """
    Compute cumulative P&L and identify break-even year for a station group.
    Returns a dict with break-even year, cumulative cash flows, and NPV.
    """
    rev_df  = project_station_revenue(annual_trips_steady, rev_per_trip, years)
    cost_df = project_station_costs(n_stations, years)

    # Year 0 is capital outlay only
    cash_flows = [-cost_df.loc[cost_df['year']==0, 'total_cost'].values[0]]

    for yr in range(1, years + 1):
        rev  = rev_df.loc[rev_df['year']==yr, 'gross_revenue'].values[0] * n_stations
        cost = cost_df.loc[cost_df['year']==yr, 'total_cost'].values[0]
        cash_flows.append(rev - cost)

    cumulative   = np.cumsum(cash_flows)
    breakeven_yr = next(
        (i for i, v in enumerate(cumulative) if v >= 0), None
    )

    # NPV
    r   = COSTS['discount_rate']
    npv = sum(cf / (1 + r) ** t for t, cf in enumerate(cash_flows))

    return {
        'cash_flows'       : cash_flows,
        'cumulative'       : cumulative.tolist(),
        'breakeven_year'   : breakeven_yr,
        'npv'              : round(npv, 2),
        'total_revenue'    : sum(max(cf, 0) + cost_df.loc[cost_df['year']==t,'total_cost'].values[0]
                                 if t > 0 else 0
                                 for t, cf in enumerate(cash_flows)),
        'total_cost'       : cost_df['total_cost'].sum() * n_stations,
    }


# Break-even table: all three scenarios for a single station
print('Break-even analysis -- single station:\n')
be_rows = []
for scenario, params in SCENARIOS.items():
    trips   = overall_avg_trips * params['multiplier']
    result  = compute_breakeven(trips)
    be_rows.append({
        'Scenario'          : f"{scenario.capitalize()} ({params['multiplier']*100:.0f}%)",
        'Annual trips (SS)' : f"{trips:,.0f}",
        'Break-even year'   : result['breakeven_year'] if result['breakeven_year'] else '>10 yrs',
        '10-yr NPV'         : f"${result['npv']:,.0f}",
        'Viable?'           : 'Yes' if result['npv'] > 0 else 'No'
    })

display(pd.DataFrame(be_rows))

# Minimum trips for break-even (solve analytically)
# At steady state (Year 3+): revenue/yr = trips * rev_per_trip
# Cumulative cost after Y years = capex + opex*Y
# Cumulative revenue = trips * rev_per_trip * (0.4 + 0.7 + 1.0*(Y-2)) for Y>=3
# Solve for trips where cumulative_rev >= cumulative_cost
# Simplified: focus on Year 5 as target break-even

target_yr    = 5
cum_cost_5yr = COSTS['install_per_station'] + COSTS['opex_per_station_yr'] * target_yr
# Revenue accumulates with ramp-up: yr1=0.4, yr2=0.7, yr3-5=1.0
trip_weight  = RAMP_UP.get(1,1.0) + RAMP_UP.get(2,1.0) + (target_yr - 2) * 1.0
min_trips_5yr_breakeven = cum_cost_5yr / (trip_weight * REV_PER_TRIP_NEW)

print(f'\nMinimum annual trips for Year-5 break-even (single station):')
print(f'  {min_trips_5yr_breakeven:,.0f} trips/year at steady state')
print(f'  = {min_trips_5yr_breakeven/365:,.1f} trips/day')
print(f'  Network average for comparison: {overall_avg_trips:,.0f} trips/yr ({overall_avg_trips/365:.1f}/day)')

Break-even analysis -- single station:



,Scenario,Annual trips (SS),Break-even year,10-yr NPV,Viable?
0,Low (50%),"9,614",4,"$154,191",Yes
1,Mid (70%),"13,459",3,"$299,548",Yes
2,High (90%),"17,304",2,"$444,905",Yes



Minimum annual trips for Year-5 break-even (single station):
  6,659 trips/year at steady state
  = 18.2 trips/day
  Network average for comparison: 19,227 trips/yr (52.7/day)


In [87]:
# Year-by-year cumulative P&L table for all scenarios
print('Cumulative P&L by year (single station):\n')

pl_rows = []
for scenario, params in SCENARIOS.items():
    trips  = overall_avg_trips * params['multiplier']
    result = compute_breakeven(trips)
    for yr, (cf, cum) in enumerate(zip(result['cash_flows'], result['cumulative'])):
        pl_rows.append({
            'scenario'        : scenario,
            'year'            : yr,
            'annual_cashflow' : round(cf, 0),
            'cumulative_pl'   : round(cum, 0),
        })

pl_df = pd.DataFrame(pl_rows)

# Pivot for readability
pivot = pl_df.pivot(index='year', columns='scenario', values='cumulative_pl')
pivot.columns = [f'Cumulative P&L ({c})' for c in pivot.columns]
pivot.index.name = 'Year'
display(pivot.style.format('${:,.0f}').applymap(
    lambda v: 'color: green' if v >= 0 else 'color: red'
))

Cumulative P&L by year (single station):



,Cumulative P&L (high),Cumulative P&L (low),Cumulative P&L (mid)
Year,,,
0,"$-75,000","$-75,000","$-75,000"
1,"$-50,634","$-70,352","$-60,493"
2,"$7,007","$-47,218","$-20,105"
3,"$97,923","$-5,598","$46,163"
4,"$188,839","$36,022","$112,431"
5,"$279,755","$77,642","$178,698"
6,"$370,671","$119,262","$244,966"
7,"$461,587","$160,882","$311,234"
8,"$552,503","$202,502","$377,502"


---
## 7. Scenario Modeling Tied to Demand Scores

The demand score from Phase 3 determines which ridership scenario applies
to each neighborhood. High-scoring neighborhoods get the optimistic scenario;
lower-scoring ones get the conservative scenario.

This makes the financial model dynamic rather than applying a flat assumption
to every neighborhood regardless of how different they are.

In [88]:
# Map demand score to scenario
# Score thresholds divide gap zones into thirds
# Top third -> high scenario | Middle third -> mid | Bottom third -> low

gap_only = top15.copy()

score_33 = gap_only['demand_score'].quantile(0.33)
score_67 = gap_only['demand_score'].quantile(0.67)

def assign_scenario(score):
    if score >= score_67:
        return 'high'
    elif score >= score_33:
        return 'mid'
    else:
        return 'low'

gap_only['ridership_scenario'] = gap_only['demand_score'].apply(assign_scenario)

print(f'Score thresholds:')
print(f'  33rd percentile (low/mid boundary): {score_33:.4f}')
print(f'  67th percentile (mid/high boundary): {score_67:.4f}')
print(f'\nScenario assignment:')
display(gap_only[['nta_name','borough','demand_score','ridership_scenario']]
        .sort_values('demand_score', ascending=False))

Score thresholds:
  33rd percentile (low/mid boundary): 0.4657
  67th percentile (mid/high boundary): 0.5341

Scenario assignment:


,nta_name,borough,demand_score,ridership_scenario
0,Parkchester,Bronx,0.67,high
1,Brownsville,Brooklyn,0.62,high
2,East Flatbush-Remsen Village,Brooklyn,0.57,high
3,Soundview-Bruckner-Bronx River,Bronx,0.55,high
4,Allerton,Bronx,0.54,high
5,Cypress Hills,Brooklyn,0.53,mid
6,East New York-City Line,Brooklyn,0.52,mid
7,Jamaica,Queens,0.49,mid
8,East New York-New Lots,Brooklyn,0.48,mid
9,East New York (North),Brooklyn,0.47,mid


In [89]:
# Compute steady-state annual trips per station for each neighborhood's scenario
gap_only['trips_per_station_steady'] = gap_only['ridership_scenario'].map({
    'low'  : overall_avg_trips * SCENARIOS['low']['multiplier'],
    'mid'  : overall_avg_trips * SCENARIOS['mid']['multiplier'],
    'high' : overall_avg_trips * SCENARIOS['high']['multiplier'],
})

# Also compute a demand-score-weighted projection
# Interpolates smoothly: score 0 -> 40% of avg, score 1 -> 100% of avg
# More nuanced than the three-bucket approach above
gap_only['trips_per_station_weighted'] = (
    overall_avg_trips * (0.40 + gap_only['demand_score'] * 0.60)
).round(0)

print('Projected annual trips per station by neighborhood:')
display(gap_only[[
    'nta_name','demand_score','ridership_scenario',
    'trips_per_station_steady','trips_per_station_weighted'
]].sort_values('demand_score', ascending=False))

Projected annual trips per station by neighborhood:


,nta_name,demand_score,ridership_scenario,trips_per_station_steady,trips_per_station_weighted
0,Parkchester,0.67,high,"17,304.38","15,419.00"
1,Brownsville,0.62,high,"17,304.38","14,855.00"
2,East Flatbush-Remsen Village,0.57,high,"17,304.38","14,261.00"
3,Soundview-Bruckner-Bronx River,0.55,high,"17,304.38","14,078.00"
4,Allerton,0.54,high,"17,304.38","13,912.00"
5,Cypress Hills,0.53,mid,"13,458.96","13,815.00"
6,East New York-City Line,0.52,mid,"13,458.96","13,723.00"
7,Jamaica,0.49,mid,"13,458.96","13,334.00"
8,East New York-New Lots,0.48,mid,"13,458.96","13,202.00"
9,East New York (North),0.47,mid,"13,458.96","13,165.00"


---
## 8. Neighborhood ROI Projections

Apply the financial model to each of the top 15 expansion candidates.
Output: break-even year, 10-year NPV, and revenue/cost summary per neighborhood.

In [90]:
roi_rows = []

for _, row in gap_only.iterrows():
    n_stations = int(row.get('proposed_stations', 3))
    trips_ss   = row['trips_per_station_weighted']
    result     = compute_breakeven(trips_ss, n_stations=n_stations)

    # Annual revenue at steady state (Year 3+)
    ann_rev_ss = trips_ss * REV_PER_TRIP_NEW * n_stations
    total_cost = COSTS['install_per_station'] * n_stations + \
                 COSTS['opex_per_station_yr'] * n_stations * COSTS['years']

    roi_rows.append({
        'expansion_rank'       : int(row['expansion_rank']),
        'nta_name'             : row['nta_name'],
        'borough'              : row['borough'],
        'demand_score'         : round(row['demand_score'], 4),
        'ridership_scenario'   : row['ridership_scenario'],
        'proposed_stations'    : n_stations,
        'trips_per_stn_yr'     : round(trips_ss, 0),
        'ann_revenue_ss'       : round(ann_rev_ss, 0),
        'total_10yr_cost'      : round(total_cost, 0),
        'npv_10yr'             : round(result['npv'], 0),
        'breakeven_year'       : result['breakeven_year'] if result['breakeven_year'] else 99,
        'viable'               : 'Yes' if result['npv'] > 0 else 'No',
    })

roi_df = pd.DataFrame(roi_rows).sort_values('expansion_rank')

print('Top 15 neighborhood ROI summary:\n')
display(roi_df[[
    'expansion_rank','nta_name','borough','demand_score',
    'proposed_stations','trips_per_stn_yr','ann_revenue_ss',
    'npv_10yr','breakeven_year','viable'
]])

viable_count = (roi_df['viable'] == 'Yes').sum()
total_npv    = roi_df['npv_10yr'].sum()
total_capex  = roi_df['proposed_stations'].sum() * COSTS['install_per_station']

print(f'\nPortfolio summary (all 15 neighborhoods):')
print(f'  Financially viable neighborhoods : {viable_count} / {len(roi_df)}')
print(f'  Total proposed stations          : {roi_df["proposed_stations"].sum()}')
print(f'  Total capital required (capex)   : ${total_capex:,.0f}')
print(f'  Combined 10-year NPV             : ${total_npv:,.0f}')

Top 15 neighborhood ROI summary:



,expansion_rank,nta_name,borough,demand_score,proposed_stations,trips_per_stn_yr,ann_revenue_ss,npv_10yr,breakeven_year,viable
0,1,Parkchester,Bronx,0.67,2,"15,419.00","197,662.00","747,276.00",3,Yes
1,2,Brownsville,Brooklyn,0.62,4,"14,855.00","380,864.00","1,409,274.00",3,Yes
2,3,East Flatbush-Remsen Village,Brooklyn,0.57,2,"14,261.00","182,817.00","659,731.00",3,Yes
3,4,Soundview-Bruckner-Bronx River,Bronx,0.55,4,"14,078.00","360,943.00","1,291,792.00",3,Yes
4,5,Allerton,Bronx,0.54,2,"13,912.00","178,343.00","633,346.00",3,Yes
5,6,Cypress Hills,Brooklyn,0.53,3,"13,815.00","265,650.00","939,019.00",3,Yes
6,7,East New York-City Line,Brooklyn,0.52,4,"13,723.00","351,841.00","1,238,115.00",3,Yes
7,8,Jamaica,Queens,0.49,6,"13,334.00","512,802.00","1,768,948.00",3,Yes
8,9,East New York-New Lots,Brooklyn,0.48,6,"13,202.00","507,725.00","1,739,010.00",3,Yes
9,10,East New York (North),Brooklyn,0.47,3,"13,165.00","253,151.00","865,309.00",3,Yes



Portfolio summary (all 15 neighborhoods):
  Financially viable neighborhoods : 15 / 15
  Total proposed stations          : 50
  Total capital required (capex)   : $3,750,000
  Combined 10-year NPV             : $15,106,230


In [91]:
# Sensitivity: how does NPV change if ridership is 20% lower than projected?
print('Downside sensitivity: NPV if actual ridership is 20% below projection\n')

sensitivity_rows = []
for _, row in roi_df.iterrows():
    trips_downside = row['trips_per_stn_yr'] * 0.80
    result_down    = compute_breakeven(
        trips_downside,
        n_stations=int(row['proposed_stations'])
    )
    sensitivity_rows.append({
        'nta_name'        : row['nta_name'],
        'npv_base'        : row['npv_10yr'],
        'npv_downside'    : round(result_down['npv'], 0),
        'npv_delta'       : round(result_down['npv'] - row['npv_10yr'], 0),
        'still_viable'    : 'Yes' if result_down['npv'] > 0 else 'No',
    })

sens_df = pd.DataFrame(sensitivity_rows)
display(sens_df)
robust = (sens_df['still_viable'] == 'Yes').sum()
print(f'\nNeighborhoods viable even under 20% downside: {robust} / {len(sens_df)}')
print('These are your most defensible recommendations for the Tableau dashboard.')

Downside sensitivity: NPV if actual ridership is 20% below projection



,nta_name,npv_base,npv_downside,npv_delta,still_viable
0,Parkchester,"747,276.00","514,140.00","-233,136.00",Yes
1,Brownsville,"1,409,274.00","960,058.00","-449,216.00",Yes
2,East Flatbush-Remsen Village,"659,731.00","444,104.00","-215,627.00",Yes
3,Soundview-Bruckner-Bronx River,"1,291,792.00","866,072.00","-425,720.00",Yes
4,Allerton,"633,346.00","422,996.00","-210,350.00",Yes
5,Cypress Hills,"939,019.00","625,694.00","-313,325.00",Yes
6,East New York-City Line,"1,238,115.00","823,131.00","-414,984.00",Yes
7,Jamaica,"1,768,948.00","1,164,116.00","-604,832.00",Yes
8,East New York-New Lots,"1,739,010.00","1,140,166.00","-598,844.00",Yes
9,East New York (North),"865,309.00","566,726.00","-298,583.00",Yes



Neighborhoods viable even under 20% downside: 15 / 15
These are your most defensible recommendations for the Tableau dashboard.


---
## 9. Sheepshead Bay Financial Case

Produce the detailed financial narrative for your anchor neighborhood.
This feeds the Tableau deep-dive tab.

In [92]:
# Find Sheepshead Bay in the ROI table
shbay_roi = roi_df[
    roi_df['nta_name'].str.contains('Sheepshead|sheepshead', case=False, na=False)
]

if len(shbay_roi) == 0:
    # Use the highest-ranked Brooklyn neighborhood as proxy
    print('Sheepshead Bay not in top 15 -- showing highest-ranked Brooklyn gap zone instead:')
    shbay_roi = roi_df[roi_df['borough'] == 'Brooklyn'].head(1)

r = shbay_roi.iloc[0]

# Rebuild full 10-year cash flow for this neighborhood
n_stn   = int(r['proposed_stations'])
trips   = r['trips_per_stn_yr']
result  = compute_breakeven(trips, n_stations=n_stn)
rev_df  = project_station_revenue(trips)
cost_df = project_station_costs(n_stn)

# Merge into year-by-year table
yr_table = pd.DataFrame({
    'Year'              : range(0, COSTS['years'] + 1),
    'Revenue ($)'       : [0] + [round(rev_df.loc[i,'gross_revenue'] * n_stn, 0)
                                  for i in range(len(rev_df))],
    'Cost ($)'          : cost_df['total_cost'].values,
    'Net Cash Flow ($)' : [round(cf, 0) for cf in result['cash_flows']],
    'Cumulative P&L ($)': [round(c, 0) for c in result['cumulative']],
})

print(f'{'='*55}')
print(f'SHEEPSHEAD BAY FINANCIAL CASE')
print(f'{'='*55}')
print(f'Neighborhood     : {r["nta_name"]}')
print(f'Borough          : {r["borough"]}')
print(f'Expansion rank   : #{int(r["expansion_rank"])}')
print(f'Demand score     : {r["demand_score"]:.4f}')
print(f'Ridership scenario: {r["ridership_scenario"].upper()}')
print(f'Proposed stations: {n_stn}')
print(f'Trips/station/yr : {trips:,.0f}')
print(f'Revenue/trip     : ${REV_PER_TRIP_NEW:.2f}')
print(f'Capital required : ${n_stn * COSTS["install_per_station"]:,.0f}')
print(f'Annual OpEx      : ${n_stn * COSTS["opex_per_station_yr"]:,.0f}/yr')
print(f'Break-even year  : Year {result["breakeven_year"]}')
print(f'10-year NPV      : ${result["npv"]:,.0f}')
print(f'{'='*55}')
print(f'\n10-Year Cash Flow:')
display(yr_table)

Sheepshead Bay not in top 15 -- showing highest-ranked Brooklyn gap zone instead:
SHEEPSHEAD BAY FINANCIAL CASE
Neighborhood     : Brownsville
Borough          : Brooklyn
Expansion rank   : #2
Demand score     : 0.6210
Ridership scenario: HIGH
Proposed stations: 4
Trips/station/yr : 14,855
Revenue/trip     : $6.41
Capital required : $300,000
Annual OpEx      : $80,000/yr
Break-even year  : Year 3
10-year NPV      : $1,409,274

10-Year Cash Flow:


,Year,Revenue ($),Cost ($),Net Cash Flow ($),Cumulative P&L ($)
0,0,0.00,300000,"-300,000.00","-300,000.00"
1,1,"152,346.00",80000,"72,346.00","-227,654.00"
2,2,"266,605.00",80000,"186,605.00","-41,049.00"
3,3,"380,864.00",80000,"300,864.00","259,815.00"
4,4,"380,864.00",80000,"300,864.00","560,680.00"
5,5,"380,864.00",80000,"300,864.00","861,544.00"
6,6,"380,864.00",80000,"300,864.00","1,162,408.00"
7,7,"380,864.00",80000,"300,864.00","1,463,273.00"
8,8,"380,864.00",80000,"300,864.00","1,764,137.00"
9,9,"380,864.00",80000,"300,864.00","2,065,001.00"


---
## 10. Export for Tableau

Four clean exports that power the Tableau financial model tab.

In [93]:
# 1. ROI summary table (one row per neighborhood)
roi_df.to_csv(EXPORTS_DIR / 'financial_roi_summary.csv', index=False)
print('Exported: financial_roi_summary.csv')

# 2. Year-by-year cash flows for all neighborhoods (for line chart in Tableau)
all_cashflows = []
for _, row in roi_df.iterrows():
    n_stn  = int(row['proposed_stations'])
    trips  = row['trips_per_stn_yr']
    result = compute_breakeven(trips, n_stations=n_stn)
    for yr, (cf, cum) in enumerate(zip(result['cash_flows'], result['cumulative'])):
        all_cashflows.append({
            'nta_name'      : row['nta_name'],
            'borough'       : row['borough'],
            'expansion_rank': row['expansion_rank'],
            'year'          : yr,
            'net_cashflow'  : round(cf, 0),
            'cumulative_pl' : round(cum, 0),
        })

cashflow_df = pd.DataFrame(all_cashflows)
cashflow_df.to_csv(EXPORTS_DIR / 'financial_cashflows.csv', index=False)
print('Exported: financial_cashflows.csv')

# 3. Scenario comparison table (all three scenarios for each neighborhood)
scenario_rows = []
for _, row in gap_only.iterrows():
    n_stn = int(row.get('proposed_stations', 3))
    for sc, params in SCENARIOS.items():
        trips  = overall_avg_trips * params['multiplier']
        result = compute_breakeven(trips, n_stations=n_stn)
        scenario_rows.append({
            'nta_name'      : row['nta_name'],
            'borough'       : row['borough'],
            'scenario'      : sc,
            'scenario_label': params['label'],
            'annual_trips'  : round(trips, 0),
            'npv_10yr'      : round(result['npv'], 0),
            'breakeven_year': result['breakeven_year'] if result['breakeven_year'] else 99,
        })

scenario_df = pd.DataFrame(scenario_rows)
scenario_df.to_csv(EXPORTS_DIR / 'financial_scenarios.csv', index=False)
print('Exported: financial_scenarios.csv')

# 4. Assumptions reference table (for Tableau parameter documentation)
assumptions = pd.DataFrame([
    {'parameter': 'Install cost per station',  'value': COSTS['install_per_station'],      'unit': 'USD'},
    {'parameter': 'Annual OpEx per station',   'value': COSTS['opex_per_station_yr'],      'unit': 'USD/yr'},
    {'parameter': 'Revenue per trip (member)', 'value': FARE['member_per_trip'],           'unit': 'USD'},
    {'parameter': 'Revenue per trip (casual)', 'value': FARE['casual_per_trip'],           'unit': 'USD'},
    {'parameter': 'E-bike surcharge (member)', 'value': FARE['ebike_surcharge_member'],    'unit': 'USD/min'},
    {'parameter': 'E-bike surcharge (casual)', 'value': FARE['ebike_surcharge_casual'],    'unit': 'USD/min'},
    {'parameter': 'E-bike share',              'value': round(EBIKE_SHARE, 2),             'unit': 'fraction'},
    {'parameter': 'New station member mix',    'value': NEW_STATION_MEMBER_PCT,            'unit': 'fraction'},
    {'parameter': 'Discount rate',             'value': COSTS['discount_rate'],            'unit': 'fraction'},
    {'parameter': 'Seasonal multiplier',       'value': SEASONAL_MULTIPLIER,               'unit': 'x'},
    {'parameter': 'Network avg trips/stn/yr',  'value': round(overall_avg_trips, 0),       'unit': 'trips'},
    {'parameter': 'Low scenario multiplier',   'value': SCENARIOS['low']['multiplier'],    'unit': 'fraction'},
    {'parameter': 'Mid scenario multiplier',   'value': SCENARIOS['mid']['multiplier'],    'unit': 'fraction'},
    {'parameter': 'High scenario multiplier',  'value': SCENARIOS['high']['multiplier'],   'unit': 'fraction'},
    {'parameter': 'Year 1 ramp-up',            'value': RAMP_UP[1],                       'unit': 'fraction'},
    {'parameter': 'Year 2 ramp-up',            'value': RAMP_UP[2],                       'unit': 'fraction'},
])
assumptions.to_csv(EXPORTS_DIR / 'financial_assumptions.csv', index=False)
print('Exported: financial_assumptions.csv')

Exported: financial_roi_summary.csv
Exported: financial_cashflows.csv
Exported: financial_scenarios.csv
Exported: financial_assumptions.csv


In [94]:
# Final summary
print('=' * 60)
print('  PHASE 4 EXPORT SUMMARY')
print('=' * 60)
for f in sorted(EXPORTS_DIR.glob('*.csv')):
    rows = sum(1 for _ in open(f)) - 1
    kb   = f.stat().st_size / 1024
    print(f'  {f.name:<45}  {rows:>6,} rows  {kb:>7.1f} KB')

print()
print('Key files for Tableau:')
print('  financial_roi_summary.csv    --> Business case tab: NPV, break-even, viability')
print('  financial_cashflows.csv      --> Line chart: cumulative P&L over 10 years')
print('  financial_scenarios.csv      --> Scenario comparison: low/mid/high per neighborhood')
print('  financial_assumptions.csv    --> Parameter reference / methodology footnotes')

con.close()
print('\nDuckDB connection closed.')
print('Phase 4 complete. Ready for Phase 5: Tableau Dashboard.')

  PHASE 4 EXPORT SUMMARY
  brooklyn_station_demand.csv                       978 rows     77.6 KB
  financial_assumptions.csv                          16 rows      0.6 KB
  financial_cashflows.csv                           165 rows      8.1 KB
  financial_roi_summary.csv                          15 rows      1.4 KB
  financial_scenarios.csv                            45 rows      3.7 KB
  gap_zone_borough_summary.csv                        5 rows      0.4 KB
  income_coverage_distribution.csv                    5 rows      0.3 KB
  nta_demand_scores.csv                             197 rows     62.7 KB
  sheepshead_bay_bus_stops.csv                      225 rows     11.1 KB
  sheepshead_bay_nearest_stations.csv                15 rows      0.9 KB
  sheepshead_bay_profile.csv                         42 rows      3.3 KB
  stations_with_demand.csv                        2,333 rows    257.7 KB
  top15_expansion_candidates.csv                     15 rows      2.4 KB
  tract_demographics_cover